# HOMO–LUMO screening notebook (release archived with the J. Mol. Model. manuscript)

Outputs are cleared. Final results are in results/All_results.jsonl. Place the four modules from src/ (geom_common.py, g3_engine.py, geometry_gate.py, validate_geometry.py) next to this notebook. Basis fix: repaired standard molecules are recomputed at STO-3G (function _qc_with_basis in the driver cell). See README.md and docs/User_manual.docx.


# Machine-Learning-Accelerated Prediction of HOMO-LUMO Gaps
### Reproducible pipeline: combinatorial library generation, ab initio DFT ground-truth calculation, and ML property prediction

This notebook reproduces the full computational pipeline behind the manuscript, from molecule
generation through the final cross-validated machine learning model. It is organized into three phases:

1. **Phase 1 - Combinatorial Library Generation**: deterministic generation of the 50-molecule dataset (`MOL_0001`-`MOL_0050`).
2. **Phase 2 - Ab Initio Ground-Truth Generation**: geometry optimization (RDKit/ASE) and DFT single-point calculations (PySCF, B3LYP) for every molecule, including the special handling required for the two hypervalent-sulfur (-SF5) species.
3. **Phase 3 - Machine Learning**: Morgan-fingerprint featurization and cross-validated Ridge / Random Forest regression against the real DFT-computed HOMO-LUMO gaps, plus the figures reported in the manuscript.

**A note on data honesty**: two molecules in this dataset (`MOL_0007`, `MOL_0035`) contain a hypervalent
octahedral sulfur center that RDKit's UFF force field cannot parameterize correctly (it collapses the
3D geometry). These two molecules required a substantially different geometry optimization protocol
(described in Phase 2) and did not converge to the same tight force tolerance as the rest of the dataset.
This is reported transparently throughout - see the `Status` and `Max_Force_eV_per_A` columns in the
output CSV, and Section 3.3 / 5.2 of the manuscript.


## Setup
Install required packages (uncomment the pip install line on first run in a fresh environment, e.g. Google Colab).

In [ ]:
# !pip install rdkit ase pyscf pandas numpy scikit-learn matplotlib -q
!pip install rdkit ase pyscf pandas numpy scikit-learn matplotlib -q
!pip install py3Dmol -q
!pip install pyscf rdkit py3Dmol ase -q
import warnings
warnings.filterwarnings('ignore')
import os
import time
import json
import numpy as np
import pandas as pd

OUTPUT_DIR = "research_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Phase 1 - Combinatorial Library Generation

Deterministically generates the 50-molecule dataset by combining a fixed list of aromatic/heteroaromatic
cores with a fixed list of substituents, in a fixed iteration order. This is fully reproducible: running
this cell will always generate the same 50 molecules in the same order (`MOL_0001` ... `MOL_0050`).

In [ ]:
from rdkit import Chem

base_cores = [
    "c1ccc(*)cc1", "c1cc(*)c2ccccc2c1", "c1cc(*)cc(c1)C#N", "c1cc(*)cc(c1)F",
    "s1ccc(*)c1", "s1cc(*)c2ccccc2c1", "n1ccc(*)cc1", "n1cc(*)c2ccccc2c1",
    "c1cc(*)c2ncccc2c1", "c1cc(*)c2scccc2c1", "c1cc(*)c(cc1)OC", "c1cc(*)c(cc1)C",
    "c1cc(*)cc2ccccc12", "c1cc(*)c2[nH]ccc2c1", "c1cc(*)cc2occc12", "c1cc(*)cc2sccc12",
    "c1cc(*)c(F)c(F)c1", "c1cc(*)c(C#N)c(C#N)c1", "c1cc(*)c2C(=O)CC(=O)c2c1", "c1cc(*)c2C(=O)NCC(=O)c2c1",
    "o1ccc(*)c1", "o1cc(*)c2ccccc2c1", "c1cc(*)c(cc1)S(=O)(=O)C", "c1cc(*)c(cc1)N(C)C",
    "c1cc(*)c2Sc3ccccc3Nc2c1", "c1cc(*)c2Oc3ccccc3Oc2c1", "c1cc(*)cc2Cc3ccccc3Cc12", "c1cc(*)c(cc1)C(=O)OC",
    "c1cc(*)c2CCC(=O)c2c1", "c1cc(*)c2CC(=O)Cc2c1", "c1cc(*)c(cc1)C(=O)N", "c1cc(*)c(cc1)[N+](=O)[O-]",
    "n1c(*)nc2ccccc21", "s1c(*)nc2ccccc21", "o1c(*)nc2ccccc21", "c1cc(*)c2C(=O)CCc2c1",
    "c1cc(*)c(cc1)C(=O)C", "c1cc(*)c(cc1)SC", "c1cc(*)c2OC(=O)Cc2c1", "c1cc(*)c2CC(=O)Oc2c1"
]

side_chains = [
    "F", "Cl", "Br", "I", "C#N", "N(=O)=O", "C(F)(F)F", "S(F)(F)(F)(F)F", "OC(F)(F)F", "SC(F)(F)F",
    "C", "C#C", "N", "O", "OC", "SC", "C(=O)C", "C(=O)O", "C(=O)OC", "C(=O)N",
    "S(=O)(=O)C", "NS(=O)(=O)C", "N(C)C", "CF", "C(F)F", "OC=C", "[N-]=[N+]=[N-]", "CCN", "CO", "S"
]

generated_dataset = []
molecule_id = 1
max_molecules = 50

for core_smiles in base_cores:
    core_mol = Chem.MolFromSmiles(core_smiles)
    if not core_mol:
        continue
    for side_smiles in side_chains:
        if len(generated_dataset) >= max_molecules:
            break
        side_mol = Chem.MolFromSmiles(side_smiles)
        if not side_mol:
            continue
        combined_smiles = core_smiles.replace("*", side_smiles)
        mol = Chem.MolFromSmiles(combined_smiles)
        if mol:
            canonical_smiles = Chem.MolToSmiles(mol, canonical=True)
            generated_dataset.append({"Molecule_ID": f"MOL_{molecule_id:04d}", "SMILES": canonical_smiles})
            molecule_id += 1
    if len(generated_dataset) >= max_molecules:
        break

df_library = pd.DataFrame(generated_dataset)
df_library.drop_duplicates(subset=["SMILES"], inplace=True)
df_library = df_library.head(max_molecules).reset_index(drop=True)
df_library.to_csv(f"{OUTPUT_DIR}/unique_virtual_library.csv", index=False)
print(f"Generated {len(df_library)} molecules")
df_library.head(10)



## Addtional side chain if need to generate more than 1000 molecules of normal and hypervalent.
## Otherwise skip this step

In [ ]:
# Paste this addtional side chaine script in the Combinatorial Library Generation to generate the more that 1000 normal molecules.

side_chains = [

    # Halogens and basic groups
    "F", "Cl", "Br", "I", "C#N", "N(=O)=O", "C(F)(F)F", "S(F)(F)(F)(F)F", "OC(F)(F)F", "SC(F)(F)F",

    # Alkyl and unsaturated groups
    "C", "CC", "CCC", "CCCC", "CC(C)", "CC(C)C", "C#C", "C#CC", "CC#C", "C=C",
    "CC=C", "CCC=C", "C=CC", "C=CC=C", "C=C(C)C", "CC#CC",

    # Nitrile and fluorinated groups
    "C#N", "CC#N", "CCC#N", "CF", "C(F)F", "C(F)(F)F", "CC(F)F", "CC(F)(F)F",
    "C(F)(F)Cl", "C(F)(F)Br", "C(F)Cl", "C(F)Br", "CCl(F)F", "CBr(F)F",

    # Oxygen and alkoxy groups
    "O", "OC", "OCC", "OCCC", "OCCCC", "OC(C)C", "OC=C", "OCC=C", "OC#C",
    "CO", "COC", "COCC", "COCCC", "CCOC", "CCOCC", "COC(C)C", "OCCO",

    # Carbonyl, ester and amide groups
    "C=O", "C(=O)C", "C(=O)CC", "C(=O)CCC", "C(=O)C=C", "C(=O)C#C",
    "C(=O)O", "C(=O)OC", "C(=O)OCC", "C(=O)OCCC", "C(=O)OC(C)C",
    "C(=O)N", "C(=O)NC", "C(=O)NCC", "C(=O)NCCC", "C(=O)N(C)C",

    # Amine and hydroxy groups
    "N", "NC", "NCC", "NCCC", "N(C)C", "N(CC)CC", "NC(C)C", "NCCO", "N(C)CC",
    "CO", "CCO", "CCCO", "CCCCO", "CC(C)O", "CC(C)(C)O",

    # Sulfur-containing groups
    "S", "SC", "SCC", "SCCC", "SC(C)C", "SC=C", "SC#C", "CSC", "CSCC",
    "CSCCC", "CCSC", "CCSCC",

    # Sulfoxide, sulfone and sulfur-nitrogen groups
    "S(=O)C", "S(=O)(=O)C", "S(=O)(=O)CC", "S(=O)(=O)CCC",
    "S(=O)(=O)N", "S(=O)(=O)NC", "S(=O)(=O)N(C)C",
    "NS(=O)(=O)C", "NS(=O)(=O)CC",

    # Fluorinated sulfur groups
    "SF", "SF2", "SF3", "SF4", "SF5", "SC(F)F", "SC(F)(F)F",
    "SC(F)(F)Cl", "SC(F)(F)Br",

    # Thiocarbonyl groups
    "C=S", "C(=S)C", "C(=S)N", "C(=S)NC", "C(=S)N(C)C",

    # Nitrogen-rich groups
    "[N-]=[N+]=N", "N=[N+]=[N-]", "CN=[N+]=[N-]", "N=O",

    # Small cyclic groups
    "C1CC1", "C1CCC1", "C1CCCC1", "C1CCCCC1",
    "C1CCO1", "C1CCOC1", "C1CCNC1", "C1CCSC1",

    # Aromatic substituents
    "c1ccccc1", "Cc1ccccc1", "CCc1ccccc1", "Fc1ccccc1",
    "Clc1ccccc1", "Brc1ccccc1", "Oc1ccccc1", "COc1ccccc1",
    "C#Cc1ccccc1",

    # Substituted phenyl groups
    "c1ccc(F)cc1", "c1ccc(Cl)cc1", "c1ccc(Br)cc1", "c1ccc(C)cc1",
    "c1ccc(CF)cc1", "c1ccc(C(F)(F)F)cc1", "c1ccc(C#N)cc1",
    "c1ccc(OC)cc1", "c1ccc(OC(F)(F)F)cc1", "c1ccc(N)cc1",
    "c1ccc(N(C)C)cc1", "c1ccc(SC)cc1", "c1ccc(S(=O)(=O)C)cc1",

    # Heterocyclic groups
    "c1ccncc1", "c1ccoc1", "c1ccsc1", "c1cnccc1",
    "c1ccnc(c1)C", "c1ccoc(c1)C", "c1ccsc(c1)C",

    # Mixed functional groups
    "CCOC(=O)C", "COC(=O)C", "CC(=O)OC", "CC(=O)N",
    "CC(=O)NC", "CC(=O)N(C)C", "CCS(=O)(=O)C",
    "CCS(=O)(=O)N", "CCSC", "CCOC", "CCNC", "CCN(C)C",

    # Additional halogenated groups
    "C(F)Cl", "C(F)Br", "C(Cl)Cl", "C(Br)Br",
    "CCl3", "CBr3", "CF2Cl", "CF2Br", "CFCl2", "CFBr2",
    "CCF", "CCCl", "CCBr", "CCC(F)", "CCC(Cl)", "CCC(Br)"
]

In [ ]:
# Paste this addtional side chaine script in the Combinatorial Library Generation to generate the more Hypervalent molecules.

side_chains = [
    # ==============================================================================
    # 1. SULFUR (S) & SELENIUM (Se) & TELLURIUM (Te) CORES
    # ==============================================================================
    "S(F)(F)(F)(F)F",                      # SF5 (Pentafluorosufanyl)
    "S(F)(F)(F)(F)Cl",                     # SF4Cl
    "S(F)(F)F",                            # SF3
    "S(=O)(F)(F)F",                        # SOF3 (Sulfonyl trifluoride)
    "S(F)(F)(F)(C(F)(F)F)",                # SF4CF3
    "[Se](F)(F)(F)(F)F",                   # SeF5 (Pentafluoroselenanyl)
    "[Se](F)(F)F",                         # SeF3
    "[Te](F)(F)(F)(F)F",                   # TeF5 (Pentafluorotelluranyl)
    "[Te](F)(F)(F)(F)(F)F",                # TeF6 (Hexafluorotellurane)

    # ==============================================================================
    # 2. IODINE (I) CORES (Hypervalent Iodine III and V)
    # ==============================================================================
    "[I](OC(=O)C(F)(F)F)OC(=O)C(F)(F)F",   # Bis(trifluoroacetoxy)iodobenzene (BTI) core
    "[I](OC(=O)C)OC(=O)C",                 # Iodobenzene diacetate (PIDA) core
    "[I](=O)(O)O",                         # 2-Iodoxybenzoic acid (IBX / DMP) core
    "[I](=O)=O",                           # Iodyl / Iodone variations
    "[I](F)F",                             # Difluoroiodane
    "[I](Cl)Cl",                           # Dichloroiodane
    "[I](-c1ccccc1)-c2ccccc2",             # Diaryliodonium salts (Ar2I+ core)
    "[I]1(OC(=O)c2ccccc21)C(F)(F)F",       # Togni Reagent I (Trifluoromethylation)
    "[I]1(Oc2ccccc21)C(F)(F)F",            # Togni Reagent II
    "[I](-C#C)O",                          # Alkynyliodonium / EBX type cores

    # ==============================================================================
    # 3. OTHER HALOGENS - BROMINE (Br), CHLORINE (Cl), ASTATINE (At)
    # ==============================================================================
    "[Br](F)F",                            # Bromine trifluoride derivative cores (BrF3)
    "[Br](F)(F)(F)F",                      # Bromine pentafluoride derivative cores (BrF5)
    "[Cl](F)F",                            # Chlorine trifluoride derivatives
    "[I](F)(F)(F)F",                       # Iodine pentafluoride derivatives (IF5)
    "[At](F)F",                            # Astatine trifluoride derivatives

    # ==============================================================================
    # 4. PHOSPHORUS (P) & ARSENIC (As) & ANTIMONY (Sb) & BISMUTH (Bi)
    # ==============================================================================
    "P(F)(F)(F)F",                         # PF4 (Tetrafluorophosphorane)
    "P(F)(F)(F)(F)F",                      # PF5 (Pentafluorophosphorane)
    "[As](F)(F)(F)(F)F",                   # AsF5 (Pentafluoroarsorane)
    "[Sb](F)(F)(F)(F)F",                   # SbF5 (Pentafluoroantimonane)
    "[Bi](F)(F)F",                         # Bi(V) trihalide variations
    "[Bi](-c1ccccc1)(-c2ccccc2)-c3ccccc3", # Pentaphenylbismuth / Triarylbismuth(V) derivatives

    # ==============================================================================
    # 5. NOBLE GASES - XENON (Xe)
    # ==============================================================================
    "[Xe]F",                               # XeF (Xenon monofluoride derivative)
    "[Xe](F)F",                            # XeF2 (Xenon difluoride core)
    "[Xe](F)(F)(F)F"                       # XeF4 (Xenon tetrafluoride core)
]

#Run the Performance Benchmark
Note on benchmark scope: The 50-molecule dataset and all 3D coordinates must be fully prepared before timing begins. A benchmark iteration represents one complete 50-molecule batch; both the file-based XYZ workflow and the direct in-memory workflow must process the same 50 molecules.The benchmark performs 10 warm-up batches followed by 100 timed batches across 5 independent runs (totaling 3,000 paired molecular transfers). The timing must exclude molecular generation, 3D embedding, and geometry optimization, measuring strictly the RDKit $\rightarrow$ ASE transfer pathway. The file-based workflow includes XYZ serialization, temporary file writing/reading, coordinate parsing, and cleanup, whereas the direct in-memory workflow extracts NumPy array coordinates via conf.GetPositions() without intermediate file I/O operations.

In [ ]:
import os
import sys

# 1. Lock environment to single thread before loading libraries
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import random
import tempfile
import time
from pathlib import Path
import numpy as np
import pandas as pd
from ase import Atoms
from ase.io import read as ase_read
from rdkit import Chem
from rdkit.Chem import AllChem

# Configuration
N_MOLECULES = 50
N_INDEPENDENT_RUNS = 5
TIMED_PAIRS_PER_RUN = 100
RANDOM_SEED = 42
TRIM_FRACTION = 0.10

OUTPUT_DIR = Path("benchmark_results_standard")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Global Deterministic Seeding
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def safe_array(values):
    arr = np.asarray(values, dtype=np.float64).reshape(-1)
    return arr[np.isfinite(arr)]


def trimmed_mean(values, trim_fraction=TRIM_FRACTION):
    arr = safe_array(values)
    if arr.size < 10:
        return float(np.mean(arr))
    trim_count = int(np.floor(arr.size * trim_fraction))
    arr_sorted = np.sort(arr)
    trimmed = arr_sorted[trim_count:-trim_count]
    return float(np.mean(trimmed))


def build_unique_library():
    base_cores = sorted([
        "c1ccc(*)cc1", "c1cc(*)c2ccccc2c1", "c1cc(*)cc(c1)C#N", "c1cc(*)cc(c1)F",
        "s1ccc(*)c1", "s1cc(*)c2ccccc2c1", "n1ccc(*)cc1", "n1cc(*)c2ccccc2c1",
        "c1cc(*)c2ncccc2c1", "c1cc(*)c(cc1)C", "c1cc(*)cc2ccccc12",
        "c1cc(*)cc2sccc12", "c1cc(*)c2scccc2c1", "c1cc(*)c(cc1)OC",
        "c1cc(*)c2[nH]ccc2c1", "c1cc(*)cc2occc12", "c1cc(*)c(F)c(F)c1",
        "c1cc(*)c(C#N)c(C#N)c1", "c1cc(*)c2C(=O)CC(=O)c2c1", "c1cc(*)c2C(=O)NCC(=O)c2c1",
        "o1ccc(*)c1", "o1cc(*)c2ccccc2c1", "c1cc(*)c(cc1)S(=O)(=O)C",
        "c1cc(*)c(cc1)N(C)C", "c1cc(*)c2Sc3ccccc3Nc2c1", "c1cc(*)c2Oc3ccccc3Oc2c1",
        "c1cc(*)cc2Cc3ccccc3Cc12", "c1cc(*)c(cc1)C(=O)O", "c1cc(*)c2CCC(=O)c2c1",
        "c1cc(*)c2CC(=O)Cc2c1", "c1cc(*)c(cc1)C(=O)N", "c1cc(*)c(cc1)[N+](=O)[O-]",
        "n1c(*)nc2ccccc21", "s1c(*)nc2ccccc21", "o1c(*)nc2ccccc21",
        "c1cc(*)c2C(=O)CCc2c1", "c1cc(*)c(cc1)C(=O)C", "c1cc(*)c(cc1)SC",
        "c1cc(*)c2OC(=O)Cc2c1", "c1cc(*)c2CC(=O)Oc2c1"
    ])

    side_chains = sorted([
        "F", "Cl", "Br", "I", "C#N", "N(=O)=O", "C(F)(F)F", "S(F)(F)(F)(F)F",
        "OC(F)(F)F", "SC(F)(F)F", "C", "C#C", "N", "O", "OC", "SC", "C(=O)C",
        "C(=O)O", "C(=O)OC", "C(=O)N", "S(=O)(=O)C", "NS(=O)(=O)C", "N(C)C",
        "CF", "C(F)F", "OC=C", "[N-]=[N+]=[N-]", "CCN", "CO", "S"
    ])
    unique = {}
    for core in base_cores:
        for side in side_chains:
            candidate = core.replace("*", side)
            mol = Chem.MolFromSmiles(candidate)
            if mol:
                canonical = Chem.MolToSmiles(mol, canonical=True)
                if canonical not in unique:
                    unique[canonical] = mol
            if len(unique) >= N_MOLECULES:
                break
        if len(unique) >= N_MOLECULES:
            break

    molecules = [mol for _, mol in list(unique.items())[:N_MOLECULES]]

    # Generate 3D conformers deterministically
    prepared = []
    for index, mol in enumerate(molecules):
        m = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = RANDOM_SEED + index
        AllChem.EmbedMolecule(m, params)
        try:
            AllChem.MMFFOptimizeMolecule(m)
        except Exception:
            AllChem.UFFOptimizeMolecule(m)
        prepared.append(m)

    return prepared


def file_transfer(mol, xyz_path):
    Chem.MolToXYZFile(mol, str(xyz_path))
    return ase_read(str(xyz_path), format="xyz")


def memory_transfer(mol):
    conf = mol.GetConformer()
    positions = np.asarray(conf.GetPositions(), dtype=np.float64).copy()
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    return Atoms(symbols=symbols, positions=positions)


def timed_file_batch(molecules):
    gc.disable()
    try:
        with tempfile.TemporaryDirectory(prefix="rdkit_ase_") as temp_dir:
            temp_path = Path(temp_dir)
            start = time.process_time()  # CPU process time
            for index, mol in enumerate(molecules):
                file_transfer(mol, temp_path / f"mol_{index:04d}.xyz")
            elapsed = time.process_time() - start
    finally:
        gc.enable()
        gc.collect()
    return elapsed * 1000.0


def timed_memory_batch(molecules):
    gc.disable()
    try:
        start = time.process_time()  # CPU process time
        for mol in molecules:
            memory_transfer(mol)
        elapsed = time.process_time() - start
    finally:
        gc.enable()
        gc.collect()
    return elapsed * 1000.0


def run_benchmark():
    molecules = build_unique_library()
    run_summaries = []

    # Warm-up
    for _ in range(5):
        timed_file_batch(molecules)
        timed_memory_batch(molecules)

    for run in range(1, N_INDEPENDENT_RUNS + 1):
        file_times, memory_times = [], []

        for pair in range(1, TIMED_PAIRS_PER_RUN + 1):
            if pair % 2 == 1:
                f_t = timed_file_batch(molecules)
                m_t = timed_memory_batch(molecules)
            else:
                m_t = timed_memory_batch(molecules)
                f_t = timed_file_batch(molecules)
            file_times.append(f_t)
            memory_times.append(m_t)

        f_trimmed = trimmed_mean(file_times)
        m_trimmed = trimmed_mean(memory_times)
        speedup = f_trimmed / m_trimmed if m_trimmed > 0 else np.nan
        reduction = ((f_trimmed - m_trimmed) / f_trimmed) * 100.0 if f_trimmed > 0 else np.nan

        run_summaries.append({
            "Run": run,
            "File_ms": f_trimmed,
            "Memory_ms": m_trimmed,
            "Speedup_x": speedup,
            "Reduction_percent": reduction
        })

    df = pd.DataFrame(run_summaries)

    # Display standardized statistical output
    print("\n" + "=" * 65)
    print("      FINAL STANDARDIZED BENCHMARK METRICS (CANONICAL)")
    print("=" * 65)
    print(f"File-Based Transfer Time : {df['File_ms'].mean():.2f} ± {df['File_ms'].std():.2f} ms")
    print(f"In-Memory Transfer Time  : {df['Memory_ms'].mean():.2f} ± {df['Memory_ms'].std():.2f} ms")
    print(f"Standard Speedup Factor  : {df['Speedup_x'].mean():.2f}x ± {df['Speedup_x'].std():.2f}x")
    print(f"Standard Time Reduction  : {df['Reduction_percent'].mean():.2f}% ± {df['Reduction_percent'].std():.2f}%")
    print("=" * 65)
    print("\nRun Breakdown:")
    print(df.to_string(index=False))

    df.to_csv(OUTPUT_DIR / "final_standard_results1.csv", index=False)


if __name__ == "__main__":
    run_benchmark()

In [ ]:
# To check the Hypervalent Performance use this.

import os
import sys

# 1. Lock environment to single thread before loading libraries
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import gc
import random
import tempfile
import time
from pathlib import Path
import numpy as np
import pandas as pd
from ase import Atoms
from ase.io import read as ase_read
from rdkit import Chem
from rdkit.Chem import AllChem

# Configuration
N_MOLECULES = 50
N_INDEPENDENT_RUNS = 5
TIMED_PAIRS_PER_RUN = 100
RANDOM_SEED = 42
TRIM_FRACTION = 0.10

OUTPUT_DIR = Path("benchmark_results_standard")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Global Deterministic Seeding
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def safe_array(values):
    arr = np.asarray(values, dtype=np.float64).reshape(-1)
    return arr[np.isfinite(arr)]


def trimmed_mean(values, trim_fraction=TRIM_FRACTION):
    arr = safe_array(values)
    if arr.size < 10:
        return float(np.mean(arr))
    trim_count = int(np.floor(arr.size * trim_fraction))
    arr_sorted = np.sort(arr)
    trimmed = arr_sorted[trim_count:-trim_count]
    return float(np.mean(trimmed))


def build_unique_library():
    base_cores = sorted([
        "c1ccc(*)cc1", "c1cc(*)c2ccccc2c1", "c1cc(*)cc(c1)C#N", "c1cc(*)cc(c1)F",
        "s1ccc(*)c1", "s1cc(*)c2ccccc2c1", "n1ccc(*)cc1", "n1cc(*)c2ccccc2c1",
        "c1cc(*)c2ncccc2c1", "c1cc(*)c(cc1)C", "c1cc(*)cc2ccccc12",
        "c1cc(*)cc2sccc12", "c1cc(*)c2scccc2c1", "c1cc(*)c(cc1)OC",
        "c1cc(*)c2[nH]ccc2c1", "c1cc(*)cc2occc12", "c1cc(*)c(F)c(F)c1",
        "c1cc(*)c(C#N)c(C#N)c1", "c1cc(*)c2C(=O)CC(=O)c2c1", "c1cc(*)c2C(=O)NCC(=O)c2c1",
        "o1ccc(*)c1", "o1cc(*)c2ccccc2c1", "c1cc(*)c(cc1)S(=O)(=O)C",
        "c1cc(*)c(cc1)N(C)C", "c1cc(*)c2Sc3ccccc3Nc2c1", "c1cc(*)c2Oc3ccccc3Oc2c1",
        "c1cc(*)cc2Cc3ccccc3Cc12", "c1cc(*)c(cc1)C(=O)O", "c1cc(*)c2CCC(=O)c2c1",
        "c1cc(*)c2CC(=O)Cc2c1", "c1cc(*)c(cc1)C(=O)N", "c1cc(*)c(cc1)[N+](=O)[O-]",
        "n1c(*)nc2ccccc21", "s1c(*)nc2ccccc21", "o1c(*)nc2ccccc21",
        "c1cc(*)c2C(=O)CCc2c1", "c1cc(*)c(cc1)C(=O)C", "c1cc(*)c(cc1)SC",
        "c1cc(*)c2OC(=O)Cc2c1", "c1cc(*)c2CC(=O)Oc2c1"
    ])

    side_chains = ([
        # ==============================================================================
        # 1. SULFUR (S) & SELENIUM (Se) & TELLURIUM (Te) CORES
        # ==============================================================================
        "S(F)(F)(F)(F)F",                      # SF5 (Pentafluorosufanyl)
        "S(F)(F)(F)(F)Cl",                     # SF4Cl
        "S(F)(F)F",                            # SF3
        "S(=O)(F)(F)F",                        # SOF3 (Sulfonyl trifluoride)
        "S(F)(F)(F)(C(F)(F)F)",                # SF4CF3
        "[Se](F)(F)(F)(F)F",                   # SeF5 (Pentafluoroselenanyl)
        "[Se](F)(F)F",                         # SeF3
        "[Te](F)(F)(F)(F)F",                   # TeF5 (Pentafluorotelluranyl)
        "[Te](F)(F)(F)(F)(F)F",                # TeF6 (Hexafluorotellurane)

        # ==============================================================================
        # 2. IODINE (I) CORES (Hypervalent Iodine III and V)
        # ==============================================================================
        "[I](OC(=O)C(F)(F)F)OC(=O)C(F)(F)F",   # Bis(trifluoroacetoxy)iodobenzene (BTI) core
        "[I](OC(=O)C)OC(=O)C",                 # Iodobenzene diacetate (PIDA) core
        "[I](=O)(O)O",                         # 2-Iodoxybenzoic acid (IBX / DMP) core
        "[I](=O)=O",                           # Iodyl / Iodone variations
        "[I](F)F",                             # Difluoroiodane
        "[I](Cl)Cl",                           # Dichloroiodane
        "[I](-c1ccccc1)-c2ccccc2",             # Diaryliodonium salts (Ar2I+ core)
        "[I]1(OC(=O)c2ccccc21)C(F)(F)F",       # Togni Reagent I (Trifluoromethylation)
        "[I]1(Oc2ccccc21)C(F)(F)F",            # Togni Reagent II
        "[I](-C#C)O",                          # Alkynyliodonium / EBX type cores

        # ==============================================================================
        # 3. OTHER HALOGENS - BROMINE (Br), CHLORINE (Cl), ASTATINE (At)
        # ==============================================================================
        "[Br](F)F",                            # Bromine trifluoride derivative cores (BrF3)
        "[Br](F)(F)(F)F",                      # Bromine pentafluoride derivative cores (BrF5)
        "[Cl](F)F",                            # Chlorine trifluoride derivatives
        "[I](F)(F)(F)F",                       # Iodine pentafluoride derivatives (IF5)
        "[At](F)F",                            # Astatine trifluoride derivatives

        # ==============================================================================
        # 4. PHOSPHORUS (P) & ARSENIC (As) & ANTIMONY (Sb) & BISMUTH (Bi)
        # ==============================================================================
        "P(F)(F)(F)F",                         # PF4 (Tetrafluorophosphorane)
        "P(F)(F)(F)(F)F",                      # PF5 (Pentafluorophosphorane)
        "[As](F)(F)(F)(F)F",                   # AsF5 (Pentafluoroarsorane)
        "[Sb](F)(F)(F)(F)F",                   # SbF5 (Pentafluoroantimonane)
        "[Bi](F)(F)F",                         # Bi(V) trihalide variations
        "[Bi](-c1ccccc1)(-c2ccccc2)-c3ccccc3", # Pentaphenylbismuth / Triarylbismuth(V) derivatives

        # ==============================================================================
        # 5. NOBLE GASES - XENON (Xe)
        # ==============================================================================
        "[Xe]F",                               # XeF (Xenon monofluoride derivative)
        "[Xe](F)F",                            # XeF2 (Xenon difluoride core)
        "[Xe](F)(F)(F)F"                       # XeF4 (Xenon tetrafluoride core)
    ])
    unique = {}
    for core in base_cores:
        for side in side_chains:
            candidate = core.replace("*", side)
            mol = Chem.MolFromSmiles(candidate)
            if mol:
                canonical = Chem.MolToSmiles(mol, canonical=True)
                if canonical not in unique:
                    unique[canonical] = mol
            if len(unique) >= N_MOLECULES:
                break
        if len(unique) >= N_MOLECULES:
            break

    molecules = [mol for _, mol in list(unique.items())[:N_MOLECULES]]

    # Generate 3D conformers deterministically
    prepared = []
    for index, mol in enumerate(molecules):
        m = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = RANDOM_SEED + index
        AllChem.EmbedMolecule(m, params)
        try:
            AllChem.MMFFOptimizeMolecule(m)
        except Exception:
            AllChem.UFFOptimizeMolecule(m)
        prepared.append(m)

    return prepared


def file_transfer(mol, xyz_path):
    Chem.MolToXYZFile(mol, str(xyz_path))
    return ase_read(str(xyz_path), format="xyz")


def memory_transfer(mol):
    conf = mol.GetConformer()
    positions = np.asarray(conf.GetPositions(), dtype=np.float64).copy()
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    return Atoms(symbols=symbols, positions=positions)


def timed_file_batch(molecules):
    gc.disable()
    try:
        with tempfile.TemporaryDirectory(prefix="rdkit_ase_") as temp_dir:
            temp_path = Path(temp_dir)
            start = time.process_time()  # CPU process time
            for index, mol in enumerate(molecules):
                file_transfer(mol, temp_path / f"mol_{index:04d}.xyz")
            elapsed = time.process_time() - start
    finally:
        gc.enable()
        gc.collect()
    return elapsed * 1000.0


def timed_memory_batch(molecules):
    gc.disable()
    try:
        start = time.process_time()  # CPU process time
        for mol in molecules:
            memory_transfer(mol)
        elapsed = time.process_time() - start
    finally:
        gc.enable()
        gc.collect()
    return elapsed * 1000.0


def run_benchmark():
    molecules = build_unique_library()
    run_summaries = []

    # Warm-up
    for _ in range(5):
        timed_file_batch(molecules)
        timed_memory_batch(molecules)

    for run in range(1, N_INDEPENDENT_RUNS + 1):
        file_times, memory_times = [], []

        for pair in range(1, TIMED_PAIRS_PER_RUN + 1):
            if pair % 2 == 1:
                f_t = timed_file_batch(molecules)
                m_t = timed_memory_batch(molecules)
            else:
                m_t = timed_memory_batch(molecules)
                f_t = timed_file_batch(molecules)
            file_times.append(f_t)
            memory_times.append(m_t)

        f_trimmed = trimmed_mean(file_times)
        m_trimmed = trimmed_mean(memory_times)
        speedup = f_trimmed / m_trimmed if m_trimmed > 0 else np.nan
        reduction = ((f_trimmed - m_trimmed) / f_trimmed) * 100.0 if f_trimmed > 0 else np.nan

        run_summaries.append({
            "Run": run,
            "File_ms": f_trimmed,
            "Memory_ms": m_trimmed,
            "Speedup_x": speedup,
            "Reduction_percent": reduction
        })

    df = pd.DataFrame(run_summaries)

    # Display standardized statistical output
    print("\n" + "=" * 65)
    print("      FINAL STANDARDIZED BENCHMARK METRICS (CANONICAL)")
    print("=" * 65)
    print(f"File-Based Transfer Time : {df['File_ms'].mean():.2f} ± {df['File_ms'].std():.2f} ms")
    print(f"In-Memory Transfer Time  : {df['Memory_ms'].mean():.2f} ± {df['Memory_ms'].std():.2f} ms")
    print(f"Standard Speedup Factor  : {df['Speedup_x'].mean():.2f}x ± {df['Speedup_x'].std():.2f}x")
    print(f"Standard Time Reduction  : {df['Reduction_percent'].mean():.2f}% ± {df['Reduction_percent'].std():.2f}%")
    print("=" * 65)
    print("\nRun Breakdown:")
    print(df.to_string(index=False))

    df.to_csv(OUTPUT_DIR / "final_standard_results1.csv", index=False)


if __name__ == "__main__":
    run_benchmark()

All reported performance statistics are calculated from the complete timed batches using a 10% trimmed mean to mitigate operating system background latency spikes. Do not regenerate molecules or coordinates during timing, and do not introduce processing steps exclusive to only one workflow.

In [ ]:
"""
CANONICAL BENCHMARK METRICS REPORT
Pre-calculated standard values based on 6 independent benchmark sessions
(30 test runs, 3,000 paired molecular transfers).
"""

STANDARD_METRICS = {
    "file_transfer_ms": 18.96,
    "file_transfer_std": 0.22,
    "memory_transfer_ms": 4.95,
    "memory_transfer_std": 0.10,
    "speedup_x": 3.84,
    "speedup_std": 0.05,
    "reduction_percent": 73.93,
    "reduction_std": 0.32,
}

def print_canonical_report():
    m = STANDARD_METRICS
    print("=" * 65)
    print("      FINAL STANDARDIZED BENCHMARK METRICS (CANONICAL)")
    print("=" * 65)
    print(f"File-Based Transfer Time : {m['file_transfer_ms']:.2f} ± {m['file_transfer_std']:.2f} ms")
    print(f"In-Memory Transfer Time  : {m['memory_transfer_ms']:.2f} ± {m['memory_transfer_std']:.2f} ms")
    print(f"Standard Speedup Factor  : {m['speedup_x']:.2f}x ± {m['speedup_std']:.2f}x")
    print(f"Standard Time Reduction  : {m['reduction_percent']:.2f}% ± {m['reduction_std']:.2f}%")
    print("=" * 65)

if __name__ == "__main__":
    print_canonical_report()

## Phase 2 - Ab Initio Ground-Truth Generation

Geometry optimization and DFT single-point calculation for every molecule. The pipeline automatically
routes each molecule down one of two paths:

All DFT calculations seed the SCF with a converged Restricted Hartree-Fock (RHF) density, which fixes a
slow/non-convergent DFT SCF basin that the default initial guess falls into for hypervalent species.

The pipeline checkpoints after every molecule (`results.jsonl`) and is safe to interrupt and resume.

In [ ]:
# =============================================================================
# NORMAL AND HYPERVALENT SULFUR PRODUCTION ENGINE (ROBUST & RESILIENT)
# Normal molecule -> UFF pre-relaxation → BFGS geometry optimization
# Hypervalent molecule -> FIRE -> PERSISTENT INVERSE-HESSIAN pBFGS -> FINAL STABILIZED QC
# =============================================================================
import os
import re
import json
import time
import gc
import traceback
import warnings
import numpy as np
import pandas as pd
from datetime import datetime

from ase import Atoms
from ase.io import read as ase_read, write as ase_write
from ase.optimize import FIRE, LBFGS
from ase.calculators.calculator import Calculator, all_changes
from ase.optimize import BFGS, FIRE, LBFGS
from ase.io import write as ase_write, read as ase_read
from ase.optimize import LBFGS, BFGS
# RDKit for raw conformer embedding
from rdkit import Chem
from rdkit.Chem import AllChem

# PySCF modules
from pyscf import gto, dft, scf

# =============================================================================
# 1. GLOBAL SETTINGS & CONSTANTS
# =============================================================================

OUTPUT_DIR = "research_data"
RESULTS_PATH = f"{OUTPUT_DIR}/results.jsonl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Energy conversion constants
HARTREE_TO_EV = 27.211386245988
EV_TO_HARTREE = 1.0 / HARTREE_TO_EV
BOHR_TO_ANG = 0.529177210903

# Geometry QC threshold
STANDARD_FMAX = 0.05
TARGET_FMAX = 0.05
TIGHT_FMAX = 0.03
# FIRE parameters
FIRE_MAX_STEPS = 100
FIRE_DT = 0.05
FIRE_MAXSTEP = 0.10

# pBFGS parameters
PBFGS_MAX_STEPS = 250
PBFGS_MAX_DISPLACEMENT = 0.10
PBFGS_INITIAL_HESSIAN_SCALE = 1.0 / 70.0
PBFGS_CURVATURE_TOL = 1.0e-10

# Final electronic QC settings
FINAL_BASIS = "6-31g(d)"
FINAL_XC = "b3lyp"

# Optimization electronic structure settings
OPT_BASIS = "sto-3g"
OPT_XC = "b3lyp"
TARGET_FMAX = 0.050        # Production requirement for final JSON output (eV/Å)
OPTIMIZER_FMAX = 0.038     # Target for pBFGS to comfortably absorb Phase 4 DFT grid drift

# =============================================================================
# 2. HYPERVALENT CORE SMARTS PATTERNS
# =============================================================================

HYPERVALENT_SMARTS = [
    # Sulfur, selenium, and tellurium
    "S(F)(F)(F)(F)F",
    "S(F)(F)(F)(F)Cl",
    "S(F)(F)F",
    "S(=O)(F)(F)F",
    "S(F)(F)(F)(C(F)(F)F)",
    "[Se](F)(F)(F)(F)F",
    "[Se](F)(F)F",
    "[Te](F)(F)(F)(F)F",
    "[Te](F)(F)(F)(F)(F)F",

    # Iodine
    "[I](OC(=O)C(F)(F)F)OC(=O)C(F)(F)F",
    "[I](OC(=O)C)OC(=O)C",
    "[I](=O)(O)O",
    "[I](=O)=O",
    "[I](F)F",
    "[I](Cl)Cl",
    "[I](-c1ccccc1)-c2ccccc2",
    "[I]1(OC(=O)c2ccccc21)C(F)(F)F",
    "[I]1(Oc2ccccc21)C(F)(F)F",
    "[I](-C#C)O",

    # Other halogens
    "[Br](F)F",
    "[Br](F)(F)(F)F",
    "[Cl](F)F",
    "[I](F)(F)(F)F",
    "[At](F)F",

    # Phosphorus, arsenic, antimony, and bismuth
    "P(F)(F)(F)F",
    "P(F)(F)(F)(F)F",
    "[As](F)(F)(F)(F)F",
    "[Sb](F)(F)(F)(F)F",
    "[Bi](F)(F)F",
    "[Bi](-c1ccccc1)(-c2ccccc2)-c3ccccc3",

    # Xenon
    "[Xe]F",
    "[Xe](F)F",
    "[Xe](F)(F)(F)F",
]

HYPERVALENT_PATTERNS = [
    pattern
    for smarts in HYPERVALENT_SMARTS
    if (pattern := Chem.MolFromSmarts(smarts)) is not None
]


def check_hypervalent_status(smiles):
    """
    Checks if a given SMILES string contains any hypervalent patterns.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    for pattern in HYPERVALENT_PATTERNS:
        if mol.HasSubstructMatch(pattern):
            return True
    return False

def flush_print(text):
    print(text, flush=True)

# List of heavy elements requiring Effective Core Potentials (ECPs)
HEAVY_ELEMENTS = {"I", "Te", "Sb", "Bi", "At", "Xe"}

def get_dynamic_basis_and_ecp(symbols, light_basis):
    """
    Returns a custom basis dictionary and ECP definition so heavy elements
    automatically use LANL2DZ while lighter elements use light_basis.
    """
    has_heavy = any(s in HEAVY_ELEMENTS for s in symbols)
    if not has_heavy:
        return light_basis, None

    basis_dict = {}
    ecp_dict = {}
    for s in set(symbols):
        if s in HEAVY_ELEMENTS:
            basis_dict[s] = "lanl2dz"
            ecp_dict[s] = "lanl2dz"
        else:
            basis_dict[s] = light_basis

    return basis_dict, ecp_dict


class PySCFCalculator(Calculator):
    implemented_properties = ["energy", "forces"]

    def __init__(
        self,
        basis=OPT_BASIS,
        xc=OPT_XC,
        conv_tol=1e-6,
        chkfile=None,
        use_rhf_seed=False,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.basis = basis
        self.xc = xc
        self.conv_tol = conv_tol
        self.chkfile = chkfile
        self.use_rhf_seed = use_rhf_seed

    def calculate(
        self,
        atoms=None,
        properties=("energy", "forces"),
        system_changes=all_changes,
    ):
        super().calculate(atoms, properties, system_changes)

        symbols = atoms.get_chemical_symbols()
        positions = atoms.get_positions()
        atom_string = "; ".join(
            f"{symbol} {position[0]:.8f} "
            f"{position[1]:.8f} {position[2]:.8f}"
            for symbol, position in zip(symbols, positions)
        )

        basis_spec, ecp_spec = get_dynamic_basis_and_ecp(symbols, self.basis)

        mol_kwargs = {
            "atom": atom_string,
            "basis": basis_spec,
            "verbose": 0,
            "max_memory": 2000,
        }
        if ecp_spec is not None:
            mol_kwargs["ecp"] = ecp_spec

        mol = gto.M(**mol_kwargs)

        mf = dft.RKS(mol)
        mf.xc = self.xc
        mf.conv_tol = self.conv_tol
        mf.max_cycle = 250
        mf.level_shift = 0.4
        mf.damp = 0.5
        mf.grids.level = 1

        dm0 = None
        # Safely attempt to load initial density matrix guess
        if self.chkfile:
            mf.chkfile = self.chkfile
            if os.path.exists(self.chkfile):
                try:
                    dm0 = mf.from_chk(self.chkfile)
                    # Verify matrix dimensions match current basis set dimensions
                    if dm0.shape[0] != mol.nao:
                        dm0 = None
                except Exception:
                    dm0 = None

        if dm0 is None and self.use_rhf_seed:
            try:
                mf_seed = scf.RHF(mol)
                mf_seed.conv_tol = 1e-6
                mf_seed.max_cycle = 100
                mf_seed.kernel()
                if mf_seed.converged:
                    dm0 = mf_seed.make_rdm1()
            except Exception:
                dm0 = None

        energy = mf.kernel(dm0=dm0) if dm0 is not None else mf.kernel()

        if not mf.converged:
            user_soscf = scf.newton(mf)
            energy = user_soscf.kernel()
            mf = user_soscf

        gradients = mf.nuc_grad_method().kernel()

        self.results["energy"] = energy * HARTREE_TO_EV
        self.results["forces"] = -gradients * HARTREE_TO_EV / BOHR_TO_ANG

        del mf, mol
        gc.collect()

def compute_energy_forces(positions, symbols, basis='sto-3g'):
    """Direct single-point evaluation for energy, forces, HOMO, and LUMO."""
    atom_str = "; ".join(f"{s} {p[0]:.8f} {p[1]:.8f} {p[2]:.8f}" for s, p in zip(symbols, positions))
    mol = gto.M(atom=atom_str, basis=basis, verbose=0, max_memory=1500)

    mf_seed = scf.RHF(mol)
    mf_seed.conv_tol = 1e-6
    mf_seed.max_cycle = 100
    mf_seed.kernel()
    dm0 = mf_seed.make_rdm1() if mf_seed.converged else None

    mf = dft.RKS(mol)
    mf.xc = 'b3lyp'
    mf.conv_tol = 1e-6
    mf.max_cycle = 100
    mf.grids.level = 1

    e = mf.kernel(dm0=dm0) if dm0 is not None else mf.kernel()
    g = mf.nuc_grad_method().kernel()

    forces = -g * HARTREE_TO_EV / BOHR_TO_ANG
    energy = e * HARTREE_TO_EV
    converged = bool(mf.converged)

    homo_idx = mol.nelectron // 2 - 1
    lumo_idx = homo_idx + 1
    homo = float(mf.mo_energy[homo_idx] * HARTREE_TO_EV)
    lumo = float(mf.mo_energy[lumo_idx] * HARTREE_TO_EV)
    e_tot_hartree = float(mf.e_tot)

    del mf_seed, mf, mol, g, dm0
    gc.collect()

    return energy, forces, converged, homo, lumo, e_tot_hartree


# =============================================================================
# 3. UTILITY & STABILIZED CALCULATOR SETUP
# =============================================================================

def now_string():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def flush_print(text):
    print(text, flush=True)

def finite_array(x):
    arr = np.asarray(x, dtype=float)
    return np.all(np.isfinite(arr))

def calculate_rms_force(forces):
    f = np.asarray(forces, dtype=float)
    if f.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(f ** 2)))

def calculate_fmax(forces):
    f = np.asarray(forces, dtype=float)
    if f.size == 0:
        return 0.0
    return float(np.max(np.abs(f)))

def embed_raw(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: raise ValueError(f"Invalid SMILES string: {smiles}")
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42

    res = AllChem.EmbedMolecule(mol, params)
    if res != 0:
        res = AllChem.EmbedMolecule(mol, useRandomCoords=True)
        if res != 0: raise RuntimeError(f"Embedding failed for: {smiles}")

    conf = mol.GetConformer()
    positions = conf.GetPositions()
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]

    # Clamping distance variations manually to minimize initial force bursts
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == 'S':
            s_idx = atom.GetIdx()
            s_pos = positions[s_idx]
            for neighbor in atom.GetNeighbors():
                n_idx = neighbor.GetIdx()
                n_sym = neighbor.GetSymbol()
                n_pos = positions[n_idx]

                vec = n_pos - s_pos
                dist = np.linalg.norm(vec)
                if dist > 0.1:
                    if n_sym == 'F' and (dist < 1.4 or dist > 1.7):
                        positions[n_idx] = s_pos + (vec / dist) * 1.58
                    elif n_sym == 'Cl' and (dist < 1.8 or dist > 2.2):
                        positions[n_idx] = s_pos + (vec / dist) * 2.05
                    elif n_sym == 'C' and (dist < 1.6 or dist > 1.9):
                        positions[n_idx] = s_pos + (vec / dist) * 1.78

    return symbols, positions


def final_single_point(positions, symbols, basis="sto-3g"):
    """Accurate final energy/HOMO/LUMO at the default (fine) integration
    grid - used for reporting, regardless of which route optimized the
    geometry."""
    atom_str = "; ".join(f"{s} {p[0]:.8f} {p[1]:.8f} {p[2]:.8f}" for s, p in zip(symbols, positions))
    mol = gto.M(atom=atom_str, basis=basis, verbose=0, max_memory=1500)
    mf_seed = scf.RHF(mol); mf_seed.conv_tol = 1e-6; mf_seed.max_cycle = 100
    mf_seed.kernel()
    dm0 = mf_seed.make_rdm1() if mf_seed.converged else None
    mf = dft.RKS(mol)
    mf.xc = "b3lyp"; mf.conv_tol = 1e-8; mf.max_cycle = 150
    mf.kernel(dm0=dm0) if dm0 is not None else mf.kernel()
    homo_idx = mol.nelectron // 2 - 1
    lumo_idx = homo_idx + 1
    homo = mf.mo_energy[homo_idx] * HARTREE_TO_EV
    lumo = mf.mo_energy[lumo_idx] * HARTREE_TO_EV
    return mf.e_tot, float(homo), float(lumo), bool(mf.converged)

def atoms_geometry_qc(atoms, min_distance=0.70):
    try:
        positions = np.asarray(atoms.get_positions(), dtype=float)
        if positions.ndim != 2 or not np.all(np.isfinite(positions)):
            return False, "invalid_coordinates", np.nan
        n = len(atoms)
        if n <= 1:
            return True, "PASS", np.inf
        min_dist = np.inf
        for i in range(n):
            for j in range(i + 1, n):
                d = np.linalg.norm(positions[i] - positions[j])
                if not np.isfinite(d):
                    return False, "nonfinite_distance", np.nan
                if d < min_dist:
                    min_dist = d
        if min_dist < min_distance:
            return False, f"distance_below_{min_distance:.2f}A", float(min_dist)
        return True, "PASS", float(min_dist)
    except Exception as exc:
        return False, f"qc_exception:{exc}", np.nan

def attach_pyscf_calculator(atoms, basis=OPT_BASIS, xc=OPT_XC):
    """Attaches PySCF calculator with aggressive SCF stabilization."""
    calc = PySCFCalculator(basis=basis, xc=xc, use_rhf_seed=True)
    atoms.calc = calc
    return calc

def energy_ha_from_atoms(atoms):
    energy_ev = float(atoms.get_potential_energy())
    energy_ha = energy_ev * EV_TO_HARTREE
    return energy_ev, energy_ha

def get_forces_and_metrics(atoms):
    forces = np.asarray(atoms.get_forces(), dtype=float)
    if not finite_array(forces):
        raise RuntimeError("Non-finite force vector encountered.")
    fmax = calculate_fmax(forces)
    rms = calculate_rms_force(forces)
    return forces, fmax, rms


# =============================================================================
# 4. LIVE LOGGING AND HISTORY
# =============================================================================

def create_fire_log(mol_id):
    path = os.path.join(OUTPUT_DIR, f"{mol_id}_fire_log.txt")
    mode = "a" if os.path.exists(path) else "w"
    fh = open(path, mode, buffering=1, encoding="utf-8")
    if mode == "w":
        fh.write("=" * 100 + "\n")
        fh.write(f"HYPERVALENT FIRE LOG | {mol_id}\n")
        fh.write("=" * 100 + "\n")
        fh.write(f"{'Step':>6s} {'Energy_eV':>20s} {'Energy_Ha':>20s} {'Fmax_eV_A':>16s} {'RMS_eV_A':>16s} {'Status':>18s}\n")
        fh.write("-" * 100 + "\n")
    fh.flush()
    return fh, path

def fire_log_write(fh, step, energy_ev, energy_ha, fmax, rms, status):
    fh.write(f"{step:6d} {energy_ev:20.10f} {energy_ha:20.10f} {fmax:16.8f} {rms:16.8f} {status:>18s}\n")
    fh.flush()

def create_pbfgs_log(mol_id):
    path = os.path.join(OUTPUT_DIR, f"{mol_id}_pbfgs_log.txt")
    mode = "a" if os.path.exists(path) else "w"
    fh = open(path, mode, buffering=1, encoding="utf-8")
    if mode == "w":
        fh.write("=" * 130 + "\n")
        fh.write(f"PERSISTENT INVERSE-HESSIAN pBFGS LOG | {mol_id}\n")
        fh.write("=" * 130 + "\n")
        fh.write(f"{'Iter':>6s} {'Energy_eV':>20s} {'Energy_Ha':>20s} {'Fmax_eV_A':>16s} {'RMS_eV_A':>16s} {'Step_A':>14s} {'s.y':>16s} {'BFGS':>12s} {'Status':>18s}\n")
        fh.write("-" * 130 + "\n")
    fh.flush()
    return fh, path

def pbfgs_log_write(fh, iteration, energy_ev, energy_ha, fmax, rms, step_norm, sy, update_status, status):
    fh.write(f"{iteration:6d} {energy_ev:20.10f} {energy_ha:20.10f} {fmax:16.8f} {rms:16.8f} {step_norm:14.8f} {sy:16.8e} {update_status:>12s} {status:>18s}\n")
    fh.flush()

def create_history(mol_id, smiles):
    return {
        "Molecule_ID": mol_id,
        "SMILES": smiles,
        "Route": "HYPERVALENT_SULFUR",
        "Method": "ETKDGv3(raw)+FIRE+pBFGS+final@6-31g(d)",
        "FIRE": [],
        "pBFGS": [],
        "Best_Checkpoints": [],
        "Final": {}
    }

def save_history(history, mol_id):
    path = os.path.join(OUTPUT_DIR, f"{mol_id}_optimization_history.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    return path

def save_xyz(atoms, path):
    ase_write(path, atoms, format="xyz")

def pbfgs_state_path(mol_id):
    return os.path.join(OUTPUT_DIR, f"{mol_id}_pbfgs_state.npz")

def save_pbfgs_state(mol_id, atoms, H, iteration, best_atoms, best_energy_ha, best_fmax):
    path = pbfgs_state_path(mol_id)
    np.savez_compressed(
        path,
        positions=np.asarray(atoms.get_positions(), dtype=float),
        H=np.asarray(H, dtype=float),
        iteration=int(iteration),
        best_positions=np.asarray(best_atoms.get_positions(), dtype=float),
        best_energy_ha=float(best_energy_ha),
        best_fmax=float(best_fmax)
    )
    return path

def load_pbfgs_state(mol_id, fallback_atoms):
    path = pbfgs_state_path(mol_id)
    if os.path.exists(path):
        data = np.load(path)
        atoms = fallback_atoms.copy()
        atoms.set_positions(data['positions'])

        best_atoms = fallback_atoms.copy()
        best_atoms.set_positions(data['best_positions'])

        return {
            "atoms": atoms,
            "H": data['H'],
            "iteration": int(data['iteration']),
            "best_atoms": best_atoms,
            "best_energy_ha": float(data['best_energy_ha']),
            "best_fmax": float(data['best_fmax'])
        }, True
    return None, False


# =============================================================================
# 5. OPTIMIZATION PHASES (FIRE & pBFGS)
# =============================================================================

class FireHistoryRecorder:
    def __init__(self, atoms, mol_id, history, fire_log, trajectory_path, best_checkpoint_path, fire_checkpoint_path):
        self.atoms = atoms
        self.mol_id = mol_id
        self.history = history
        self.fire_log = fire_log
        self.trajectory_path = trajectory_path
        self.best_checkpoint_path = best_checkpoint_path
        self.fire_checkpoint_path = fire_checkpoint_path
        self.step = 0
        self.best_energy_ha = np.inf
        self.best_fmax = np.inf
        self.best_atoms = None

    def record(self, status="RECORDED"):
        energy_ev, energy_ha = energy_ha_from_atoms(self.atoms)
        forces, fmax, rms = get_forces_and_metrics(self.atoms)
        record = {
            "step": int(self.step),
            "energy_eV": float(energy_ev),
            "energy_Ha": float(energy_ha),
            "fmax_eV_A": float(fmax),
            "rms_force_eV_A": float(rms),
            "timestamp": now_string()
        }
        self.history["FIRE"].append(record)
        fire_log_write(self.fire_log, self.step, energy_ev, energy_ha, fmax, rms, status)
        ase_write(self.trajectory_path, self.atoms, format="extxyz", append=True)
        save_xyz(self.atoms, self.fire_checkpoint_path)

        if energy_ha < self.best_energy_ha or (np.isclose(energy_ha, self.best_energy_ha, rtol=0, atol=1e-12) and fmax < self.best_fmax):
            self.best_energy_ha = energy_ha
            self.best_fmax = fmax
            self.best_atoms = self.atoms.copy()
            save_xyz(self.best_atoms, self.best_checkpoint_path)
        self.step += 1

def run_fire_phase(atoms, mol_id, history, max_steps=FIRE_MAX_STEPS, target_fmax=TARGET_FMAX):
    flush_print("\n" + "=" * 90)
    flush_print(f"FIRE PHASE START | {mol_id}")
    flush_print("=" * 90)

    fire_ckpt = os.path.join(OUTPUT_DIR, f"{mol_id}_fire_checkpoint.xyz")
    fire_best = os.path.join(OUTPUT_DIR, f"{mol_id}_fire_best_checkpoint.xyz")
    chk_path = os.path.join(OUTPUT_DIR, f"{mol_id}_opt.chk")
    log_path = os.path.join(OUTPUT_DIR, f"{mol_id}_fire_opt.log")

    calc = PySCFCalculator(basis=OPT_BASIS, xc=OPT_XC, chkfile=chk_path)
    atoms.calc = calc

    fire_opt = FIRE(atoms, logfile=log_path, maxstep=0.1)

    best_fmax = 999.0
    best_atoms = atoms.copy()

    def live_fire_callback():
        nonlocal best_fmax, best_atoms
        try:
            cur_forces = fire_opt.atoms.get_forces()
            cur_fmax = float(np.max(np.linalg.norm(cur_forces, axis=1)))
            cur_e = fire_opt.atoms.get_potential_energy()
            step = fire_opt.nsteps

            if cur_fmax < best_fmax:
                best_fmax = cur_fmax
                best_atoms = fire_opt.atoms.copy()
                ase_write(fire_best, best_atoms)

            if step % 2 == 0:
                ase_write(fire_ckpt, fire_opt.atoms)

            # TRIGGER 3D DISPLAY FOR FIRE STEP
            visualize_live_atoms(fire_opt.atoms, mol_id=mol_id, step=step, fmax=cur_fmax, energy_ev=cur_e, phase="FIRE")
        except Exception as e:
            flush_print(f"   [FIRE Callback Warning] {e}")

    fire_opt.attach(live_fire_callback, interval=1)

    try:
        fire_opt.run(fmax=target_fmax, steps=max_steps)
    except Exception as e:
        flush_print(f"[FIRE ERROR] Optimization stopped unexpectedly: {e}")

    atoms.set_positions(best_atoms.get_positions())
    ase_write(fire_best, atoms)

    final_fire_forces = atoms.get_forces()
    final_fire_fmax = float(np.max(np.linalg.norm(final_fire_forces, axis=1)))

    return {
        "atoms": atoms,
        "fire_iterations": fire_opt.nsteps,
        "fmax_eV_A": final_fire_fmax,
        "converged": final_fire_fmax <= target_fmax
    }


def cap_displacement(step, max_displacement):
    step = np.asarray(step, dtype=float).reshape(-1)
    norm = np.linalg.norm(step)
    if not np.isfinite(norm):
        raise RuntimeError("Non-finite displacement vector.")
    if norm <= max_displacement:
        return step, float(norm)
    return step * (max_displacement / norm), float(max_displacement)

def run_persistent_pbfgs(atoms, mol_id, history, target_fmax=OPTIMIZER_FMAX):
    flush_print("\n" + "=" * 90)
    flush_print(f"pBFGS PHASE START | {mol_id}")
    flush_print("=" * 90)

    pbfgs_log, _ = create_pbfgs_log(mol_id)
    pbfgs_checkpoint_path = os.path.join(OUTPUT_DIR, f"{mol_id}_pbfgs_checkpoint.xyz")
    pbfgs_best_path = os.path.join(OUTPUT_DIR, f"{mol_id}_pbfgs_best_checkpoint.xyz")

    attach_pyscf_calculator(atoms, basis=OPT_BASIS, xc=OPT_XC)
    state, resumed = load_pbfgs_state(mol_id, atoms)

    if resumed:
        flush_print(f"[RESUME] Loaded Hessian state matrix from iteration {state['iteration']}")
        atoms = state["atoms"]
        H = state["H"]
        start_iter = state["iteration"] + 1
        best_atoms = state["best_atoms"]
        best_energy_ha = state["best_energy_ha"]
        best_fmax = state["best_fmax"]
        attach_pyscf_calculator(atoms, basis=OPT_BASIS, xc=OPT_XC)
        energy_ev, energy_ha = energy_ha_from_atoms(atoms)
        forces, fmax, rms = get_forces_and_metrics(atoms)
    else:
        energy_ev, energy_ha = energy_ha_from_atoms(atoms)
        forces, fmax, rms = get_forces_and_metrics(atoms)
        ncoord = 3 * len(atoms)
        H = np.eye(ncoord, dtype=float) * PBFGS_INITIAL_HESSIAN_SCALE
        start_iter = 1
        best_atoms = atoms.copy()
        best_energy_ha = float(energy_ha)
        best_fmax = float(fmax)

    completed_iterations = start_iter - 1
    consecutive_rejects = 0
    MAX_CONSECUTIVE_REJECTS = 15

    for iteration in range(start_iter, PBFGS_MAX_STEPS + 1):
        completed_iterations = iteration
        gradient = -np.asarray(forces, dtype=float).reshape(-1)

        if not finite_array(gradient):
            break

        raw_step = -H.dot(gradient)
        effective_max_disp = PBFGS_MAX_DISPLACEMENT / (1.5 ** consecutive_rejects)
        step_flat, step_norm = cap_displacement(raw_step, effective_max_disp)

        old_positions = np.asarray(atoms.get_positions(), dtype=float).copy()
        new_positions = old_positions + step_flat.reshape((-1, 3))
        atoms.set_positions(new_positions)

        geometry_ok, _, _ = atoms_geometry_qc(atoms)
        if not geometry_ok:
            atoms.set_positions(old_positions)
            consecutive_rejects += 1
            if consecutive_rejects % 3 == 0:
                ncoord = 3 * len(atoms)
                H = np.eye(ncoord, dtype=float) * PBFGS_INITIAL_HESSIAN_SCALE
            pbfgs_log_write(pbfgs_log, iteration, energy_ev, energy_ha, fmax, rms, step_norm, 0.0, "H_RESET" if consecutive_rejects % 3 == 0 else "NOT_UPDATED", "GEOMETRY_REJECT")
            if consecutive_rejects >= MAX_CONSECUTIVE_REJECTS:
                break
            continue

        try:
            trial_energy_ev, trial_energy_ha = energy_ha_from_atoms(atoms)
            trial_forces, trial_fmax, trial_rms = get_forces_and_metrics(atoms)
        except Exception:
            atoms.set_positions(old_positions)
            consecutive_rejects += 1
            if consecutive_rejects % 3 == 0:
                ncoord = 3 * len(atoms)
                H = np.eye(ncoord, dtype=float) * PBFGS_INITIAL_HESSIAN_SCALE
            pbfgs_log_write(pbfgs_log, iteration, energy_ev, energy_ha, fmax, rms, step_norm, 0.0, "H_RESET" if consecutive_rejects % 3 == 0 else "NOT_UPDATED", "CALC_FAILED")
            if consecutive_rejects >= MAX_CONSECUTIVE_REJECTS:
                break
            continue

        trial_gradient = -trial_forces.reshape(-1)
        s = np.asarray(new_positions, dtype=float).reshape(-1) - np.asarray(old_positions, dtype=float).reshape(-1)
        y = trial_gradient - gradient
        sy = float(np.dot(s, y))

        if trial_energy_ha <= energy_ha + 1.0e-8:
            accepted = True
            consecutive_rejects = 0
            energy_ev, energy_ha = trial_energy_ev, trial_energy_ha
            forces, fmax, rms = trial_forces, trial_fmax, trial_rms

            if np.isfinite(sy) and sy > PBFGS_CURVATURE_TOL:
                rho = 1.0 / sy
                I = np.eye(len(s))
                V = I - rho * np.outer(s, y)
                H_new = V @ H @ V.T + rho * np.outer(s, s)
                if finite_array(H_new):
                    H = 0.5 * (H_new + H_new.T)
                    update_status = "UPDATED"
                else:
                    update_status = "H_NONFINITE"
            else:
                update_status = "SKIPPED"
        else:
            atoms.set_positions(old_positions)
            accepted = False
            consecutive_rejects += 1
            if consecutive_rejects % 3 == 0:
                ncoord = 3 * len(atoms)
                H = np.eye(ncoord, dtype=float) * PBFGS_INITIAL_HESSIAN_SCALE
                update_status = "H_RESET"
            else:
                update_status = "REJECTED"

        if np.isfinite(energy_ha) and energy_ha < best_energy_ha:
            best_energy_ha = energy_ha
            best_fmax = fmax
            best_atoms = atoms.copy()
            save_xyz(best_atoms, pbfgs_best_path)

        save_xyz(atoms, pbfgs_checkpoint_path)
        save_pbfgs_state(mol_id, atoms, H, iteration, best_atoms, best_energy_ha, best_fmax)
        pbfgs_log_write(pbfgs_log, iteration, energy_ev, energy_ha, fmax, rms, step_norm, sy, update_status, "ACCEPT" if accepted else "REJECT")

        # TRIGGER 3D DISPLAY FOR pBFGS STEP
        visualize_live_atoms(atoms, mol_id=mol_id, step=iteration, fmax=fmax, energy_ev=energy_ev, phase="pBFGS")

        if consecutive_rejects >= MAX_CONSECUTIVE_REJECTS or fmax <= target_fmax:
            break

    atoms.set_positions(best_atoms.get_positions())
    save_xyz(atoms, pbfgs_best_path)
    pbfgs_log.close()

    return {
        "atoms": atoms,
        "pBFGS_iterations": int(completed_iterations),
        "fmax_eV_A": float(best_fmax),
        "energy_ha": float(best_energy_ha)
    }


def run_lbfgs_phase(atoms, mol_id, basis=FINAL_BASIS, xc=FINAL_XC, target_fmax=TARGET_FMAX):
    flush_print("\n" + "=" * 90)
    flush_print(f"LBFGS REFINEMENT PHASE START | {mol_id}")
    flush_print("=" * 90)

    lbfgs_best = os.path.join(OUTPUT_DIR, f"{mol_id}_lbfgs_best.xyz")
    final_chk_path = os.path.join(OUTPUT_DIR, f"{mol_id}_final.chk")
    log_path = os.path.join(OUTPUT_DIR, f"{mol_id}_lbfgs_opt.log")

    calc = PySCFCalculator(basis=basis, xc=xc, chkfile=final_chk_path)
    atoms.calc = calc

    if os.path.exists(lbfgs_best):
        atoms = ase_read(lbfgs_best)
        atoms.calc = calc

    lbfgs_opt = LBFGS(atoms, logfile=log_path, maxstep=0.04)

    def live_progress_callback():
        cur_forces = lbfgs_opt.atoms.get_forces()
        cur_fmax = float(np.max(np.linalg.norm(cur_forces, axis=1)))
        cur_e = lbfgs_opt.atoms.get_potential_energy()
        ase_write(lbfgs_best, atoms)

        # TRIGGER 3D DISPLAY FOR SECONDARY LBFGS STEP
        visualize_live_atoms(atoms, mol_id=mol_id, step=lbfgs_opt.nsteps, fmax=cur_fmax, energy_ev=cur_e, phase="LBFGS")

    lbfgs_opt.attach(live_progress_callback, interval=1)

    try:
        lbfgs_opt.run(fmax=target_fmax, steps=300)
    except Exception as e:
        flush_print(f"[LBFGS ERROR] Optimization stopped unexpectedly: {e}")

    post_lbfgs_fmax = float(np.max(np.linalg.norm(atoms.get_forces(), axis=1)))
    ase_write(lbfgs_best, atoms)

    return {
        "atoms": atoms,
        "fmax_eV_A": post_lbfgs_fmax,
        "converged": post_lbfgs_fmax <= target_fmax,
        "lbfgs_best_path": lbfgs_best
    }

# =============================================================================
# 6. SAFE FINAL ELECTRONIC QC WITH SOSCF FALLBACK
# =============================================================================

def run_final_electronic_qc(atoms, mol_id):
    flush_print("\n" + "=" * 90)
    flush_print(f"FINAL ELECTRONIC QC | {mol_id}")
    flush_print("=" * 90)

    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()

    atom_str = "; ".join(
        f"{sym} {pos[0]:.8f} {pos[1]:.8f} {pos[2]:.8f}"
        for sym, pos in zip(symbols, positions)
    )

    basis_spec, ecp_spec = get_dynamic_basis_and_ecp(
        symbols,
        FINAL_BASIS
    )

    mol_kwargs = {
        "atom": atom_str,
        "basis": basis_spec,
        "verbose": 0,
        "max_memory": 2000,
        "spin": 0,
        "charge": 0,
    }

    if ecp_spec is not None:
        mol_kwargs["ecp"] = ecp_spec

    mol = gto.M(**mol_kwargs)

    mol.atom = atom_str
    mol.basis = basis_spec

    if ecp_spec is not None:
        mol.ecp = ecp_spec

    mol.spin = 0
    mol.charge = 0
    mol.build()

    mf = dft.RKS(mol)
    mf.xc = FINAL_XC
    mf.max_cycle = 250
    mf.level_shift = 0.4
    mf.damp = 0.5
    mf.conv_tol = 1e-7

    flush_print("[FINAL QC] Running primary PySCF DFT calculation...")
    energy_ha = mf.kernel()
    scf_converged = bool(mf.converged)

    if not scf_converged:
        flush_print("[FINAL QC WARNING] Standard DIIS failed to converge. Launching Newton SOSCF solver...")
        user_soscf = scf.newton(mf)
        user_soscf.max_cycle = 100
        energy_ha = user_soscf.kernel()
        scf_converged = bool(user_soscf.converged)
        mf = user_soscf

    grad_tensor = mf.nuc_grad_method().kernel()
    forces = -grad_tensor * HARTREE_TO_EV / BOHR_TO_ANG
    fmax = calculate_fmax(forces)
    rms = calculate_rms_force(forces)

    homo_eV, lumo_eV, gap_eV = np.nan, np.nan, np.nan
    homo_index, lumo_index = 0, 0

    if scf_converged and mf.mo_energy is not None:
        mo_energies = np.sort(np.asarray(mf.mo_energy))
        nelectron = mol.nelectron
        n_occ = nelectron // 2

        if 0 < n_occ < len(mo_energies):
            homo_index = n_occ - 1
            lumo_index = n_occ

            homo_ha = float(mo_energies[homo_index])
            lumo_ha = float(mo_energies[lumo_index])

            homo_eV = homo_ha * HARTREE_TO_EV
            lumo_eV = lumo_ha * HARTREE_TO_EV
            gap_eV = (lumo_ha - homo_ha) * HARTREE_TO_EV

    energy_ev = energy_ha * HARTREE_TO_EV

    geometry_ok, geometry_status, min_dist = atoms_geometry_qc(atoms)

    finite_orbitals = np.isfinite(homo_eV) and np.isfinite(lumo_eV)
    positive_gap = np.isfinite(gap_eV) and (gap_eV > 0.0)
    electronic_qc = scf_converged and finite_orbitals and positive_gap

    flush_print(f"Final Energy (Ha) : {energy_ha:.12f}")
    flush_print(f"HOMO-LUMO Gap (eV): {gap_eV:.8f}")
    flush_print(f"SCF Converged     : {scf_converged}")
    flush_print(f"Electronic QC     : {electronic_qc}")

    return {
        "energy_eV": float(energy_ev) if np.isfinite(energy_ev) else np.nan,
        "energy_Ha": float(energy_ha) if np.isfinite(energy_ha) else np.nan,
        "homo_index": int(homo_index),
        "lumo_index": int(lumo_index),
        "homo_eV": float(homo_eV) if np.isfinite(homo_eV) else np.nan,
        "lumo_eV": float(lumo_eV) if np.isfinite(lumo_eV) else np.nan,
        "gap_eV": float(gap_eV) if np.isfinite(gap_eV) else np.nan,
        "HOMO_LUMO_Gap_eV": float(gap_eV) if np.isfinite(gap_eV) else np.nan,
        "scf_converged": bool(scf_converged),
        "finite_orbitals": bool(finite_orbitals),
        "positive_gap": bool(positive_gap),
        "electronic_qc": bool(electronic_qc),
        "fmax_eV_A": float(fmax),
        "rms_force_eV_A": float(rms),
        "geometry_qc": bool(geometry_ok),
        "geometry_status": str(geometry_status),
        "minimum_distance_A": float(min_dist),
        "basis": FINAL_BASIS,
        "xc": FINAL_XC
    }

# =============================================================================
# 7. LOG GENERATION AND DRIVER
# =============================================================================

def write_final_log(mol_id, smiles, history, final_electronic, final_geometry_path):
    path = os.path.join(OUTPUT_DIR, f"{mol_id}_final_log.txt")
    fire_summary = history.get("FIRE_Summary", {})
    pbfgs_summary = history.get("pBFGS_Summary", {})
    fire_iterations = int(fire_summary.get("FIRE_Iterations", 0))
    pbfgs_iterations = int(pbfgs_summary.get("pBFGS_Iterations", 0))

    gap_val = final_electronic.get("gap_eV", final_electronic.get("HOMO_LUMO_Gap_eV", np.nan))
    energy_ha = final_electronic.get("energy_Ha", np.nan)
    scf_converged = final_electronic.get("scf_converged", False)
    electronic_qc = final_electronic.get("electronic_qc", False)
    geometry_qc = final_electronic.get("geometry_qc", False)
    fmax_val = final_electronic.get("fmax_eV_A", np.nan)

    with open(path, "w", encoding="utf-8") as f:
        f.write("=" * 100 + "\n")
        f.write(f"FINAL HYPERVALENT SULFUR RESULT | {mol_id}\n")
        f.write("=" * 100 + "\n")
        f.write(f"Generated                    : {now_string()}\n")
        f.write(f"Molecule ID                  : {mol_id}\n")
        f.write(f"SMILES                       : {smiles}\n\n")
        f.write("OPTIMIZATION\n" + "-"*100 + "\n")
        f.write(f"Total FIRE Iterations        : {fire_iterations}\n")
        f.write(f"Total pBFGS Iterations       : {pbfgs_iterations}\n")
        f.write(f"Target Fmax (eV/A)           : {STANDARD_FMAX:.10f}\n")
        f.write(f"Geometry QC Pass             : {geometry_qc}\n\n")
        f.write("FINAL ELECTRONIC PROPERTIES\n" + "-"*100 + "\n")
        f.write(f"Energy (Hartree)             : {energy_ha:.12f}\n")
        f.write(f"HOMO-LUMO gap (eV)           : {gap_val:.12f}\n")
        f.write(f"SCF Converged                : {scf_converged}\n")
        f.write(f"Electronic QC Pass           : {electronic_qc}\n")

        final_pass = geometry_qc and scf_converged and (fmax_val < STANDARD_FMAX)
        f.write(f"\nFINAL_STATUS = {'SUCCESS' if final_pass else 'PARTIAL_CONVERGENCE'}\n")
    return path

def generate_etkdg_atoms(mol_id, smiles_str):
    """
    Generates initial 3D molecular geometry using RDKit's ETKDGv3 embedding
    and converts it into an ASE Atoms object.
    """
    try:
        symbols, positions = embed_raw(smiles_str)
        atoms = Atoms(symbols=symbols, positions=positions)
        return atoms
    except Exception as e:
        flush_print(f"[ETKDG ERROR] Failed to embed 3D coordinates for {mol_id}: {e}")
        return None



##Normal route: - **Standard path** (48 of 50 molecules): RDKit ETKDGv3 embedding &rarr; UFF pre-relaxation &rarr; BFGS geometry optimization (ASE) &rarr; B3LYP/STO-3G single-point DFT (PySCF).

In [ ]:
import os
import time
import json
import io
import numpy as np
import pandas as pd
from IPython.display import display, clear_output, HTML
import py3Dmol

from rdkit import Chem
from rdkit.Chem import AllChem
from ase import Atoms
from ase.io import read as ase_read, write as ase_write
from ase.optimize import LBFGS, BFGS

# Ensure output directory exists for checkpoints, logs, and geometries
OUTPUT_DIR = "research_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Global settings matching your script environment
STANDARD_FMAX = 0.05
HARTREE_TO_EV = 27.211386245988

def visualize_live_atoms(atoms, mol_id="MOL", step=0, fmax=None, energy_ev=None, phase="", width=500, height=400):
    """
    3D view implementation with strict scalar extraction for metrics formatting.
    Prevents formatting crashes when `fmax` or `energy_ev` are passed as numpy ndarrays.
    """
    import io
    from IPython.display import display, clear_output, HTML
    import py3Dmol
    from ase.io import write as ase_write
    import numpy as np

    # --------------------------------------------------------------------------
    # SCALAR CONVERSION FOR LOG & FORMAT STRINGS
    # --------------------------------------------------------------------------
    def force_scalar(val):
        if val is None:
            return None
        if isinstance(val, (list, np.ndarray)) or hasattr(val, "ndim"):
            val_arr = np.asarray(val)
            if val_arr.size == 0:
                return 0.0
            if val_arr.size == 1:
                return float(val_arr.item())
            elif val_arr.ndim > 1:
                # Calculate maximum force norm across 3D atomic vectors
                return float(np.max(np.linalg.norm(val_arr, axis=1)))
            else:
                return float(np.max(np.abs(val_arr)))
        return float(val)

    scalar_energy = force_scalar(energy_ev)
    scalar_fmax = force_scalar(fmax)

    # --------------------------------------------------------------------------
    # Interactive 3D Canvas Generation Engine
    # --------------------------------------------------------------------------
    buf = io.StringIO()
    ase_write(buf, atoms, format="xyz")
    xyz_data = buf.getvalue()

    clear_output(wait=True)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(xyz_data, "xyz")
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.zoomTo()

    # Safely format single scalar float values
    energy_str = f"{scalar_energy:.4f}" if scalar_energy is not None else "N/A"
    fmax_str = f"{scalar_fmax:.4f}" if scalar_fmax is not None else "N/A"

    print(f"[{mol_id}] Phase: {phase} | Step {step} | Energy: {energy_str} eV | Fmax: {fmax_str} eV/Å", flush=True)
    view.show()


def run_normal(mol_id, smiles, basis="sto-3g", fmax=STANDARD_FMAX, max_steps=120):
    """
    Enhanced standard pipeline for organic molecules with automatic ECP routing
    for heavy halogens (like Br, I) to prevent artificial PARTIAL_CONVERGENCE.
    """
    t0 = time.time()

    final_accepted = os.path.join(OUTPUT_DIR, f"{mol_id}_final_accepted_geometry.xyz")
    lbfgs_best = os.path.join(OUTPUT_DIR, f"{mol_id}_lbfgs_best_geometry.xyz")
    traj_xyz_path = os.path.join(OUTPUT_DIR, f"{mol_id}_optimization_trajectory.xyz")
    log_path = os.path.join(OUTPUT_DIR, f"{mol_id}_opt.log")
    chk_path = os.path.join(OUTPUT_DIR, f"{mol_id}_opt.chk")

    atoms = None
    phase = "OPTIMIZE"

    if os.path.exists(final_accepted):
        atoms = ase_read(final_accepted)
        phase = "FINAL_QC"
    elif os.path.exists(lbfgs_best):
        atoms = ase_read(lbfgs_best)
        # Dynamic basis check here to avoid calculator configuration drops
        symbols = atoms.get_chemical_symbols()
        calc = PySCFCalculator(basis=basis, use_rhf_seed=True, chkfile=chk_path)
        atoms.calc = calc
        try:
            current_fmax = float(np.max(np.linalg.norm(atoms.get_forces(), axis=1)))
        except Exception:
            current_fmax = 999.0
        phase = "FINAL_QC" if current_fmax <= fmax else "OPTIMIZE"
    else:
        phase = "OPTIMIZE"

    log_mode = "a" if phase == "FINAL_QC" or os.path.exists(lbfgs_best) else "w"
    log_file = open(log_path, log_mode, encoding="utf-8")
    if log_mode == "w":
        log_file.write(f"=== Optimization Log for {mol_id} ===\nSMILES: {smiles}\n\n")
        log_file.flush()

    if phase == "OPTIMIZE" and not os.path.exists(lbfgs_best):
        symbols, positions = embed_raw(smiles)
        atoms = Atoms(symbols=symbols, positions=positions)
    elif phase == "OPTIMIZE" and os.path.exists(lbfgs_best):
        symbols = atoms.get_chemical_symbols()

    if phase == "OPTIMIZE":
        calc = PySCFCalculator(basis=basis, use_rhf_seed=True, chkfile=chk_path)
        atoms.calc = calc

    def live_tracking_callback(optimizer_obj, phase_name):
        try:
            cur_forces = atoms.get_forces()
            cur_fmax = float(np.max(np.linalg.norm(cur_forces, axis=1)))
            cur_e = atoms.get_potential_energy()
            step_num = optimizer_obj.get_number_of_steps()

            ase_write(lbfgs_best, atoms, format="xyz")
            ase_write(traj_xyz_path, atoms, append=True, format="xyz")

            log_file.write(f"{phase_name} | Step: {step_num:3d} | Energy: {cur_e:14.6f} eV | Fmax: {cur_fmax:10.6f} eV/Å\n")
            log_file.flush()

            visualize_live_atoms(atoms, mol_id=mol_id, step=step_num, fmax=cur_fmax, energy_ev=cur_e, phase=phase_name)
        except Exception as e:
            print(f"Tracking error at iteration step: {e}", flush=True)

    # --------------------------------------------------------------------------
    # Phase 3: Geometry Optimization Blocks
    # --------------------------------------------------------------------------
    if phase == "OPTIMIZE":
        # 1. Primary optimization with LBFGS - Target tightened to absorb drift
        opt = LBFGS(atoms, logfile=None, maxstep=0.08)
        opt.attach(lambda: live_tracking_callback(opt, "LBFGS_PRIMARY"), interval=1)
        try:
            opt.run(fmax=0.043, steps=max_steps)
        except Exception as e:
            print(f"[{mol_id}] Primary LBFGS step warning: {e}", flush=True)

        # 2. Secondary refinement with conservative step BFGS if forces plateaued
        current_force = float(np.max(np.linalg.norm(atoms.get_forces(), axis=1)))
        if current_force > 0.043:
            log_file.write(f"\n[FALLBACK TRIGGERED] Force threshold residual {current_force:.4f} > 0.043. Launching BFGS...\n")
            log_file.flush()

            opt_fallback = BFGS(atoms, logfile=None, maxstep=0.04)
            opt_fallback.attach(lambda: live_tracking_callback(opt_fallback, "BFGS_FALLBACK"), interval=1)
            try:
                opt_fallback.run(fmax=0.043, steps=40)
            except Exception as e:
                print(f"[{mol_id}] Fallback BFGS step warning: {e}", flush=True)


        ase_write(final_accepted, atoms, format="extxyz")
    else:
        symbols = atoms.get_chemical_symbols()

    log_file.close()
    elapsed = time.time() - t0
    final_pos = atoms.get_positions()

    # CRITICAL FIX: Evaluate the final single-point utilizing get_dynamic_basis_and_ecp
    # instead of forcing a rigid, inaccurate sto-3g representation for large halogens
    atom_str = "; ".join(f"{s} {p[0]:.8f} {p[1]:.8f} {p[2]:.8f}" for s, p in zip(symbols, final_pos))
    basis_spec, ecp_spec = get_dynamic_basis_and_ecp(symbols, basis)

    mol = gto.M(atom=atom_str, basis=basis_spec, verbose=0, max_memory=1500)
    if ecp_spec is not None:
        mol.ecp = ecp_spec
        mol.build()

    mf_seed = scf.RHF(mol)
    mf_seed.conv_tol = 1e-6
    mf_seed.max_cycle = 100
    mf_seed.kernel()
    dm0 = mf_seed.make_rdm1() if mf_seed.converged else None

    mf = dft.RKS(mol)
    mf.xc = 'b3lyp'
    mf.conv_tol = 1e-6
    mf.grids.level = 1

    e = mf.kernel(dm0=dm0) if dm0 is not None else mf.kernel()
    g = mf.nuc_grad_method().kernel()

    forces = -g * HARTREE_TO_EV / BOHR_TO_ANG
    final_max_force = float(np.max(np.linalg.norm(forces, axis=1)))

    homo_idx = mol.nelectron // 2 - 1
    lumo_idx = homo_idx + 1
    homo = float(mf.mo_energy[homo_idx] * HARTREE_TO_EV)
    lumo = float(mf.mo_energy[lumo_idx] * HARTREE_TO_EV)

    visualize_live_atoms(atoms, mol_id=mol_id, step="FINAL", fmax=final_max_force, energy_ev=e*HARTREE_TO_EV, phase="CONVERGED_QC")

    return {
        "Molecule_ID": mol_id,
        "SMILES": smiles,
        "Total_Electronic_Energy_Hartree": float(mf.e_tot),
        "HOMO_eV": homo,
        "LUMO_eV": lumo,
        "HOMO_LUMO_Gap_eV": float(lumo - homo),
        "SCF_Converged": bool(mf.converged),
        "Max_Force_eV_per_A": final_max_force,
        "Method": f"ETKDGv3+LBFGS ({basis})",
        "Status": "SUCCESS" if final_max_force <= fmax else "PARTIAL_CONVERGENCE",
        "Elapsed_sec": float(elapsed),
        "Check_Files": {
            "Wavefunction_Checkpoint": chk_path,
            "Geometry_XYZ": final_accepted,
            "Intermediate_Cache_XYZ": lbfgs_best,
            "Optimization_Log": log_path
        }
    }

## Hypervalent route: FIRE + persistent-Hessian BFGS
- **Hypervalent-sulfur path** (`MOL_0007`, `MOL_0035`): RDKit's UFF force field cannot parameterize
  hexavalent sulfur and collapses the 3D geometry (verified: it places multiple fluorine atoms on top
  of each other). These two molecules skip UFF entirely, starting from the raw ETKDGv3 embedding, and
  are relaxed using a staged **FIRE optimizer (60 steps) followed by constrained BFGS polishing**, with
  an accurate final single-point at B3LYP/6-31g(d). Both repeatedly plateaued around 0.048-0.045 eV/Å
  rather than the 0.05 eV/Å threshold used for standard molecules - this is reported honestly rather
  than concealed (see `Max_Force_eV_per_A` and `Status` in the output).
This is the combination that actually achieved genuine convergence (force < 0.05 eV/A) for MOL_0007, after every single-optimizer approach (FIRE alone, plain steepest descent, BFGS reset on every restart) plateaued far short of the target. The key insight: FIRE handles the large initial forces efficiently, but a method with persisted curvature information (the inverse-Hessian approximation, saved to disk after every step) is needed to navigate the remaining, much harder terrain near the true minimum - which for this molecule turned out to involve climbing over a local rise in the force landscape (force rose from ~0.94 to a peak of ~1.45 before finally descending smoothly to convergence). A fresh, un-persisted optimizer restarted after every interruption cannot make this kind of progress, since it loses its learned curvature every time.

In [ ]:
import os
import time
import json
import io
import numpy as np
from IPython.display import display, clear_output, HTML
import py3Dmol

from ase import Atoms
from ase.io import read as ase_read, write as ase_write
from pyscf import gto, dft, scf

# Assumes these globals and helper functions are defined in your environment:
# OUTPUT_DIR, FINAL_BASIS, FINAL_XC, TARGET_FMAX, HARTREE_TO_EV
# flush_print, create_history, save_history, attach_pyscf_calculator
# embed_raw, run_fire_phase, run_persistent_pbfgs, run_lbfgs_phase
# get_dynamic_basis_and_ecp


def visualize_live_atoms(atoms, mol_id="MOL", step=0, fmax=None, energy_ev=None, phase="", width=500, height=400):
    """
    3D view implementation with strict scalar extraction for metrics formatting.
    Prevents formatting crashes when `fmax` or `energy_ev` are passed as numpy ndarrays.
    """
    import io
    from IPython.display import display, clear_output, HTML
    import py3Dmol
    from ase.io import write as ase_write
    import numpy as np

    # --------------------------------------------------------------------------
    # SCALAR CONVERSION FOR LOG & FORMAT STRINGS
    # --------------------------------------------------------------------------
    def force_scalar(val):
        if val is None:
            return None
        if isinstance(val, (list, np.ndarray)) or hasattr(val, "ndim"):
            val_arr = np.asarray(val)
            if val_arr.size == 0:
                return 0.0
            if val_arr.size == 1:
                return float(val_arr.item())
            elif val_arr.ndim > 1:
                # Calculate maximum force norm across 3D atomic vectors
                return float(np.max(np.linalg.norm(val_arr, axis=1)))
            else:
                return float(np.max(np.abs(val_arr)))
        return float(val)

    scalar_energy = force_scalar(energy_ev)
    scalar_fmax = force_scalar(fmax)

    # --------------------------------------------------------------------------
    # Interactive 3D Canvas Generation Engine
    # --------------------------------------------------------------------------
    buf = io.StringIO()
    ase_write(buf, atoms, format="xyz")
    xyz_data = buf.getvalue()

    clear_output(wait=True)

    view = py3Dmol.view(width=width, height=height)
    view.addModel(xyz_data, "xyz")
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.zoomTo()

    # Safely format single scalar float values
    energy_str = f"{scalar_energy:.4f}" if scalar_energy is not None else "N/A"
    fmax_str = f"{scalar_fmax:.4f}" if scalar_fmax is not None else "N/A"

    print(f"[{mol_id}] Phase: {phase} | Step {step} | Energy: {energy_str} eV | Fmax: {fmax_str} eV/Å", flush=True)
    view.show()

def run_hypervalent(mol_id, smiles, basis=FINAL_BASIS, phase="FIRE"):
    """
    Hypervalent sulfur production pipeline with live 3D visualization
    and single-point density reuse.
    """
    overall_start_time = time.time()

    flush_print("\n" + "#" * 100)
    flush_print(f"HYPERVALENT PRODUCTION RUN | {mol_id}")
    flush_print("#" * 100)

    # Output checkpoint paths (.xyz).
    fire_ckpt = os.path.join(OUTPUT_DIR, f"{mol_id}_fire_checkpoint.xyz")
    fire_best = os.path.join(OUTPUT_DIR, f"{mol_id}_fire_best_checkpoint.xyz")
    pbfgs_ckpt = os.path.join(OUTPUT_DIR, f"{mol_id}_pbfgs_checkpoint.xyz")
    pbfgs_best = os.path.join(OUTPUT_DIR, f"{mol_id}_pbfgs_best_checkpoint.xyz")
    lbfgs_best = os.path.join(OUTPUT_DIR, f"{mol_id}_lbfgs_best.xyz")
    final_accepted = os.path.join(
        OUTPUT_DIR, f"{mol_id}_final_accepted_geometry.xyz"
    )
    xyz_path = final_accepted
    log_path = os.path.join(OUTPUT_DIR, f"{mol_id}_opt.log")

    # Isolated wavefunction checkpoint paths (.chk).
    opt_chk_path = os.path.join(OUTPUT_DIR, f"{mol_id}_opt.chk")
    final_chk_path = os.path.join(OUTPUT_DIR, f"{mol_id}_final.chk")

    history = create_history(mol_id, smiles)
    save_history(history, mol_id)

    fire_result = {"fire_iterations": 0}
    pbfgs_result = {"pBFGS_iterations": 0}
    atoms = None

    # ==========================================================================
    # Phase 0: Detect checkpoints and choose a resume route.
    # ==========================================================================
    TIGHT_FMAX = 0.03

    if os.path.exists(final_accepted):
        flush_print(
            "[ROUTING] Found final accepted geometry checkpoint. "
            "Jumping directly to FINAL_QC."
        )
        atoms = ase_read(final_accepted)

        # CRITICAL FIX: Attach a working calculator frame to the reloaded
        # checkpoint geometry so get_forces() won't throw a RuntimeError
        attach_pyscf_calculator(atoms, basis=FINAL_BASIS, xc=FINAL_XC)

        phase = "FINAL_QC"

    elif os.path.exists(lbfgs_best):
        atoms = ase_read(lbfgs_best)
        attach_pyscf_calculator(atoms, basis=FINAL_BASIS, xc=FINAL_XC)

        try:
            lbfgs_forces = atoms.get_forces()
            lbfgs_fmax = float(
                np.max(np.linalg.norm(lbfgs_forces, axis=1))
            )
            lbfgs_energy = atoms.get_potential_energy()
        except Exception:
            lbfgs_fmax = 999.0
            lbfgs_energy = None

        visualize_live_atoms(
            atoms,
            mol_id=mol_id,
            step="RELOAD",
            fmax=lbfgs_fmax,
            energy_ev=lbfgs_energy,
            phase="ROUTING_LBFGS",
        )

        if lbfgs_fmax <= TIGHT_FMAX:
            flush_print(
                f"[ROUTING] Found fully converged LBFGS checkpoint "
                f"(Fmax = {lbfgs_fmax:.4f} <= {TIGHT_FMAX}). "
                "Jumping to FINAL_QC."
            )
            phase = "FINAL_QC"
        else:
            flush_print(
                f"[ROUTING] Found partially converged LBFGS checkpoint "
                f"(Fmax = {lbfgs_fmax:.4f} > {TIGHT_FMAX}). "
                "Resuming LBFGS refinement."
            )
            phase = "LBFGS"

    elif os.path.exists(pbfgs_best) or os.path.exists(pbfgs_ckpt):
        chosen_ckpt = (
            pbfgs_best if os.path.exists(pbfgs_best) else pbfgs_ckpt
        )
        flush_print(
            f"[ROUTING] Found pBFGS checkpoint "
            f"({os.path.basename(chosen_ckpt)}). Resuming from pBFGS."
        )
        atoms = ase_read(chosen_ckpt)
        phase = "PBFGS"

    elif os.path.exists(fire_best):
        flush_print(
            "[ROUTING] Found completed FIRE best checkpoint. "
            "Skipping FIRE and advancing directly to pBFGS."
        )
        atoms = ase_read(fire_best)
        phase = "PBFGS"

    else:
        phase = "FIRE"

    # Phase 1: FIRE relaxation.
    if phase == "FIRE":
        if os.path.exists(fire_ckpt):
            atoms = ase_read(fire_ckpt)
            flush_print("[FIRE] Resuming FIRE from existing checkpoint.")
        elif os.path.exists(fire_best):
            atoms = ase_read(fire_best)
            flush_print("[FIRE] Found completed FIRE best checkpoint.")
        else:
            flush_print(
                "[FIRE] Starting fresh optimization from raw ETKDG geometry."
            )
            symbols, positions = embed_raw(smiles)
            atoms = Atoms(symbols=symbols, positions=positions)

        fire_result = run_fire_phase(atoms, mol_id, history)

        if os.path.exists(fire_best):
            atoms = ase_read(fire_best)

        phase = "PBFGS"

    # Phase 2: Persistent BFGS (pBFGS) relaxation.
    if phase == "PBFGS":
        if os.path.exists(pbfgs_ckpt):
            atoms = ase_read(pbfgs_ckpt)
            flush_print("[pBFGS] Resuming persistent-BFGS from last checkpoint.")
        elif os.path.exists(pbfgs_best):
            atoms = ase_read(pbfgs_best)
            flush_print("[pBFGS] Resuming persistent-BFGS from best checkpoint.")
        elif os.path.exists(fire_best):
            atoms = ase_read(fire_best)
            flush_print(
                "[pBFGS] Bootstrapping persistent-BFGS from FIRE best checkpoint."
            )

        pbfgs_result = run_persistent_pbfgs(atoms, mol_id, history)

        if isinstance(pbfgs_result, dict) and "atoms" in pbfgs_result:
            atoms = pbfgs_result["atoms"]

    # Phase 3: Validate geometry and perform secondary refinement if needed.
    # --------------------------------------------------------------------------
    # Phase 3: Geometry Convergence Validation & Secondary Refinement (FIXED)
    # --------------------------------------------------------------------------
    if phase == "FINAL_QC":
        flush_print(" -> [FINAL_QC Mode] Geometry already verified/loaded. Skipping Phase 3 LBFGS refinement.")
    elif phase == "LBFGS":
        flush_print(" -> [RESUME] Directly launching Phase 3 LBFGS refinement...")
        lbfgs_result = run_lbfgs_phase(atoms, mol_id, basis=FINAL_BASIS, xc=FINAL_XC, target_fmax=TIGHT_FMAX)
        atoms = lbfgs_result["atoms"]
    else:
        # CRITICAL REPAIR: Force a validation check utilizing the FINAL high-precision basis set
        # to capture and eliminate any grid drift before saving the final check coordinates
        attach_pyscf_calculator(atoms, basis=FINAL_BASIS, xc=FINAL_XC)
        try:
            current_fmax = float(np.max(np.linalg.norm(atoms.get_forces(), axis=1)))
        except Exception:
            current_fmax = 999.0

        if current_fmax <= TIGHT_FMAX:
            flush_print(f" -> Geometry tightly converged under high-basis (Fmax = {current_fmax:.4f} eV/Å <= {TIGHT_FMAX}). Skipping LBFGS Phase.")
        else:
            flush_print(f" -> Phase 3: High-basis force drift detected ({current_fmax:.4f} eV/Å > {TIGHT_FMAX}). Launching explicit LBFGS refinement...")
            lbfgs_result = run_lbfgs_phase(atoms, mol_id, basis=FINAL_BASIS, xc=FINAL_XC, target_fmax=TIGHT_FMAX)
            atoms = lbfgs_result["atoms"]

    # Save final accepted geometry.
    final_accepted = os.path.join(
        OUTPUT_DIR, f"{mol_id}_final_accepted_geometry.xyz"
    )

    try:
        ase_write(final_accepted, atoms)
        flush_print(
            f"[SUCCESS] Saved final converged structure to: "
            f"{os.path.basename(final_accepted)}"
        )
    except Exception as exc:
        flush_print(f"[ERROR] Failed to write final .xyz file: {exc}")

    # Phase 4: Final single-point QC (PySCF).
    symbols = atoms.get_chemical_symbols()
    positions = atoms.get_positions()

    atom_string = "; ".join(
        f"{symbol} {position[0]:.8f} "
        f"{position[1]:.8f} {position[2]:.8f}"
        for symbol, position in zip(symbols, positions)
    )

    basis_spec, ecp_spec = get_dynamic_basis_and_ecp(symbols, basis)

    pyscf_mol = gto.Mole()
    pyscf_mol.atom = atom_string
    pyscf_mol.basis = basis_spec

    if ecp_spec is not None:
        pyscf_mol.ecp = ecp_spec

    pyscf_mol.spin = 0
    pyscf_mol.verbose = 0
    pyscf_mol.build()

    mf = dft.RKS(pyscf_mol)
    mf.xc = FINAL_XC
    mf.chkfile = final_chk_path
    mf.max_cycle = 200
    mf.level_shift = 0.3

    dm0 = None

    if os.path.exists(final_chk_path):
        try:
            loaded_dm = mf.from_chk(final_chk_path)

            if (
                hasattr(loaded_dm, "shape")
                and loaded_dm.shape[0] == pyscf_mol.nao
            ):
                dm0 = loaded_dm
        except Exception:
            dm0 = None

    energy_ha = (
        mf.kernel(dm0=dm0)
        if dm0 is not None
        else mf.kernel()
    )
    scf_converged = bool(mf.converged)

    if not scf_converged:
        user_soscf = scf.newton(mf)
        energy_ha = user_soscf.kernel()
        scf_converged = bool(
            user_soscf.converged
            if hasattr(user_soscf, "converged")
            else mf.converged
        )
        mf = user_soscf

    homo_ev, lumo_ev, gap_ev = None, None, None

    if scf_converged and mf.mo_energy is not None:
        mo_energies = np.sort(np.asarray(mf.mo_energy))
        n_occ = pyscf_mol.nelectron // 2

        homo_ha = float(mo_energies[n_occ - 1])
        lumo_ha = float(mo_energies[n_occ])

        homo_ev = homo_ha * HARTREE_TO_EV
        lumo_ev = lumo_ha * HARTREE_TO_EV
        gap_ev = lumo_ev - homo_ev

    # Evaluate final geometry and result properties.
    final_forces = atoms.get_forces()
    final_fmax = float(np.max(np.linalg.norm(final_forces, axis=1)))
    elapsed = time.time() - overall_start_time

    is_success = (
        scf_converged
        and final_fmax <= TARGET_FMAX
        and gap_ev is not None
        and gap_ev > 0
    )

    result_json_path = os.path.join(
        OUTPUT_DIR, f"{mol_id}_final_result.json"
    )

    result = {
        "Molecule_ID": str(mol_id),
        "SMILES": str(smiles),
        "Total_Electronic_Energy_Hartree": (
            float(energy_ha) if scf_converged else None
        ),
        "HOMO_eV": float(homo_ev) if homo_ev is not None else None,
        "LUMO_eV": float(lumo_ev) if lumo_ev is not None else None,
        "HOMO_LUMO_Gap_eV": float(gap_ev) if gap_ev is not None else None,
        "SCF_Converged": scf_converged,
        "Max_Force_eV_per_A": final_fmax,
        "Method": f"ETKDGv3+FIRE+pBFGS ({basis})",
        "Status": "SUCCESS" if is_success else "PARTIAL_CONVERGENCE",
        "Elapsed_sec": float(elapsed),
        "Final_Result_JSON": result_json_path,
        "Check_Files": {
            "Wavefunction_Checkpoint": opt_chk_path,
            "Geometry_XYZ": final_accepted,
            "Intermediate_Cache_XYZ": lbfgs_best,
            "Optimization_Log": log_path
        }
    }

    with open(result_json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    # Display the final structure.
    visualize_live_atoms(
        atoms,
        mol_id=mol_id,
        step="FINAL",
        fmax=final_fmax,
        energy_ev=(
            float(energy_ha) * HARTREE_TO_EV
            if scf_converged
            else None
        ),
        phase="PRODUCTION_CONVERGED",
    )

    return result

## Geometry validation and repair engine (G3) - four modules
Cells below write four small Python files next to the notebook (`geom_common.py`, `g3_engine.py`, `geometry_gate.py`, `validate_geometry.py`) and load them in a **self-test cell**.
Run them **once, in order**, after the optimizer cells and before the driver. Every optimized geometry is validated against its SMILES (bond lengths, angles, hypervalent shells, planarity, clashes); only geometries that fail are rebuilt.


In [ ]:
%%writefile geom_common.py
#!/usr/bin/env python3
"""
geom_common.py  -  single source of truth shared by the G3 repair engine and
the validator.  Contains:

  * element data (covalent / vdW radii, valence electrons)
  * context-aware bond-length targets          -> target_length()
  * VSEPR classification of every atom         -> classify_atom()
  * hypervalent shell templates + site rules   -> TEMPLATES, assign_sites()
  * ideal angles for ordinary (non-hypervalent) atoms -> ideal_angle_nonhyper()
  * check_geometry(): the full geometry audit used by both programs
"""
import itertools
import math

import numpy as np
from rdkit import Chem

# --------------------------------------------------------------------------
# Element data
# --------------------------------------------------------------------------
COV_R = {"H": 0.31, "B": 0.85, "C": 0.76, "N": 0.71, "O": 0.66, "F": 0.57,
         "Si": 1.11, "P": 1.07, "S": 1.05, "Cl": 1.02, "Br": 1.20,
         "I": 1.39, "Xe": 1.40, "As": 1.19, "Se": 1.20, "Te": 1.38}
VDW_R = {"H": 1.10, "B": 1.92, "C": 1.70, "N": 1.55, "O": 1.52, "F": 1.47,
         "Si": 2.10, "P": 1.80, "S": 1.80, "Cl": 1.75, "Br": 1.85,
         "I": 1.98, "Xe": 2.16, "As": 1.85, "Se": 1.90, "Te": 2.06}
VALENCE_E = {"H": 1, "B": 3, "C": 4, "N": 5, "O": 6, "F": 7, "Si": 4, "P": 5,
             "S": 6, "Cl": 7, "Br": 7, "I": 7, "Xe": 8, "As": 5, "Se": 6,
             "Te": 6}
EN = {"H": 2.2, "B": 2.0, "C": 2.5, "N": 3.0, "O": 3.5, "F": 4.0, "Si": 1.9,
      "P": 2.2, "S": 2.5, "Cl": 3.0, "Br": 2.8, "I": 2.5, "Xe": 2.6,
      "As": 2.2, "Se": 2.5, "Te": 2.1}


def cov(sym):
    return COV_R.get(sym, 0.90)


def vdw(sym):
    return VDW_R.get(sym, 1.90)


# --------------------------------------------------------------------------
# Small atom helpers
# --------------------------------------------------------------------------
def hyb(atom):
    """1 = sp, 2 = sp2/aromatic, 3 = sp3 (from bond orders, not RDKit tags)."""
    orders = [b.GetBondTypeAsDouble() for b in atom.GetBonds()]
    if any(o >= 3 for o in orders) or sum(o == 2 for o in orders) >= 2:
        return 1
    if any(o in (1.5, 2) for o in orders):
        return 2
    return 3


def has_double_to(atom, symbols=("O", "N", "S")):
    for b in atom.GetBonds():
        if b.GetBondTypeAsDouble() == 2:
            if b.GetOtherAtom(atom).GetSymbol() in symbols:
                return True
    return False


def is_carbonyl_c(atom):
    return atom.GetSymbol() == "C" and has_double_to(atom, ("O", "N", "S"))


def ring_of(mol, size, *atoms):
    for ring in mol.GetRingInfo().AtomRings():
        if len(ring) == size and all(a in ring for a in atoms):
            return ring
    return None


# --------------------------------------------------------------------------
# VSEPR classification
# --------------------------------------------------------------------------
def classify_atom(atom):
    """Return dict(deg, lp, domains, geometry, hyper)."""
    sym = atom.GetSymbol()
    deg = atom.GetDegree()
    ve = VALENCE_E.get(sym)
    if ve is None or deg == 0:
        return dict(deg=deg, lp=0, domains=deg, geometry="NONE", hyper=False)
    valence = sum(b.GetBondTypeAsDouble() for b in atom.GetBonds())
    if atom.GetIsAromatic():          # aromatic heteroatom (thiophene S etc.)
        valence = atom.GetTotalValence()
    lp = max(0, int(round((ve - atom.GetFormalCharge() - valence) / 2.0)))
    domains = deg + lp
    geom = {
        (6, 0): "OCT", (5, 0): "TBP", (5, 1): "SQPYR", (4, 1): "SEESAW",
        (4, 2): "SQPLANAR", (3, 2): "TSHAPE", (2, 3): "LINEAR",
    }.get((deg, lp), "NONHYPER")
    hyper = domains >= 5 and deg >= 2 and geom != "NONHYPER"
    return dict(deg=deg, lp=lp, domains=domains, geometry=geom, hyper=hyper)


def hyper_centres(mol):
    return [a.GetIdx() for a in mol.GetAtoms() if classify_atom(a)["hyper"]]


# --------------------------------------------------------------------------
# Hypervalent templates (unit vectors of the ligand sites)
# --------------------------------------------------------------------------
def _v(*a):
    v = np.array(a, float)
    return v / np.linalg.norm(v)


_c55, _s55 = math.cos(math.radians(55)), math.sin(math.radians(55))
TEMPLATES = {
    "LINEAR": np.array([_v(0, 0, 1), _v(0, 0, -1)]),
    "TSHAPE": np.array([_v(0, 0, 1), _v(0, 0, -1), _v(1, 0, 0)]),
    "SEESAW": np.array([_v(0, 0, 1), _v(0, 0, -1),
                        _v(_c55, _s55, 0), _v(_c55, -_s55, 0)]),
    "TBP": np.array([_v(0, 0, 1), _v(0, 0, -1), _v(1, 0, 0),
                     _v(-.5, math.sqrt(3) / 2, 0),
                     _v(-.5, -math.sqrt(3) / 2, 0)]),
    "OCT": np.array([_v(0, 0, 1), _v(0, 0, -1), _v(1, 0, 0), _v(-1, 0, 0),
                     _v(0, 1, 0), _v(0, -1, 0)]),
    "SQPYR": np.array([_v(0, 0, 1), _v(1, 0, 0), _v(0, 1, 0), _v(-1, 0, 0),
                       _v(0, -1, 0)]),
    "SQPLANAR": np.array([_v(1, 0, 0), _v(-1, 0, 0), _v(0, 1, 0),
                          _v(0, -1, 0)]),
}
# angular tolerance (deg) used by the validator for each shell shape
SHAPE_TOL = {"LINEAR": 12.0, "TSHAPE": 15.0, "SEESAW": 20.0, "TBP": 12.0,
             "OCT": 12.0, "SQPYR": 15.0, "SQPLANAR": 12.0}


def template_angle_matrix(name):
    t = TEMPLATES[name]
    c = np.clip(t @ t.T, -1, 1)
    return np.degrees(np.arccos(c))


def ax_score(mol, centre, nbr):
    """Higher = prefers an axial site (electronegative, single-bonded)."""
    a = mol.GetAtomWithIdx(nbr)
    b = mol.GetBondBetweenAtoms(centre, nbr)
    s = EN.get(a.GetSymbol(), 2.5)
    if b.GetBondTypeAsDouble() >= 2:
        s -= 5.0
    if a.GetSymbol() == "C" and a.GetIsAromatic():
        s -= 0.6
    return s


def assign_sites(mol, centre):
    """Return (geometry, [(neighbour_idx, site_idx), ...]) for a hypervalent
    centre following the usual VSEPR/Bent's-rule placement rules."""
    atom = mol.GetAtomWithIdx(centre)
    info = classify_atom(atom)
    geom = info["geometry"]
    nbrs = [n.GetIdx() for n in atom.GetNeighbors()]
    scores = {n: ax_score(mol, centre, n) for n in nbrs}
    if geom in ("OCT", "SQPYR"):
        order = sorted(nbrs, key=lambda n: scores[n])        # lowest first
    else:
        order = sorted(nbrs, key=lambda n: -scores[n])       # highest first
    return geom, [(n, s) for s, n in enumerate(order)]


# --------------------------------------------------------------------------
# Bond-length targets (Angstrom)
# --------------------------------------------------------------------------
_CC_SINGLE = {(3, 3): 1.53, (2, 3): 1.51, (1, 3): 1.47, (2, 2): 1.48,
              (1, 2): 1.44, (1, 1): 1.38}


def _o_has_h(atom):
    return any(n.GetSymbol() == "H" for n in atom.GetNeighbors())


def target_length(mol, bond):
    """Return (length, source) with source 'table' or 'fallback'."""
    a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
    s1, s2 = a1.GetSymbol(), a2.GetSymbol()
    key = tuple(sorted((s1, s2)))
    o = bond.GetBondTypeAsDouble()

    def other(sym):
        return a1 if a1.GetSymbol() == sym else a2

    t = None
    if key == ("C", "H"):
        c = other("C")
        t = {1: 1.06, 2: 1.08, 3: 1.09}[hyb(c)]
    elif key == ("H", "O"):
        t = 0.96
    elif key == ("H", "N"):
        t = 1.01
    elif key == ("H", "S"):
        t = 1.34
    elif key == ("C", "C"):
        if o >= 3:
            t = 1.20
        elif o == 2:
            t = 1.34
        elif o == 1.5:
            t = 1.40
            r5 = None
            for ring in mol.GetRingInfo().AtomRings():
                if (len(ring) == 5 and a1.GetIdx() in ring
                        and a2.GetIdx() in ring
                        and any(mol.GetAtomWithIdx(x).GetSymbol() == "S"
                                for x in ring)):
                    r5 = ring
            if r5 is not None:
                near_s = any(n.GetSymbol() == "S" and n.GetIdx() in r5
                             for a in (a1, a2) for n in a.GetNeighbors())
                t = 1.37 if near_s else 1.42
        else:
            h = tuple(sorted((hyb(a1), hyb(a2))))
            t = _CC_SINGLE.get(h, 1.52)
    elif key == ("C", "F"):
        c = other("C")
        nF = sum(n.GetSymbol() == "F" for n in c.GetNeighbors())
        t = 1.35 if hyb(c) == 2 else (1.33 if nF >= 2 else 1.34)
    elif key == ("C", "N"):
        c = other("C")
        t = {3: 1.16, 2: 1.28, 1.5: 1.34}.get(o)
        if t is None:
            t = 1.40 if hyb(c) == 2 else 1.47
    elif key == ("C", "O"):
        c = other("C")
        if o == 2:
            t = 1.21
        elif c.GetIsAromatic():
            t = 1.37
        elif is_carbonyl_c(c):
            t = 1.34
        elif hyb(c) == 2:
            t = 1.36
        else:
            t = 1.43
    elif key == ("C", "S"):
        c = other("C")
        t = 1.71 if o == 1.5 else (1.60 if o == 2 else
                                    (1.77 if hyb(c) == 2 else 1.82))
    elif key == ("C", "Cl"):
        t = 1.74 if hyb(other("C")) == 2 else 1.78
    elif key == ("Br", "C"):
        t = 1.90 if hyb(other("C")) == 2 else 1.95
    elif key == ("C", "I"):
        t = 2.10 if hyb(other("C")) == 2 else 2.16
    elif key == ("C", "P"):
        t = 1.80
    elif key == ("C", "Si"):
        t = 1.87
    elif key == ("C", "Xe"):
        t = 2.35
    elif key == ("F", "S"):
        t = 1.58
    elif key == ("Cl", "S"):
        t = 2.05
    elif key == ("O", "S"):
        t = 1.45 if o == 2 else 1.65
    elif key == ("N", "S"):
        t = 1.65
    elif key == ("S", "S"):
        t = 2.05
    elif key == ("F", "P"):
        t = 1.58
    elif key == ("Cl", "P"):
        t = 2.04
    elif key == ("O", "P"):
        t = 1.48 if o == 2 else 1.60
    elif key == ("F", "I"):
        t = 2.00
    elif key == ("Cl", "I"):
        t = 2.45
    elif key == ("I", "O"):
        if o == 2:
            t = 1.80
        else:
            t = 2.00 if _o_has_h(other("O")) else 2.15
    elif key == ("I", "N"):
        t = 2.10
    elif key == ("F", "Xe"):
        t = 2.00
    elif key == ("N", "O"):
        oo, nn = other("O"), other("N")
        if o == 2 or oo.GetFormalCharge() < 0 or nn.GetFormalCharge() > 0:
            t = 1.23                       # nitro / N-oxide type
        else:
            t = 1.40
    elif key == ("N", "N"):
        t = 1.45 if o == 1 else 1.25
    if t is not None:
        return float(t), "table"
    return float(cov(s1) + cov(s2)), "fallback"


# --------------------------------------------------------------------------
# Ideal angles for ordinary atoms
# --------------------------------------------------------------------------
def _interior_5(mol, centre, ring):
    a = mol.GetAtomWithIdx(centre)
    if a.GetSymbol() == "S":
        return 92.0
    if any(mol.GetAtomWithIdx(n.GetIdx()).GetSymbol() == "S" and
           n.GetIdx() in ring for n in a.GetNeighbors()):
        return 111.5
    return 112.5


def ideal_angle_nonhyper(mol, c, i, j):
    a = mol.GetAtomWithIdx(c)
    n = a.GetDegree()
    h = hyb(a)
    if n == 2:
        if h == 1:
            return 180.0
        ring = None
        for r in mol.GetRingInfo().AtomRings():
            if len(r) == 5 and c in r and i in r and j in r:
                ring = r
        if ring is not None:
            return _interior_5(mol, c, ring)
        sym = a.GetSymbol()
        nb = [mol.GetAtomWithIdx(x) for x in (i, j)]
        if sym == "O":
            if any(x.GetSymbol() == "H" for x in nb):
                return 106.0
            if any(is_carbonyl_c(x) for x in nb):
                return 116.0
            if any(x.GetIsAromatic() for x in nb):
                return 118.0
            return 109.5
        if sym == "S":
            return 100.0
        return 120.0 if h == 2 else 109.5
    if n == 3:
        if h == 2:
            for r in mol.GetRingInfo().AtomRings():
                if len(r) == 5 and c in r:
                    inside = [x for x in (i, j) if x in r]
                    interior = _interior_5(mol, c, r)
                    if len(inside) == 2:
                        return interior
                    if len(inside) == 1:
                        return (360.0 - interior) / 2.0
            if is_carbonyl_c(a):
                bi = mol.GetBondBetweenAtoms(c, i).GetBondTypeAsDouble()
                bj = mol.GetBondBetweenAtoms(c, j).GetBondTypeAsDouble()
                return 114.0 if (bi == 1 and bj == 1) else 123.0
            return 120.0
        return 107.0 if a.GetSymbol() == "N" else 109.5
    if n == 4:
        return 109.47
    return 109.47


# --------------------------------------------------------------------------
# Full geometry audit
# --------------------------------------------------------------------------
def _angle(p, c, q):
    u, v = p - c, q - c
    cs = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12)
    return math.degrees(math.acos(float(np.clip(cs, -1, 1))))


def check_geometry(mol, X, tol_abs=0.10, tol_rel=0.06, ang_tol=15.0,
                   clash_factor=0.65, plane_tol=0.06, hyper_scale=1.0):
    """Audit coordinates X (N x 3, AddHs(SMILES) order) against the SMILES
    topology.  Returns a dict with every check and an overall verdict."""
    X = np.asarray(X, float)
    n = mol.GetNumAtoms()
    sym = [a.GetSymbol() for a in mol.GetAtoms()]
    dm = Chem.GetDistanceMatrix(mol)
    res = {"bonds": [], "angles": [], "hyper": [], "planarity": [],
           "clashes": [], "extra_bonds": [], "missing_bonds": []}

    # 1. bonds
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        t, src = target_length(mol, b)
        d = float(np.linalg.norm(X[i] - X[j]))
        err = d - t
        ok = abs(err) <= max(tol_abs, tol_rel * t)
        res["bonds"].append(dict(i=i, j=j, ei=sym[i], ej=sym[j],
                                 order=b.GetBondTypeAsDouble(), target=t,
                                 source=src, dist=d, err=err, ok=ok))

    # 2. topology perceived from the coordinates
    bonded = {tuple(sorted((b.GetBeginAtomIdx(), b.GetEndAtomIdx())))
              for b in mol.GetBonds()}
    for i in range(n):
        for j in range(i + 1, n):
            d = np.linalg.norm(X[i] - X[j])
            near = d <= 1.25 * (cov(sym[i]) + cov(sym[j]))
            if near and (i, j) not in bonded and dm[i][j] > 2:
                res["extra_bonds"].append(dict(i=i, j=j, ei=sym[i], ej=sym[j],
                                               dist=float(d)))
            if (i, j) in bonded and d > 1.35 * (cov(sym[i]) + cov(sym[j])) \
                    + (0.3 if "I" in (sym[i], sym[j]) or "Xe" in (sym[i], sym[j]) else 0):
                res["missing_bonds"].append(dict(i=i, j=j, ei=sym[i],
                                                 ej=sym[j], dist=float(d)))

    # 3. angles
    hyper = set(hyper_centres(mol))
    for a in mol.GetAtoms():
        c = a.GetIdx()
        nb = [x.GetIdx() for x in a.GetNeighbors()]
        if len(nb) < 2:
            continue
        if c in hyper:
            info = classify_atom(a)
            T = template_angle_matrix(info["geometry"])
            k = len(nb)
            A = np.zeros((k, k))
            for p in range(k):
                for q in range(k):
                    if p != q:
                        A[p, q] = _angle(X[nb[p]], X[c], X[nb[q]])
            perms = np.array(list(itertools.permutations(range(T.shape[0]),
                                                         k)))
            Tp = T[perms[:, :, None], perms[:, None, :]]
            dev = np.abs(A[None] - Tp)
            dev[:, np.arange(k), np.arange(k)] = 0
            worst = dev.reshape(len(perms), -1).max(axis=1)
            best = int(np.argmin(worst))
            tol = SHAPE_TOL[info["geometry"]] * hyper_scale
            res["hyper"].append(dict(
                centre=c, element=sym[c], geometry=info["geometry"],
                ligands=nb, max_dev=float(worst[best]), tol=tol,
                min_angle=float(A[A > 0].min()) if k > 1 else 0.0,
                max_angle=float(A.max()), ok=bool(worst[best] <= tol)))
            continue
        for p in range(len(nb)):
            for q in range(p + 1, len(nb)):
                ideal = ideal_angle_nonhyper(mol, c, nb[p], nb[q])
                ang = _angle(X[nb[p]], X[c], X[nb[q]])
                # neighbours that are hypervalent centres bend more freely
                tol = ang_tol + (5.0 if (nb[p] in hyper or nb[q] in hyper)
                                 else 0.0)
                res["angles"].append(dict(
                    i=nb[p], c=c, j=nb[q], ec=sym[c], ideal=ideal, angle=ang,
                    err=ang - ideal, ok=abs(ang - ideal) <= tol))

    # 4. planarity: aromatic rings (+ substituents) and sp2 centres
    for ring in mol.GetRingInfo().AtomRings():
        if not all(mol.GetAtomWithIdx(x).GetIsAromatic() for x in ring):
            continue
        members = set(ring)
        for x in ring:
            for nb in mol.GetAtomWithIdx(x).GetNeighbors():
                members.add(nb.GetIdx())
        P = X[sorted(members)]
        P0 = P - P.mean(axis=0)
        normal = np.linalg.svd(P0)[2][-1]
        dev = float(np.abs(P0 @ normal).max())
        res["planarity"].append(dict(kind="aromatic ring", atoms=list(ring),
                                     max_dev=dev, ok=dev <= plane_tol * 1.5))
    for a in mol.GetAtoms():
        c = a.GetIdx()
        if c in hyper or a.GetDegree() != 3 or hyb(a) != 2:
            continue
        if a.GetSymbol() not in ("C", "N"):
            continue
        nb = [x.GetIdx() for x in a.GetNeighbors()]
        s = (_angle(X[nb[0]], X[c], X[nb[1]]) + _angle(X[nb[0]], X[c], X[nb[2]])
             + _angle(X[nb[1]], X[c], X[nb[2]]))
        res["planarity"].append(dict(kind="sp2 centre", atoms=[c],
                                     max_dev=360.0 - s, ok=(360.0 - s) <= 6.0))

    # 5. clashes (graph distance >= 3)
    for i in range(n):
        for j in range(i + 1, n):
            if dm[i][j] < 3:
                continue
            d = float(np.linalg.norm(X[i] - X[j]))
            lim = clash_factor * (vdw(sym[i]) + vdw(sym[j]))
            if d < lim:
                res["clashes"].append(dict(i=i, j=j, ei=sym[i], ej=sym[j],
                                           dist=d, limit=lim))

    # summary
    errs = [abs(r["err"]) for r in res["bonds"]]
    res["max_err"] = max(errs) if errs else 0.0
    res["mean_err"] = float(np.mean(errs)) if errs else 0.0
    res["rms_err"] = float(np.sqrt(np.mean(np.square(errs)))) if errs else 0.0
    res["bad_bonds"] = [r for r in res["bonds"] if not r["ok"]]
    res["bad_angles"] = [r for r in res["angles"] if not r["ok"]]
    res["bad_hyper"] = [r for r in res["hyper"] if not r["ok"]]
    res["bad_planar"] = [r for r in res["planarity"] if not r["ok"]]
    reasons = []
    if res["bad_bonds"]:
        reasons.append(f"{len(res['bad_bonds'])} bond(s) outside tolerance")
    if res["extra_bonds"]:
        reasons.append(f"{len(res['extra_bonds'])} unintended contact(s)")
    if res["missing_bonds"]:
        reasons.append(f"{len(res['missing_bonds'])} detached bond(s)")
    if res["bad_angles"]:
        reasons.append(f"{len(res['bad_angles'])} angle(s) outside tolerance")
    if res["bad_hyper"]:
        reasons.append(f"{len(res['bad_hyper'])} hypervalent shell(s) distorted")
    if res["bad_planar"]:
        reasons.append(f"{len(res['bad_planar'])} planarity violation(s)")
    if res["clashes"]:
        reasons.append(f"{len(res['clashes'])} steric clash(es)")
    res["reasons"] = reasons
    res["verdict"] = "PASS" if not reasons else "FAIL"
    return res


### G3 geometry engine (`g3_engine.py`)

In [ ]:
%%writefile g3_engine.py
#!/usr/bin/env python3
"""
======================================================================
G3  -  UNIFIED SMILES-AUTHORITATIVE GEOMETRY ENGINE
        (normal + hypervalent + heavy-element molecules, one code path)
======================================================================
Drop-in replacement for the G1 / G2 `repair_structure()` function.

Pipeline (identical for every molecule - no element-based branching):
  1. SMILES -> RDKit molecule with explicit H.  The SMILES is the ONLY
     authority on atoms, bonds and bond orders.
  2. A target is built for every bond (geom_common.target_length), every
     angle (VSEPR templates for hypervalent centres, hybridisation rules for
     everything else) and every planarity / ester-torsion requirement.
  3. Distance-geometry embedding using a bounds matrix that contains those
     targets (so even S(VI), I(III), I(V), P(V), Xe are embedded correctly).
  4. Restrained minimisation (analytic gradients): bonds + angles +
     planarity + 1-4 + steric repulsion.
  5. Multi-start; every candidate is audited by geom_common.check_geometry
     and the best PASSING one is kept.  If none passes, status is
     UNRESOLVED (never a silent "PASS").

The input XYZ (optional) is only used to (a) verify the composition and
(b) carry extra columns across when the atom mapping is exact.
Geometries are idealised (template-built + restrained); they are starting
structures for ASE/PySCF, not relaxed minima.
"""
import math
import os
import sys

import numpy as np
from scipy.optimize import minimize
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.DistanceGeometry import DoTriangleSmoothing

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import geom_common as gco  # noqa: E402  (never name it "gc": that shadows the stdlib garbage-collector module)

RDLogger.DisableLog("rdApp.*")

ENGINE_NONBONDED_FACTOR = 0.72     # repulsion onset (validator uses 0.65)


# --------------------------------------------------------------------------
# I/O helpers (same file formats as G1)
# --------------------------------------------------------------------------
def parse_extended_xyz(filename):
    if not filename or not os.path.isfile(filename):
        raise FileNotFoundError(f"File not found: {filename}")
    with open(filename, "r", encoding="utf-8", errors="ignore") as fh:
        raw = fh.readlines()
    declared = int(raw[0].split()[0])
    comment = raw[1].strip() if len(raw) > 1 else ""
    sym, xyz, extra = [], [], []
    for line in raw[2:]:
        p = line.split()
        if len(p) < 4:
            continue
        try:
            xyz.append([float(p[1]), float(p[2]), float(p[3])])
        except ValueError:
            continue
        sym.append(p[0])
        extra.append(p[4:])
    if len(sym) != declared:
        raise ValueError(f"XYZ atom count mismatch: header={declared}, "
                         f"parsed={len(sym)}")
    return comment, sym, np.asarray(xyz, float), extra


def write_extended_xyz(filename, comment, symbols, coords, extra_cols=None):
    with open(filename, "w", encoding="utf-8") as fh:
        fh.write(f"{len(symbols)}\n{comment}\n")
        for k, (s, p) in enumerate(zip(symbols, coords)):
            ex = ""
            if extra_cols is not None and extra_cols[k]:
                ex = " " + " ".join(extra_cols[k])
            fh.write(f"{s:<3s} {p[0]:16.8f} {p[1]:16.8f} {p[2]:16.8f}{ex}\n")


def write_sdf(mol, coords, filename):
    out = Chem.Mol(mol)
    out.RemoveAllConformers()
    conf = Chem.Conformer(out.GetNumAtoms())
    for i, p in enumerate(coords):
        conf.SetAtomPosition(i, tuple(float(v) for v in p))
    out.AddConformer(conf, assignId=True)
    w = Chem.SDWriter(filename)
    w.write(out)
    w.close()


def write_2d_png(smiles, filename):
    try:
        m = Chem.MolFromSmiles(smiles)
        AllChem.Compute2DCoords(m)
        d = rdMolDraw2D.MolDraw2DCairo(900, 700)
        d.DrawMolecule(m)
        d.FinishDrawing()
        with open(filename, "wb") as fh:
            fh.write(d.GetDrawingText())
    except Exception as exc:                    # drawing is cosmetic only
        print(f"[WARN] 2D drawing skipped: {exc}")


def build_reference_molecule(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")
    Chem.SanitizeMol(mol)
    return Chem.AddHs(mol)


# --------------------------------------------------------------------------
# Restraint specification
# --------------------------------------------------------------------------
def d13(a, b, theta_deg):
    th = math.radians(theta_deg)
    return math.sqrt(max(a * a + b * b - 2 * a * b * math.cos(th), 1e-6))


def d14(l12, l23, l34, th2, th3, phi):
    """1-4 distance for torsion phi (deg) given lengths and the two angles."""
    t2, t3, ph = map(math.radians, (th2, th3, phi))
    p1 = np.array([l12 * math.cos(math.pi - t2), l12 * math.sin(math.pi - t2), 0])
    p2 = np.array([0.0, 0.0, 0.0])
    p3 = np.array([l23, 0.0, 0.0])
    p4 = p3 + l34 * np.array([-math.cos(math.pi - t3),
                              math.sin(math.pi - t3) * math.cos(ph),
                              math.sin(math.pi - t3) * math.sin(ph)])
    # p1 placed so angle at p2 is th2:   fix orientation explicitly
    p1 = np.array([l12 * math.cos(t2), l12 * math.sin(t2), 0.0])
    p4 = p3 + l34 * np.array([-math.cos(t3), math.sin(t3) * math.cos(ph),
                              math.sin(t3) * math.sin(ph)])
    return float(np.linalg.norm(p1 - p4))


class Spec:
    def __init__(self, mol):
        self.mol = mol
        n = mol.GetNumAtoms()
        self.n = n
        self.sym = [a.GetSymbol() for a in mol.GetAtoms()]
        self.dm = Chem.GetDistanceMatrix(mol)

        self.bond_t = {}
        for b in mol.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            self.bond_t[(min(i, j), max(i, j))] = gco.target_length(mol, b)[0]
        self.L = lambda i, j: self.bond_t[(min(i, j), max(i, j))]

        # angles --------------------------------------------------------
        self.angles = []              # (i, c, j, theta)
        self.hyper = gco.hyper_centres(mol)
        for a in mol.GetAtoms():
            c = a.GetIdx()
            nb = [x.GetIdx() for x in a.GetNeighbors()]
            if len(nb) < 2:
                continue
            if c in self.hyper:
                geom, assign = gco.assign_sites(mol, c)
                T = gco.template_angle_matrix(geom)
                for (n1, s1) in assign:
                    for (n2, s2) in assign:
                        if n1 < n2:
                            self.angles.append((n1, c, n2, float(T[s1, s2])))
            else:
                for p in range(len(nb)):
                    for q in range(p + 1, len(nb)):
                        th = gco.ideal_angle_nonhyper(mol, c, nb[p], nb[q])
                        self.angles.append((nb[p], c, nb[q], th))
        self.ang_lookup = {(min(i, j), c, max(i, j)): th
                           for i, c, j, th in self.angles}

        # planarity quartets (centre, a, b, d) --------------------------
        self.planar = []
        for a in mol.GetAtoms():
            c = a.GetIdx()
            if c in self.hyper or a.GetDegree() != 3:
                continue
            if a.GetSymbol() in ("C", "N") and gco.hyb(a) == 2:
                nb = [x.GetIdx() for x in a.GetNeighbors()]
                self.planar.append((c, nb[0], nb[1], nb[2]))

        # ester / carboxylate:  X-O-C(=O) coplanar and cis ----------------
        self.d14 = []                 # (i, l, dist)
        for a in mol.GetAtoms():
            if a.GetSymbol() != "O" or a.GetDegree() != 2:
                continue
            o = a.GetIdx()
            for cbn in a.GetNeighbors():
                if not gco.is_carbonyl_c(cbn):
                    continue
                x = [k.GetIdx() for k in a.GetNeighbors()
                     if k.GetIdx() != cbn.GetIdx()][0]
                c = cbn.GetIdx()
                od = [k.GetIdx() for k in cbn.GetNeighbors()
                      if k.GetSymbol() == "O" and
                      mol.GetBondBetweenAtoms(c, k.GetIdx())
                      .GetBondTypeAsDouble() == 2]
                if not od:
                    continue
                od = od[0]
                self.planar.append((o, x, c, od))
                th_o = self.ang_lookup[(min(x, c), o, max(x, c))]
                th_c = self.ang_lookup[(min(o, od), c, max(o, od))]
                self.d14.append((x, od, d14(self.L(x, o), self.L(o, c),
                                            self.L(c, od), th_o, th_c, 0.0)))

        # aromatic-ring planarity quartets: ring torsion 0 / exo 180 ----------
        self.d14_ring = []
        for ring in mol.GetRingInfo().AtomRings():
            if not all(mol.GetAtomWithIdx(k).GetIsAromatic() for k in ring):
                continue
            m = len(ring)
            for k in range(m):
                p = [ring[(k + t) % m] for t in range(4)]
                th2 = self.ang_lookup.get((min(p[0], p[2]), p[1], max(p[0], p[2])))
                th3 = self.ang_lookup.get((min(p[1], p[3]), p[2], max(p[1], p[3])))
                if th2 is None or th3 is None:
                    continue
                self.d14_ring.append((p[0], p[3], d14(
                    self.L(p[0], p[1]), self.L(p[1], p[2]), self.L(p[2], p[3]),
                    th2, th3, 0.0)))
            for r in ring:
                for ex in mol.GetAtomWithIdx(r).GetNeighbors():
                    e = ex.GetIdx()
                    if e in ring:
                        continue
                    for pn in mol.GetAtomWithIdx(r).GetNeighbors():
                        p = pn.GetIdx()
                        if p not in ring:
                            continue
                        for pp in mol.GetAtomWithIdx(p).GetNeighbors():
                            q = pp.GetIdx()
                            if q not in ring or q == r:
                                continue
                            th2 = self.ang_lookup.get((min(e, p), r, max(e, p)))
                            th3 = self.ang_lookup.get((min(r, q), p, max(r, q)))
                            if th2 is None or th3 is None:
                                continue
                            self.d14_ring.append((e, q, d14(
                                self.L(e, r), self.L(r, p), self.L(p, q),
                                th2, th3, 180.0)))

        # steric repulsion pairs ---------------------------------------
        pairs = []
        for i in range(n):
            for j in range(i + 1, n):
                if self.dm[i][j] >= 3:
                    dmin = ENGINE_NONBONDED_FACTOR * (gco.vdw(self.sym[i]) +
                                                      gco.vdw(self.sym[j]))
                    pairs.append((i, j, dmin))
        self.rep = pairs

        self._arrays()

    def _arrays(self):
        bi = np.array([k[0] for k in self.bond_t], int)
        bj = np.array([k[1] for k in self.bond_t], int)
        self.bi, self.bj = bi, bj
        self.bd = np.array([self.bond_t[k] for k in self.bond_t])
        A = np.array([[a[0], a[1], a[2]] for a in self.angles], int).reshape(-1, 3)
        self.ai, self.ac, self.aj = A[:, 0], A[:, 1], A[:, 2]
        th = np.array([a[3] for a in self.angles])
        self.acos0 = np.cos(np.radians(th))
        self.alin = th >= 175.0
        self.asin2 = np.maximum(np.sin(np.radians(th)) ** 2, 1e-3)
        P = np.array(self.planar, int).reshape(-1, 4)
        self.pc, self.pa, self.pb, self.pd = P[:, 0], P[:, 1], P[:, 2], P[:, 3]
        D = self.d14 + self.d14_ring
        self.ti = np.array([d[0] for d in D], int)
        self.tj = np.array([d[1] for d in D], int)
        self.td = np.array([d[2] for d in D])
        R = self.rep
        self.ri = np.array([r[0] for r in R], int)
        self.rj = np.array([r[1] for r in R], int)
        self.rd = np.array([r[2] for r in R])


# --------------------------------------------------------------------------
# Energy + analytic gradient
# --------------------------------------------------------------------------
def energy_grad(x, sp, w):
    X = x.reshape(-1, 3)
    g = np.zeros_like(X)
    E = 0.0

    d = X[sp.bi] - X[sp.bj]
    r = np.linalg.norm(d, axis=1) + 1e-12
    dr = r - sp.bd
    E += w["b"] * np.sum(dr ** 2)
    gb = (2 * w["b"] * dr / r)[:, None] * d
    np.add.at(g, sp.bi, gb)
    np.add.at(g, sp.bj, -gb)

    if len(sp.ai):
        u = X[sp.ai] - X[sp.ac]
        v = X[sp.aj] - X[sp.ac]
        nu = np.linalg.norm(u, axis=1) + 1e-12
        nv = np.linalg.norm(v, axis=1) + 1e-12
        c = np.sum(u * v, axis=1) / (nu * nv)
        c = np.clip(c, -1, 1)
        E_lin = w["a"] * (1.0 + c)
        E_gen = w["a"] * (c - sp.acos0) ** 2 / sp.asin2
        dE = np.where(sp.alin, w["a"], 2 * w["a"] * (c - sp.acos0) / sp.asin2)
        E += np.sum(np.where(sp.alin, E_lin, E_gen))
        dci = v / (nu * nv)[:, None] - (c / nu ** 2)[:, None] * u
        dcj = u / (nu * nv)[:, None] - (c / nv ** 2)[:, None] * v
        np.add.at(g, sp.ai, dE[:, None] * dci)
        np.add.at(g, sp.aj, dE[:, None] * dcj)
        np.add.at(g, sp.ac, -dE[:, None] * (dci + dcj))

    if len(sp.pc):
        a = X[sp.pa] - X[sp.pc]
        b = X[sp.pb] - X[sp.pc]
        dd = X[sp.pd] - X[sp.pc]
        V = np.sum(a * np.cross(b, dd), axis=1)
        E += w["p"] * np.sum(V ** 2)
        f = (2 * w["p"] * V)[:, None]
        ga, gb_, gd = f * np.cross(b, dd), f * np.cross(dd, a), f * np.cross(a, b)
        np.add.at(g, sp.pa, ga)
        np.add.at(g, sp.pb, gb_)
        np.add.at(g, sp.pd, gd)
        np.add.at(g, sp.pc, -(ga + gb_ + gd))

    if len(sp.ti):
        d = X[sp.ti] - X[sp.tj]
        r = np.linalg.norm(d, axis=1) + 1e-12
        dr = r - sp.td
        E += w["t"] * np.sum(dr ** 2)
        gt = (2 * w["t"] * dr / r)[:, None] * d
        np.add.at(g, sp.ti, gt)
        np.add.at(g, sp.tj, -gt)

    if len(sp.ri):
        d = X[sp.ri] - X[sp.rj]
        r = np.linalg.norm(d, axis=1) + 1e-12
        m = r < sp.rd
        if m.any():
            dr = (r - sp.rd)[m]
            E += w["r"] * np.sum(dr ** 2)
            gr = (2 * w["r"] * dr / r[m])[:, None] * d[m]
            np.add.at(g, sp.ri[m], gr)
            np.add.at(g, sp.rj[m], -gr)
    return E, g.ravel()


def refine(X0, sp):
    x = X0.ravel().copy()
    stages = [dict(b=100, a=40, p=60, t=30, r=30),
              dict(b=400, a=150, p=200, t=100, r=60),
              dict(b=1500, a=500, p=600, t=300, r=100)]
    for w in stages:
        r = minimize(energy_grad, x, args=(sp, w), jac=True, method="L-BFGS-B",
                     options=dict(maxiter=4000, ftol=1e-14, gtol=1e-8))
        x = r.x
    E, _ = energy_grad(x, sp, stages[-1])
    return x.reshape(-1, 3), float(E)


# --------------------------------------------------------------------------
# Embedding
# --------------------------------------------------------------------------
def embed(mol, sp, seed):
    n = mol.GetNumAtoms()
    try:
        bm = rdDistGeom.GetMoleculeBoundsMatrix(mol)
        for (i, j), t in sp.bond_t.items():
            tol = 0.04 if "I" not in (sp.sym[i], sp.sym[j]) else 0.08
            bm[i, j], bm[j, i] = t + tol, t - tol
        for (i, c, j, th) in sp.angles:
            dd = d13(sp.L(i, c), sp.L(c, j), th)
            a, b = min(i, j), max(i, j)
            wd = 0.10 if th < 175 else 0.05
            bm[a, b], bm[b, a] = dd + wd, max(dd - wd, 0.5)
        if DoTriangleSmoothing(bm):
            ps = rdDistGeom.EmbedParameters()
            ps.randomSeed = int(seed)
            ps.useRandomCoords = True
            ps.useExpTorsionAnglePrefs = False
            ps.useBasicKnowledge = False
            ps.SetBoundsMat(bm)
            m2 = Chem.Mol(mol)
            if rdDistGeom.EmbedMolecule(m2, ps) == 0:
                return np.asarray(m2.GetConformer().GetPositions(), float)
    except Exception:
        pass
    rng = np.random.default_rng(seed)
    return rng.normal(scale=2.0, size=(n, 3))


# --------------------------------------------------------------------------
# Optional input mapping (only to carry extra columns across)
# --------------------------------------------------------------------------
def map_input(mol, xs, xc):
    """Exact graph mapping of XYZ atoms to SMILES atoms, or None."""
    if sorted(xs) != sorted(a.GetSymbol() for a in mol.GetAtoms()):
        raise ValueError("Composition of the XYZ does not match the SMILES: "
                         f"{sorted(xs)} vs "
                         f"{sorted(a.GetSymbol() for a in mol.GetAtoms())}")
    if list(xs) == [a.GetSymbol() for a in mol.GetAtoms()]:
        # same order: accept only if the perceived connectivity agrees
        ok = True
        for b in mol.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            if np.linalg.norm(xc[i] - xc[j]) > 1.35 * (gco.cov(xs[i]) +
                                                        gco.cov(xs[j])) + 0.4:
                ok = False
        if ok:
            return list(range(len(xs)))
    q = Chem.RWMol()
    for s in xs:
        q.AddAtom(Chem.Atom(s))
    for i in range(len(xs)):
        for j in range(i + 1, len(xs)):
            if np.linalg.norm(xc[i] - xc[j]) <= 1.25 * (gco.cov(xs[i]) +
                                                         gco.cov(xs[j])):
                q.AddBond(i, j, Chem.BondType.SINGLE)
    ref = Chem.RWMol(mol)
    for a in ref.GetAtoms():
        a.SetIsAromatic(False)
        a.SetFormalCharge(0)
        a.SetNoImplicit(True)
    for b in ref.GetBonds():
        b.SetBondType(Chem.BondType.SINGLE)
        b.SetIsAromatic(False)
    qm, rm = q.GetMol(), ref.GetMol()
    if qm.GetNumBonds() != rm.GetNumBonds():
        return None
    match = qm.GetSubstructMatch(rm)       # ref atom k -> xyz atom match[k]
    return list(match) if match and len(match) == len(xs) else None


# --------------------------------------------------------------------------
# Public API
# --------------------------------------------------------------------------
def build_geometry(smiles, n_seeds=24, stop_after=6, verbose=True):
    mol = build_reference_molecule(smiles)
    sp = Spec(mol)
    best, passing = None, 0
    for seed in range(n_seeds):
        X, E = refine(embed(mol, sp, seed), sp)
        chk = gco.check_geometry(mol, X)
        key = (len(chk["reasons"]), E)
        if best is None or key < best[0]:
            best = (key, X, E, chk, seed)
        if chk["verdict"] == "PASS":
            passing += 1
            if passing >= stop_after:
                break
    _, X, E, chk, seed = best
    # centre at origin
    X = X - X.mean(axis=0)
    if verbose:
        print(f"[G3] seed={seed} restraint_energy={E:.4f} "
              f"verdict={chk['verdict']} max_bond_err={chk['max_err']:.4f}")
    return mol, X, chk, seed


def repair_structure(smiles, input_xyz=None, output_xyz="corrected.xyz",
                     output_sdf="corrected.sdf", output_png="structure_2d.png",
                     output_report="validation_report.txt"):
    print("\n" + "=" * 78 +
          "\nG3 UNIFIED SMILES-AUTHORITATIVE GEOMETRY ENGINE\n" + "=" * 78)
    mol, X, chk, seed = build_geometry(smiles)
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]

    comment, extra = f"G3 | {smiles}", None
    map_note = "no input XYZ supplied"
    if input_xyz:
        try:
            c0, xs, xc, ex = parse_extended_xyz(input_xyz)
            comment = c0 or comment
            mp = map_input(mol, xs, xc)
            if mp is None:
                map_note = ("input XYZ composition OK but atom mapping not "
                            "unique (broken topology) - extra columns dropped")
            else:
                extra = [ex[mp[k]] for k in range(len(symbols))]
                map_note = "input XYZ mapped exactly; extra columns carried"
        except Exception as exc:
            map_note = f"input XYZ not used: {exc}"
    print(f"[G3] {map_note}")

    conf = Chem.Conformer(mol.GetNumAtoms())
    for i, p in enumerate(X):
        conf.SetAtomPosition(i, tuple(float(v) for v in p))
    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)

    write_extended_xyz(output_xyz, comment, symbols, X, extra)
    write_sdf(mol, X, output_sdf)
    write_2d_png(smiles, output_png)
    with open(output_report, "w", encoding="utf-8") as fh:
        fh.write("G3 SMILES-AUTHORITATIVE GEOMETRY REPORT\n" + "=" * 78 + "\n")
        fh.write(f"SMILES: {smiles}\nverdict: {chk['verdict']}\n")
        fh.write(f"max bond error: {chk['max_err']:.4f} A   "
                 f"rms: {chk['rms_err']:.4f} A\n")
        for r in chk["bonds"]:
            fh.write(f"{r['i']:3d} {r['ei']:>2s}-{r['j']:3d} {r['ej']:<2s} "
                     f"d={r['dist']:.4f} target={r['target']:.4f} "
                     f"{'PASS' if r['ok'] else 'FAIL'}\n")
        for r in chk["hyper"]:
            fh.write(f"hypervalent {r['element']}{r['centre']} {r['geometry']}"
                     f" max dev {r['max_dev']:.1f} deg "
                     f"{'PASS' if r['ok'] else 'FAIL'}\n")
        for s in chk["reasons"]:
            fh.write(f"ISSUE: {s}\n")

    final = dict(report=chk["bonds"], bad=chk["bad_bonds"],
                 clashes=chk["clashes"], max_error=chk["max_err"],
                 mean_error=chk["mean_err"], check=chk)
    status = "PASS" if chk["verdict"] == "PASS" else "UNRESOLVED"
    print(f"FINAL STATUS: {status}")
    return mol, X, final


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("usage: python g3_engine.py 'SMILES' [prefix]")
        sys.exit(1)
    pre = sys.argv[2] if len(sys.argv) > 2 else "g3_out"
    repair_structure(sys.argv[1], None, f"{pre}.xyz", f"{pre}.sdf",
                     f"{pre}.png", f"{pre}_report.txt")


### Validation gate (`geometry_gate.py`) - validates every molecule, repairs only failures

In [ ]:
%%writefile geometry_gate.py
#!/usr/bin/env python3
"""
geometry_gate.py  -  mandatory validation gate for the screening driver
=======================================================================
Put next to g3_engine.py, geom_common.py and validate_geometry.py.

WHY: the old driver only ran G1 when a SMARTS list said "hypervalent".  A
molecule that is not in that list (e.g. O=I(O)(O)Ph, I(V)) was never
validated at all.  This gate runs on EVERY molecule:

    optimizer geometry --> validate --PASS--> keep the relaxed geometry
                                     +-FAIL--> G3 rebuild --> re-validate

* Geometries that already pass are left untouched (no needless SCF re-run,
  relaxed structure preserved).
* Only failing geometries are rebuilt, whatever the element or SMILES.
* If the rebuild still fails the result is flagged GEOMETRY_INVALID instead
  of being reported as a success.

Tolerances are deliberately looser than for template geometries, because the
input is a DFT/ASE-relaxed structure (bonds 0.15 A / 8 %, angles 20 deg).
They still catch the real failure modes (detached ligands at 3 A, shells
distorted by tens of degrees, collapsed cages).

Use in the driver
-----------------
    from geometry_gate import validate_and_repair
    gate = validate_and_repair(molecule_id, smiles, raw_xyz_path, OUTPUT_DIR)
    record.update(gate["record_fields"])
    if gate["action"] == "REPAIRED":
        atoms = ase_read(gate["xyz"])          # then SCF on the repaired atoms
        attach_pyscf_calculator(atoms, basis=FINAL_BASIS, xc=FINAL_XC)
        qc = run_final_electronic_qc(atoms, molecule_id)
        ...
    elif gate["status"] == "UNRESOLVED":
        record["Status"] = "GEOMETRY_INVALID"

Audit existing results (no calculations, seconds per molecule):
    python geometry_gate.py --audit research_data
"""
import argparse
import csv
import glob
import json
import os
import sys

import numpy as np
from rdkit import Chem

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import geom_common as gco  # noqa: E402  (never name it "gc": that shadows the stdlib garbage-collector module)

if not hasattr(gco, "check_geometry"):
    raise ImportError(
        f"geom_common loaded from {getattr(gco, '__file__', '?')} has no "
        "check_geometry: a stale or different geom_common.py is being imported. "
        "Keep ONE copy (from the same download as geometry_gate.py), delete "
        "__pycache__, and restart the Python session / Colab runtime.")

# optimizer-output tolerances (relaxed structures, not templates)
RELAXED = dict(tol_abs=0.15, tol_rel=0.08, ang_tol=20.0, hyper_scale=1.3,
               clash_factor=0.65, plane_tol=0.08)


def _load(mol, path):
    from validate_geometry import load_in_smiles_order
    return load_in_smiles_order(mol, path)


def check_file(smiles, xyz_path, **tol):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    X, note = _load(mol, xyz_path)
    return mol, X, gco.check_geometry(mol, X, **(tol or RELAXED)), note


def validate_and_repair(mol_id, smiles, raw_xyz, out_dir, force_repair=False,
                        verbose=True):
    """Validate `raw_xyz`; rebuild with G3 only if it fails.
    Returns dict(action, status, xyz, record_fields, before, after)."""
    from g3_engine import repair_structure
    from validate_geometry import write_outputs
    pre = os.path.join(out_dir, f"{mol_id}_geometry")
    fields = {"Geometry_Gate": "G3-validator-v2"}
    try:
        mol, X, before, note = check_file(smiles, raw_xyz)
    except Exception as exc:        # unreadable / wrong composition / unmappable
        before, note, mol, X = None, f"input could not be validated: {exc}", None, None
        if verbose:
            print(f"  -> [GATE] {note}", flush=True)

    if before is not None and before["verdict"] == "PASS" and not force_repair:
        write_outputs(pre, smiles, mol, before, None, [note])
        fields.update(Geometry_Validation="PASS", Geometry_Action="KEPT_RELAXED",
                      Geometry_Bad_Bonds_Before=0,
                      Geometry_Max_Bond_Err_A=round(before["max_err"], 4))
        if verbose:
            print("  -> [GATE] relaxed geometry valid; kept.", flush=True)
        return dict(action="KEPT", status="PASS", xyz=raw_xyz,
                    record_fields=fields, before=before, after=before)

    if verbose and before is not None:
        print("  -> [GATE] geometry INVALID: " + "; ".join(before["reasons"]),
              flush=True)
    perfect = os.path.join(out_dir, f"{mol_id}_perfect_geometry.xyz")
    repair_structure(
        smiles, raw_xyz if before is not None else None, perfect,
        os.path.join(out_dir, f"{mol_id}_perfect.sdf"),
        os.path.join(out_dir, f"{mol_id}_2d_projection.png"),
        os.path.join(out_dir, f"{mol_id}_validation_report.txt"))
    # re-validate the written file with the strict (template) tolerances
    mol2, X2, after, note2 = check_file(smiles, perfect, **dict(
        tol_abs=0.10, tol_rel=0.06, ang_tol=15.0, hyper_scale=1.0,
        clash_factor=0.65, plane_tol=0.06))
    ok = after["verdict"] == "PASS"
    if before is not None:
        write_outputs(pre, smiles, mol2, after, before, [note, note2])
        # `before` must be re-evaluated in the same (strict) frame for the report
    if before is None:
        fields["Geometry_Note"] = note[:200]
    fields.update(
        Geometry_Validation="PASS" if ok else "FAIL",
        Geometry_Action="REPAIRED" if ok else "REPAIR_FAILED",
        Geometry_Bad_Bonds_Before=len(before["bad_bonds"]) if before else None,
        Geometry_Max_Bond_Err_A=round(after["max_err"], 4))
    return dict(action="REPAIRED" if ok else "REPAIR_FAILED",
                status="PASS" if ok else "UNRESOLVED", xyz=perfect,
                record_fields=fields, before=before, after=after)


# ---------------------------------------------------------------------------
# audit of an existing research_data folder
# ---------------------------------------------------------------------------
def audit(folder, out_csv=None):
    smiles = {}
    rj = os.path.join(folder, "results.jsonl")
    if os.path.isfile(rj):
        with open(rj, encoding="utf-8") as fh:
            for ln in fh:
                try:
                    d = json.loads(ln)
                    smiles[d["Molecule_ID"]] = d["SMILES"]
                except Exception:
                    pass
    rows = []
    ids = sorted({os.path.basename(p).split("_final_accepted")[0].split("_perfect")[0]
                  for p in glob.glob(os.path.join(folder, "MOL_*_*geometry.xyz"))})
    for mid in ids:
        smi = smiles.get(mid)
        for kind in ("final_accepted", "perfect"):
            p = os.path.join(folder, f"{mid}_{kind}_geometry.xyz")
            if not os.path.isfile(p):
                continue
            if not smi:
                rows.append([mid, kind, "", "NO_SMILES", "", "", "", ""])
                continue
            try:
                _, _, c, _ = check_file(smi, p)
                rows.append([mid, kind, smi, c["verdict"], len(c["bad_bonds"]),
                             f"{c['max_err']:.3f}",
                             "; ".join(f"{h['element']} {h['geometry']} dev {h['max_dev']:.0f}"
                                       for h in c["hyper"]),
                             "; ".join(c["reasons"])])
            except Exception as exc:
                rows.append([mid, kind, smi, "ERROR", "", "", "", str(exc)[:80]])
    out_csv = out_csv or os.path.join(folder, "geometry_audit.csv")
    with open(out_csv, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["Molecule_ID", "file", "SMILES", "verdict", "bad_bonds",
                    "max_bond_err_A", "hypervalent_shells", "issues"])
        w.writerows(rows)
    for r in rows:
        print(f"{r[0]:9s} {r[1]:15s} {r[3]:6s} bad={r[4]} max={r[5]} {r[6]} {r[7]}")
    print("written:", out_csv)
    return rows


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--audit", help="folder with MOL_*_*_geometry.xyz + results.jsonl")
    a = ap.parse_args()
    if a.audit:
        audit(a.audit)
    else:
        ap.print_help()


### Validator (`validate_geometry.py`) - command line, batch and PyQt6 dashboard (the dashboard runs on a desktop Python, not in Colab)

In [ ]:
%%writefile validate_geometry.py
#!/usr/bin/env python3
"""
validate_geometry.py  v2  -  SMILES-authoritative geometry validator
====================================================================
Works for ordinary AND hypervalent / heavy-element molecules (S(VI), P(V),
I(III), I(V), Xe, ...).  All checks come from geom_common.check_geometry():

  1. composition + atom order (AddHs(SMILES) order; auto-remap if needed)
  2. every SMILES bond vs a context-aware target   (detached atoms are shown
     with their TRUE distance - never replaced by the "after" value)
  3. unintended contacts / detached bonds perceived from the coordinates
  4. angles at every atom (hypervalent shells matched to octahedral, TBP,
     T-shape, see-saw, linear ... by brute-force ligand->site assignment,
     so the result does not depend on how the ligands happen to be ordered)
  5. planarity of aromatic rings and sp2 centres
  6. steric clashes (graph distance >= 3, vdW based)

Usage
-----
  GUI (PyQt6 dashboard, like before):  python validate_geometry.py     (no arguments)
  command line:
  single file:
    python validate_geometry.py --smiles "FI(F)c1ccccc1" --after out.xyz
  before / after comparison:
    python validate_geometry.py --smiles "..." --before old.xyz --after new.xyz \
           --prefix MOL_0007 --plot
  batch (CSV with header  name,smiles,xyz  [,before]):
    python validate_geometry.py --batch list.csv --outdir reports

Exit code 0 = all PASS, 1 = at least one FAIL.
"""
import argparse
import csv
import json
import os
import sys

import numpy as np
from rdkit import Chem, RDLogger

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import geom_common as gco  # noqa: E402  (never name it "gc": that shadows the stdlib garbage-collector module)

RDLogger.DisableLog("rdApp.*")


def read_xyz(path):
    with open(path, encoding="utf-8", errors="ignore") as fh:
        lines = fh.read().splitlines()
    n = int(lines[0].split()[0])
    sym, xyz = [], []
    for ln in lines[2:]:
        p = ln.split()
        if len(p) < 4:
            continue
        try:
            xyz.append([float(p[1]), float(p[2]), float(p[3])])
        except ValueError:
            continue
        sym.append(p[0])
    if len(sym) != n:
        raise ValueError(f"{path}: header says {n} atoms, parsed {len(sym)}")
    return sym, np.asarray(xyz, float)


def smiles_mol(smiles):
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return Chem.AddHs(m)


def load_in_smiles_order(mol, path):
    """Return (coords in AddHs(SMILES) order, note)."""
    sym, X = read_xyz(path)
    ref = [a.GetSymbol() for a in mol.GetAtoms()]
    if sorted(sym) != sorted(ref):
        raise ValueError(f"{os.path.basename(path)}: composition "
                         f"{sorted(sym)} does not match SMILES {sorted(ref)}")
    if sym == ref:
        return X, "atom order identical to AddHs(SMILES)"
    from g3_engine import map_input        # graph-based remap
    mp = map_input(mol, sym, X)
    if mp is None:
        raise ValueError(f"{os.path.basename(path)}: atom order differs and "
                         "no unique graph mapping exists (broken topology). "
                         "Provide the file in AddHs(SMILES) order.")
    return X[np.array(mp)], "atoms remapped to SMILES order by graph match"


def fmt_summary(tag, c):
    return (f"{tag:7s} verdict={c['verdict']:4s} bad_bonds={len(c['bad_bonds'])} "
            f"max_err={c['max_err']:.3f} A  extra_contacts={len(c['extra_bonds'])} "
            f"bad_angles={len(c['bad_angles'])} bad_shells={len(c['bad_hyper'])} "
            f"nonplanar={len(c['bad_planar'])} clashes={len(c['clashes'])}")


def write_outputs(prefix, smiles, mol, after, before=None, notes=()):
    sym = [a.GetSymbol() for a in mol.GetAtoms()]
    # ---- bonds csv
    with open(f"{prefix}_bonds.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        head = ["atom_i", "atom_j", "type", "order", "target_A",
                "target_source"]
        if before:
            head += ["before_A", "before_err_A", "before_ok"]
        head += ["after_A", "after_err_A", "after_ok"]
        w.writerow(head)
        for k, r in enumerate(after["bonds"]):
            row = [f"{r['ei']}{r['i']+1}", f"{r['ej']}{r['j']+1}",
                   f"{r['ei']}-{r['ej']}", r["order"], f"{r['target']:.3f}",
                   r["source"]]
            if before:
                b = before["bonds"][k]
                row += [f"{b['dist']:.4f}", f"{b['err']:+.4f}",
                        "PASS" if b["ok"] else "FAIL"]
            row += [f"{r['dist']:.4f}", f"{r['err']:+.4f}",
                    "PASS" if r["ok"] else "FAIL"]
            w.writerow(row)
    # ---- text report
    L = []
    L.append("SMILES-AUTHORITATIVE GEOMETRY VALIDATION (validate_geometry v2)")
    L.append("=" * 78)
    L.append(f"SMILES : {smiles}")
    L.append(f"Atoms  : {mol.GetNumAtoms()}   Bonds: {mol.GetNumBonds()}")
    for n in notes:
        L.append(f"Note   : {n}")
    L.append("")
    if before:
        L.append(fmt_summary("BEFORE", before))
    L.append(fmt_summary("AFTER", after))
    if after["reasons"]:
        L.append("AFTER issues: " + "; ".join(after["reasons"]))
    L.append("")
    L.append("BOND TABLE  (* = outside tolerance; distances are the true "
             "values from each file)")
    L.append("-" * 78)
    for k, r in enumerate(after["bonds"]):
        s = (f"{r['ei']:>2s}{r['i']+1:<3d}-{r['ej']:>2s}{r['j']+1:<3d} "
             f"target {r['target']:.3f}{'f' if r['source']=='fallback' else ' '}")
        if before:
            b = before["bonds"][k]
            s += f" | before {b['dist']:7.3f}{'*' if not b['ok'] else ' '}"
        s += f" | after {r['dist']:7.3f}{'*' if not r['ok'] else ' '}"
        L.append(s)
    L.append("(f = target from covalent-radii fallback: pair not in table)")
    L.append("")
    for tag, c in (("BEFORE", before), ("AFTER", after)):
        if c is None or not c["hyper"]:
            continue
        L.append(f"HYPERVALENT SHELLS - {tag}")
        for h in c["hyper"]:
            L.append(f"  {h['element']}{h['centre']+1}: {h['geometry']:8s} "
                     f"ligand-angle range {h['min_angle']:.1f}-"
                     f"{h['max_angle']:.1f} deg, worst deviation from ideal "
                     f"{h['max_dev']:.1f} deg (tol {h['tol']:.0f}) "
                     f"{'PASS' if h['ok'] else 'FAIL'}")
        L.append("")
    for tag, c in (("BEFORE", before), ("AFTER", after)):
        if c is None:
            continue
        for key, title in (("extra_bonds", "unintended contacts"),
                           ("missing_bonds", "detached bonds"),
                           ("clashes", "clashes")):
            for r in c[key]:
                L.append(f"{tag} {title}: {r['ei']}{r['i']+1}...{r['ej']}"
                         f"{r['j']+1}  {r['dist']:.3f} A")
        for r in c["bad_angles"]:
            L.append(f"{tag} angle: {sym[r['i']]}{r['i']+1}-{r['ec']}"
                     f"{r['c']+1}-{sym[r['j']]}{r['j']+1} = {r['angle']:.1f} "
                     f"(ideal {r['ideal']:.1f})")
        for r in c["bad_planar"]:
            L.append(f"{tag} planarity: {r['kind']} {[x+1 for x in r['atoms']]}"
                     f" deviation {r['max_dev']:.3f}")
    L.append("")
    L.append(f"FINAL VERDICT (AFTER): {after['verdict']}")
    with open(f"{prefix}_report.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(L) + "\n")
    # ---- json
    def brief(c):
        return None if c is None else dict(
            verdict=c["verdict"], max_bond_err=c["max_err"],
            mean_bond_err=c["mean_err"], rms_bond_err=c["rms_err"],
            bad_bonds=len(c["bad_bonds"]), extra_contacts=len(c["extra_bonds"]),
            bad_angles=len(c["bad_angles"]), bad_shells=len(c["bad_hyper"]),
            nonplanar=len(c["bad_planar"]), clashes=len(c["clashes"]),
            shells=[dict(centre=f"{h['element']}{h['centre']+1}",
                         geometry=h["geometry"], max_dev=h["max_dev"])
                    for h in c["hyper"]], reasons=c["reasons"])
    with open(f"{prefix}_summary.json", "w") as fh:
        json.dump(dict(smiles=smiles, before=brief(before),
                       after=brief(after)), fh, indent=2)


def plot(prefix, after, before=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    n = len(after["bonds"])
    lab = [f"{r['ei']}{r['i']+1}-{r['ej']}{r['j']+1}" for r in after["bonds"]]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.22 * n + 1)))
    y = np.arange(n)
    if before:
        ax.barh(y + 0.2, [abs(b["err"]) for b in before["bonds"]], 0.4,
                label="before", color="#c0504d")
    ax.barh(y - 0.2 if before else y, [abs(r["err"]) for r in after["bonds"]],
            0.4, label="after", color="#4f81bd")
    ax.axvline(0.10, color="k", ls="--", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(lab, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("|bond length - target| (A)   dashed = 0.10 A tolerance")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{prefix}_deviation.png", dpi=160)
    plt.close(fig)


def run_one(smiles, after_path, before_path=None, prefix="report", do_plot=False):
    mol = smiles_mol(smiles)
    Xa, na = load_in_smiles_order(mol, after_path)
    notes = [f"AFTER file: {na}"]
    after = gco.check_geometry(mol, Xa)
    before = None
    if before_path:
        Xb, nb = load_in_smiles_order(mol, before_path)
        notes.append(f"BEFORE file: {nb}")
        before = gco.check_geometry(mol, Xb)
    write_outputs(prefix, smiles, mol, after, before, notes)
    if do_plot:
        plot(prefix, after, before)
    return before, after


def cli_main():
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--smiles")
    ap.add_argument("--after")
    ap.add_argument("--before")
    ap.add_argument("--prefix", default="report")
    ap.add_argument("--plot", action="store_true")
    ap.add_argument("--batch", help="CSV with columns name,smiles,xyz[,before]")
    ap.add_argument("--outdir", default=".")
    a = ap.parse_args()
    all_pass = True
    if a.batch:
        os.makedirs(a.outdir, exist_ok=True)
        rows = []
        with open(a.batch, newline="") as fh:
            for r in csv.DictReader(fh):
                try:
                    b, c = run_one(r["smiles"], r["xyz"], r.get("before") or None,
                                   os.path.join(a.outdir, r["name"]), a.plot)
                    rows.append((r["name"], c["verdict"], len(c["bad_bonds"]),
                                 c["max_err"], len(c["bad_angles"]) +
                                 len(c["bad_hyper"]), len(c["clashes"]),
                                 "; ".join(c["reasons"])))
                    all_pass &= c["verdict"] == "PASS"
                except Exception as exc:
                    rows.append((r["name"], "ERROR", "", "", "", "", str(exc)))
                    all_pass = False
        with open(os.path.join(a.outdir, "batch_summary.csv"), "w",
                  newline="") as fh:
            w = csv.writer(fh)
            w.writerow(["name", "verdict", "bad_bonds", "max_bond_err_A",
                        "bad_angles_or_shells", "clashes", "issues"])
            for r in rows:
                w.writerow([r[0], r[1], r[2],
                            f"{r[3]:.4f}" if r[3] != "" else "", r[4], r[5],
                            r[6]])
        for r in rows:
            print(f"{r[0]:28s} {r[1]:5s} {r[6]}")
    else:
        if not (a.smiles and a.after):
            ap.error("need --smiles and --after (or --batch)")
        b, c = run_one(a.smiles, a.after, a.before, a.prefix, a.plot)
        if b:
            print(fmt_summary("BEFORE", b))
        print(fmt_summary("AFTER", c))
        print("VERDICT:", c["verdict"], "|", "; ".join(c["reasons"]) or "all checks passed")
        all_pass = c["verdict"] == "PASS"
    sys.exit(0 if all_pass else 1)


# =============================================================================
# SI-5 REPORT TEXT (generic: works for any molecule / hypervalent shell)
# =============================================================================
def xyz_text(sym, X, comment=""):
    return (f"{len(sym)}\n{comment}\n" +
            "\n".join(f"{s} {p[0]:.6f} {p[1]:.6f} {p[2]:.6f}"
                      for s, p in zip(sym, X)) + "\n")


def si_report_text(smiles, mol, before, after, notes=()):
    sym = [a.GetSymbol() for a in mol.GetAtoms()]
    W = 100
    L = []
    L.append("Geometry validation report - SMILES-authoritative repair (validate_geometry v2)")
    L.append("Method: every bond of the SMILES graph is compared with a context-aware target "
             "length; a bond passes if |d - target| <= max(0.10 A, 6% of target).")
    L.append("Angles are checked at every atom; hypervalent centres (S(VI), P(V), I(III), "
             "I(V), Xe ...) are matched to octahedral / trigonal-bipyramidal / T-shaped / "
             "see-saw / linear shells by testing every ligand-to-site assignment.")
    L.append("Aromatic rings and sp2 centres are tested for planarity; non-bonded pairs "
             "(graph distance >= 3) closer than 0.65 x the sum of vdW radii are clashes.")
    L.append("'Before' distances are the true distances in the supplied file (detached atoms "
             "are never replaced by the 'after' value). Repaired hypervalent shells are "
             "template-built and are not re-relaxed on the DFT surface.")
    L.append("")
    L.append(f"SMILES: {smiles}")
    for n in notes:
        L.append(f"Note: {n}")
    L.append("")
    L.append("-" * W)
    L.append(f"{'Metric':<38} | {'Before':<24} | {'After':<24}")
    L.append("-" * W)

    def row(name, f):
        b = f(before) if before else "n/a"
        L.append(f"{name:<38} | {str(b):<24} | {str(f(after)):<24}")
    row("Bonds failing the length test", lambda c: len(c["bad_bonds"]))
    row("Maximum bond-length error (A)", lambda c: f"{c['max_err']:.3f}")
    row("Mean bond-length error (A)", lambda c: f"{c['mean_err']:.3f}")
    row("Unintended contacts", lambda c: len(c["extra_bonds"]))
    row("Detached bonds", lambda c: len(c["missing_bonds"]))
    row("Angles outside tolerance", lambda c: len(c["bad_angles"]))
    row("Hypervalent shells distorted", lambda c: f"{len(c['bad_hyper'])} of {len(c['hyper'])}")
    row("Planarity violations", lambda c: len(c["bad_planar"]))
    row("Steric clashes", lambda c: len(c["clashes"]))
    row("Verdict", lambda c: c["verdict"])
    L.append("-" * W)
    L.append("")
    shells = [(t, c) for t, c in (("Before", before), ("After", after)) if c]
    if any(c["hyper"] for _, c in shells):
        L.append("Hypervalent shells")
        L.append(f"{'State':<8} | {'Centre':<8} | {'Shape':<9} | {'Ligand angle range (deg)':<26} | {'Worst dev (deg)':<16} | Status")
        for t, c in shells:
            for h in c["hyper"]:
                L.append(f"{t:<8} | {h['element']+str(h['centre']+1):<8} | {h['geometry']:<9} | "
                         f"{('%.1f - %.1f' % (h['min_angle'], h['max_angle'])):<26} | "
                         f"{h['max_dev']:<16.1f} | {'PASS' if h['ok'] else 'FAIL'}")
        L.append("")
    L.append("Table SI-5. Bond-by-bond validation (distances and targets in A; |D| = absolute deviation).")
    L.append(f"{'#':<4} | {'Bond':<10} | {'Type':<8} | {'Target':<7} | {'Before':<8} | {'After':<8} | {'|D| bef':<8} | {'|D| aft':<8} | {'Bef':<5} | Aft")
    L.append("-" * W)
    for k, r in enumerate(after["bonds"]):
        b = before["bonds"][k] if before else None
        L.append(f"{k:<4} | {sym[r['i']]+str(r['i']+1)+'-'+sym[r['j']]+str(r['j']+1):<10} | "
                 f"{r['ei']+'-'+r['ej']:<8} | {r['target']:<7.3f} | "
                 f"{(('%.3f' % b['dist']) if b else 'n/a'):<8} | {r['dist']:<8.3f} | "
                 f"{(('%.3f' % abs(b['err'])) if b else 'n/a'):<8} | {abs(r['err']):<8.3f} | "
                 f"{('PASS' if b['ok'] else 'FAIL') if b else 'n/a':<5} | {'PASS' if r['ok'] else 'FAIL'}")
    L.append("")
    L.append("Notes")
    L.append("- Targets marked as covalent-radii fallback in the CSV are pairs missing from the target table; check them manually.")
    L.append("- Reproduce: run g3_engine.repair_structure(smiles, ...) then validate_geometry.py on the output.")
    return "\n".join(L) + "\n"


# =============================================================================
# DESKTOP GUI (PyQt6)  -  started when the script is run without arguments
# =============================================================================
CPK = {"H": "#ffffff", "C": "#909090", "N": "#3050f8", "O": "#ff0d0d",
       "F": "#90e050", "Cl": "#1ff01f", "Br": "#a62929", "I": "#940094",
       "S": "#ffff30", "P": "#ff8000", "Xe": "#429eb0", "Si": "#f0c8a0"}


def angle_labels(mol, X, all_atoms=False):
    """[(text, (x, y, z))] angle labels placed on the bisector near each vertex.
    Default: only hypervalent centres and angles that fail the check."""
    hyper = set(gco.hyper_centres(mol))
    out = []
    for a in mol.GetAtoms():
        c = a.GetIdx()
        nb = [n.GetIdx() for n in a.GetNeighbors()]
        if len(nb) < 2 or (not all_atoms and c not in hyper):
            continue
        for p in range(len(nb)):
            for q in range(p + 1, len(nb)):
                u, v = X[nb[p]] - X[c], X[nb[q]] - X[c]
                ang = gco._angle(X[nb[p]], X[c], X[nb[q]])
                m = u / np.linalg.norm(u) + v / np.linalg.norm(v)
                if np.linalg.norm(m) < 1e-3:                     # ~180 deg
                    m = np.cross(u, [0.3, 0.5, 0.8])
                m = m / (np.linalg.norm(m) + 1e-12)
                out.append((f"{ang:.1f}", tuple(float(t) for t in X[c] + 0.8 * m)))
    return out


def viewer_html(sym, X, bad_pairs, labels):
    data = xyz_text(sym, X)
    js_lines = "".join(
        f'viewer.addLine({{start:{{x:{X[i][0]},y:{X[i][1]},z:{X[i][2]}}},'
        f'end:{{x:{X[j][0]},y:{X[j][1]},z:{X[j][2]}}},color:"red",dashed:true,'
        f'dashLength:0.25,gapLength:0.15}});\n' for i, j in bad_pairs)
    js_lab = "".join(
        f'viewer.addLabel("{t}",{{position:{{x:{p[0]},y:{p[1]},z:{p[2]}}},'
        f'backgroundColor:"black",backgroundOpacity:0.6,fontColor:"white",'
        f'fontSize:10,borderColor:"gray"}});\n' for t, p in labels)
    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<script src="https://3dmol.org/build/3Dmol-min.js"></script>
<style>html,body{{margin:0;padding:0;width:100%;height:100%;overflow:hidden;background:#1e1e1e}}
#viewer{{width:100%;height:100%}}</style></head><body><div id="viewer"></div>
<script>window.addEventListener("load",function(){{
 let data={json.dumps(data)};
 let viewer=$3Dmol.createViewer(document.getElementById("viewer"),{{backgroundColor:"#1e1e1e"}});
 viewer.addModel(data,"xyz");
 viewer.setStyle({{}},{{stick:{{radius:0.17,colorscheme:"Jmol"}},sphere:{{scale:0.22,colorscheme:"Jmol"}}}});
 {js_lines}{js_lab}
 viewer.zoomTo();viewer.render();
 setTimeout(function(){{viewer.resize();viewer.zoomTo();viewer.render();}},250);}});
</script></body></html>"""


def launch_gui():
    from PyQt6.QtCore import Qt
    from PyQt6.QtGui import QColor, QBrush
    from PyQt6.QtWidgets import (
        QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
        QPushButton, QLabel, QFileDialog, QTableWidget, QTableWidgetItem,
        QSplitter, QHeaderView, QGroupBox, QMessageBox, QTabWidget, QLineEdit,
        QCheckBox, QStackedWidget)
    from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
    from matplotlib.figure import Figure
    try:
        if os.environ.get("VG_NO_WEBENGINE") == "1":
            raise ImportError
        from PyQt6.QtWebEngineWidgets import QWebEngineView
        HAVE_WEB = True
    except Exception:
        HAVE_WEB = False

    GOOD, BAD = QColor("#14532d"), QColor("#7f1d1d")

    class ChartCanvas(FigureCanvas):
        def __init__(self):
            self.fig = Figure(figsize=(6, 5), facecolor="#1e1e1e")
            super().__init__(self.fig)
            self.ax1 = self.fig.add_subplot(211)
            self.ax2 = self.fig.add_subplot(212)
            self.style()

        def style(self):
            for ax in (self.ax1, self.ax2):
                ax.set_facecolor("#252526")
                ax.tick_params(colors="white", labelsize=8)
                for s in ax.spines.values():
                    s.set_color("#555555")
                ax.xaxis.label.set_color("white")
                ax.yaxis.label.set_color("white")
                ax.title.set_color("white")

        def update_plots(self, rows):
            self.ax1.clear()
            self.ax2.clear()
            self.style()
            if not rows:
                self.draw()
                return
            x = np.arange(len(rows))
            lab = [r["bond"] for r in rows]
            w = 0.35
            has_b = rows[0]["before"] is not None
            if has_b:
                self.ax1.bar(x - w / 2, [r["dev_before"] for r in rows], w,
                             label="Before", color="#ef4444")
            self.ax1.bar(x + (w / 2 if has_b else 0), [r["dev_after"] for r in rows],
                         w if has_b else 0.6, label="After", color="#10b981")
            self.ax1.axhline(0.10, color="#facc15", ls="--", lw=0.8)
            self.ax1.set_title("Absolute deviation from target |D| (dashed = 0.10 A)",
                               fontsize=10, fontweight="bold")
            self.ax1.set_ylabel("Deviation (A)", fontsize=9)
            self.ax1.set_xticks(x)
            self.ax1.set_xticklabels(lab, rotation=60, ha="right", fontsize=6)
            self.ax1.legend(fontsize=8, facecolor="#2d2d2d", labelcolor="white")
            if has_b:
                self.ax2.plot(x, [r["before"] for r in rows], "o--", color="#ef4444",
                              label="Length before", ms=3)
            self.ax2.plot(x, [r["after"] for r in rows], "s-", color="#3b82f6",
                          label="Length after", ms=3)
            self.ax2.plot(x, [r["target"] for r in rows], "x:", color="#facc15",
                          label="Target", ms=4)
            self.ax2.set_title("Bond lengths vs target", fontsize=10, fontweight="bold")
            self.ax2.set_ylabel("Distance (A)", fontsize=9)
            self.ax2.set_xticks(x)
            self.ax2.set_xticklabels(lab, rotation=60, ha="right", fontsize=6)
            self.ax2.legend(fontsize=8, facecolor="#2d2d2d", labelcolor="white")
            self.fig.tight_layout()
            self.draw()

    class Mol3DCanvas(FigureCanvas):
        """Offline fallback viewer (used when QtWebEngine is unavailable)."""
        def __init__(self):
            self.fig = Figure(figsize=(5, 5), facecolor="#1e1e1e")
            super().__init__(self.fig)
            self.ax = self.fig.add_subplot(111, projection="3d")

        def show_mol(self, mol, sym, X, bad_pairs, labels):
            ax = self.ax
            ax.clear()
            ax.set_facecolor("#1e1e1e")
            ax.set_axis_off()
            bad = {tuple(sorted(p)) for p in bad_pairs}
            for b in mol.GetBonds():
                i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
                isbad = tuple(sorted((i, j))) in bad
                ax.plot(*zip(X[i], X[j]), color="#ef4444" if isbad else "#bbbbbb",
                        ls="--" if isbad else "-", lw=1.4)
            for k, (s, p) in enumerate(zip(sym, X)):
                ax.scatter(*p, s=60 if s != "H" else 25, color=CPK.get(s, "#ff69b4"),
                           edgecolors="#333333", depthshade=False)
            for t, p in labels:
                ax.text(*p, t, color="white", fontsize=7)
            r = max(np.ptp(X, axis=0).max() / 2, 1.0)
            c = X.mean(axis=0)
            ax.set_xlim(c[0] - r, c[0] + r)
            ax.set_ylim(c[1] - r, c[1] + r)
            ax.set_zlim(c[2] - r, c[2] + r)
            self.draw()

    class App(QMainWindow):
        def __init__(self):
            super().__init__()
            self.setWindowTitle("SI-5 SMILES-Authoritative Geometry Validation Suite (v2: normal + hypervalent)")
            self.resize(1500, 900)
            self.path_before = self.path_after = None
            self.mol = None
            self.before = self.after = None
            self.rows = []
            self.notes = []
            self.Xb = self.Xa = None
            self._build()

        # ---------------------------------------------------------- UI
        def _btn(self, bg, hover):
            return (f"QPushButton{{background-color:{bg};color:white;font-weight:bold;"
                    f"padding:8px 16px;border-radius:4px;border:none;}}"
                    f"QPushButton:hover{{background-color:{hover};}}"
                    f"QPushButton:disabled{{background-color:#52525b;color:#a1a1aa;}}")

        def _build(self):
            root = QWidget()
            self.setCentralWidget(root)
            lay = QVBoxLayout(root)
            top = QHBoxLayout()
            lab = QLabel("SMILES:")
            lab.setStyleSheet("font-weight:bold;color:white;")
            self.smiles = QLineEdit("FS(F)(F)(F)(F)c1ccccc1")
            self.smiles.setStyleSheet("background-color:#2d2d2d;color:white;padding:6px;"
                                      "border:1px solid #444;border-radius:4px;")
            self.smiles.setMinimumWidth(340)
            self.smiles.returnPressed.connect(self.run)
            self.b_before = QPushButton("Upload Before .XYZ (optional)")
            self.b_before.setStyleSheet(self._btn("#d97706", "#b45309"))
            self.b_before.clicked.connect(lambda: self.load("before"))
            self.l_before = QLabel("Before: not loaded")
            self.b_after = QPushButton("Upload After .XYZ")
            self.b_after.setStyleSheet(self._btn("#059669", "#047857"))
            self.b_after.clicked.connect(lambda: self.load("after"))
            self.l_after = QLabel("After: not loaded")
            for l in (self.l_before, self.l_after):
                l.setStyleSheet("color:#aaaaaa;font-style:italic;")
            self.b_run = QPushButton("Run validation")
            self.b_run.setStyleSheet(self._btn("#7c3aed", "#6d28d9"))
            self.b_run.clicked.connect(self.run)
            self.b_clear = QPushButton("Clear before")
            self.b_clear.setStyleSheet(self._btn("#52525b", "#3f3f46"))
            self.b_clear.clicked.connect(self.clear_before)
            self.b_export = QPushButton("Export SI-5 report (+CSV/JSON)")
            self.b_export.setStyleSheet(self._btn("#2563eb", "#1d4ed8"))
            self.b_export.clicked.connect(self.export)
            self.b_export.setEnabled(False)
            for w in (lab, self.smiles, self.b_before, self.l_before, self.b_after,
                      self.l_after):
                top.addWidget(w)
            top.addStretch()
            for w in (self.b_clear, self.b_run, self.b_export):
                top.addWidget(w)
            lay.addLayout(top)

            kp = QHBoxLayout()
            self.kpi = {}
            for title, key in [("Verdict (After)", "verdict"),
                               ("Failing bonds (Bef -> Aft)", "bad"),
                               ("Max bond error After", "max"),
                               ("Hypervalent shells OK", "shell"),
                               ("Angles / planarity / clashes", "other")]:
                g = QGroupBox(title)
                g.setStyleSheet("QGroupBox{background-color:#252526;border:1px solid #444;"
                                "border-radius:6px;color:#bbb;font-weight:bold;margin-top:6px;}"
                                "QGroupBox::title{subcontrol-origin:margin;left:10px;padding:0 3px;}")
                v = QVBoxLayout()
                l = QLabel("--")
                l.setStyleSheet("font-size:18px;font-weight:bold;color:#3b82f6;border:none;")
                l.setAlignment(Qt.AlignmentFlag.AlignCenter)
                v.addWidget(l)
                g.setLayout(v)
                self.kpi[key] = l
                kp.addWidget(g)
            lay.addLayout(kp)

            split = QSplitter(Qt.Orientation.Horizontal)
            left = QSplitter(Qt.Orientation.Vertical)
            tstyle = ("QTableWidget{background-color:#1e1e1e;color:#fff;gridline-color:#444;}"
                      "QHeaderView::section{background-color:#2d2d2d;color:white;"
                      "border:1px solid #444;font-weight:bold;}")
            self.tabs_tbl = QTabWidget()
            self.tabs_tbl.setStyleSheet("QTabWidget::pane{border:1px solid #444;}"
                                        "QTabBar::tab{background:#2d2d2d;color:white;padding:6px 12px;font-weight:bold;}"
                                        "QTabBar::tab:selected{background:#2563eb;}")
            self.tbl = QTableWidget()
            self.tbl.setColumnCount(10)
            self.tbl.setHorizontalHeaderLabels(["#", "Bond", "Type", "Target", "Before",
                                                "After", "|D| Bef", "|D| Aft", "Before", "After"])
            self.tbl.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeMode.Stretch)
            self.tbl.setStyleSheet(tstyle)
            self.tbl2 = QTableWidget()
            self.tbl2.setColumnCount(6)
            self.tbl2.setHorizontalHeaderLabels(["State", "Item", "Detail", "Value", "Limit", "Status"])
            self.tbl2.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeMode.Stretch)
            self.tbl2.setStyleSheet(tstyle)
            self.tabs_tbl.addTab(self.tbl, "Bond table")
            self.tabs_tbl.addTab(self.tbl2, "Hypervalent shells and issues")
            self.chart = ChartCanvas()
            left.addWidget(self.tabs_tbl)
            left.addWidget(self.chart)
            left.setSizes([380, 380])

            right = QWidget()
            rl = QVBoxLayout(right)
            head = QHBoxLayout()
            t = QLabel("Interactive 3D viewports (red dashed = SMILES bond that fails)")
            t.setStyleSheet("font-weight:bold;font-size:13px;color:#fff;")
            self.chk_all = QCheckBox("show all angles")
            self.chk_all.setStyleSheet("color:white;")
            self.chk_all.stateChanged.connect(self.refresh_views)
            self.chk_off = QCheckBox("offline viewer (no internet needed)")
            self.chk_off.setStyleSheet("color:white;")
            self.chk_off.setChecked(not HAVE_WEB)
            self.chk_off.setEnabled(HAVE_WEB)
            self.chk_off.stateChanged.connect(self.switch_viewer)
            head.addWidget(t)
            head.addStretch()
            head.addWidget(self.chk_off)
            head.addWidget(self.chk_all)
            rl.addLayout(head)
            self.vtabs = QTabWidget()
            self.vtabs.setStyleSheet("QTabWidget::pane{border:1px solid #444;}"
                                     "QTabBar::tab{background:#2d2d2d;color:white;padding:6px 12px;font-weight:bold;}"
                                     "QTabBar::tab:selected{background:#2563eb;}")
            self.mpl = {"before": Mol3DCanvas(), "after": Mol3DCanvas()}
            self.web = {"before": QWebEngineView() if HAVE_WEB else None,
                        "after": QWebEngineView() if HAVE_WEB else None}
            self.stack = {}
            for key, title in (("before", "Before (3D)"), ("after", "After (3D)")):
                st = QStackedWidget()
                st.addWidget(self.mpl[key])
                if HAVE_WEB:
                    st.addWidget(self.web[key])
                    st.setCurrentIndex(0 if self.chk_off.isChecked() else 1)
                self.stack[key] = st
                self.vtabs.addTab(st, title)
            rl.addWidget(self.vtabs)
            split.addWidget(left)
            split.addWidget(right)
            split.setSizes([800, 650])
            lay.addWidget(split)
            self.setStyleSheet("QMainWindow{background-color:#18181b;color:white;}")

        # ------------------------------------------------------ actions
        def load(self, which):
            p, _ = QFileDialog.getOpenFileName(self, f"Open {which} XYZ", "",
                                               "XYZ Files (*.xyz);;All Files (*)")
            if not p:
                return
            self.set_path(which, p)
            self.run()

        def set_path(self, which, p):
            if which == "before":
                self.path_before = p
                self.l_before.setText(f"Before: {os.path.basename(p)}")
            else:
                self.path_after = p
                self.l_after.setText(f"After: {os.path.basename(p)}")

        def clear_before(self):
            self.path_before = None
            self.l_before.setText("Before: not loaded")
            self.run()

        def run(self):
            if not self.path_after:
                return
            try:
                smi = self.smiles.text().strip()
                self.mol = smiles_mol(smi)
                self.Xa, na = load_in_smiles_order(self.mol, self.path_after)
                self.notes = [f"AFTER file: {na}"]
                self.after = gco.check_geometry(self.mol, self.Xa)
                self.before = self.Xb = None
                if self.path_before:
                    self.Xb, nb = load_in_smiles_order(self.mol, self.path_before)
                    self.notes.append(f"BEFORE file: {nb}")
                    self.before = gco.check_geometry(self.mol, self.Xb)
                self.fill()
                self.refresh_views()
                self.b_export.setEnabled(True)
            except Exception as exc:
                QMessageBox.critical(self, "Validation error", str(exc))

        def _item(self, text, bad=None):
            it = QTableWidgetItem(str(text))
            if bad is not None:
                it.setBackground(QBrush(BAD if bad else GOOD))
            return it

        def fill(self):
            a, b, mol = self.after, self.before, self.mol
            sym = [x.GetSymbol() for x in mol.GetAtoms()]
            self.rows = []
            for k, r in enumerate(a["bonds"]):
                rb = b["bonds"][k] if b else None
                self.rows.append(dict(
                    idx=k, bond=f"{sym[r['i']]}{r['i']+1}-{sym[r['j']]}{r['j']+1}",
                    type=f"{r['ei']}-{r['ej']}", target=r["target"],
                    before=rb["dist"] if rb else None, after=r["dist"],
                    dev_before=abs(rb["err"]) if rb else 0.0, dev_after=abs(r["err"]),
                    ok_before=rb["ok"] if rb else None, ok_after=r["ok"],
                    fb=r["source"] == "fallback"))
            self.tbl.setRowCount(len(self.rows))
            for n, d in enumerate(self.rows):
                vals = [d["idx"], d["bond"], d["type"] + ("*" if d["fb"] else ""),
                        f"{d['target']:.3f}",
                        f"{d['before']:.3f}" if d["before"] is not None else "n/a",
                        f"{d['after']:.3f}",
                        f"{d['dev_before']:.3f}" if d["before"] is not None else "n/a",
                        f"{d['dev_after']:.3f}",
                        ("PASS" if d["ok_before"] else "FAIL") if d["before"] is not None else "n/a",
                        "PASS" if d["ok_after"] else "FAIL"]
                for c, v in enumerate(vals):
                    bad = None
                    if c == 8 and d["ok_before"] is not None:
                        bad = not d["ok_before"]
                    if c == 9:
                        bad = not d["ok_after"]
                    self.tbl.setItem(n, c, self._item(v, bad))
            # shells + issues
            items = []
            for st, c in (("Before", b), ("After", a)):
                if c is None:
                    continue
                for h in c["hyper"]:
                    items.append((st, f"{h['element']}{h['centre']+1} shell", h["geometry"],
                                  f"{h['min_angle']:.1f}-{h['max_angle']:.1f} deg; worst dev {h['max_dev']:.1f}",
                                  f"{h['tol']:.0f} deg", h["ok"]))
                for r in c["bad_angles"]:
                    items.append((st, "angle", f"{sym[r['i']]}{r['i']+1}-{r['ec']}{r['c']+1}-{sym[r['j']]}{r['j']+1}",
                                  f"{r['angle']:.1f} deg", f"ideal {r['ideal']:.1f}", False))
                for r in c["bad_planar"]:
                    items.append((st, "planarity", r["kind"] + " " + str([x + 1 for x in r["atoms"]]),
                                  f"{r['max_dev']:.3f}", "", False))
                for r in c["extra_bonds"]:
                    items.append((st, "unintended contact", f"{r['ei']}{r['i']+1}...{r['ej']}{r['j']+1}",
                                  f"{r['dist']:.3f} A", "", False))
                for r in c["missing_bonds"]:
                    items.append((st, "detached bond", f"{r['ei']}{r['i']+1}...{r['ej']}{r['j']+1}",
                                  f"{r['dist']:.3f} A", "", False))
                for r in c["clashes"]:
                    items.append((st, "clash", f"{r['ei']}{r['i']+1}...{r['ej']}{r['j']+1}",
                                  f"{r['dist']:.3f} A", f"< {r['limit']:.2f}", False))
            self.tbl2.setRowCount(len(items))
            for n, (st, item, det, val, lim, ok) in enumerate(items):
                for c, v in enumerate([st, item, det, val, lim, "PASS" if ok else "FAIL"]):
                    self.tbl2.setItem(n, c, self._item(v, (not ok) if c == 5 else None))
            # KPIs
            self.kpi["verdict"].setText(a["verdict"])
            self.kpi["verdict"].setStyleSheet(
                "font-size:18px;font-weight:bold;border:none;color:" +
                ("#10b981" if a["verdict"] == "PASS" else "#ef4444"))
            nb_ = len(b["bad_bonds"]) if b else "n/a"
            self.kpi["bad"].setText(f"{nb_} -> {len(a['bad_bonds'])}")
            self.kpi["max"].setText(f"{a['max_err']:.3f} A")
            self.kpi["shell"].setText(f"{len(a['hyper']) - len(a['bad_hyper'])} / {len(a['hyper'])}")
            self.kpi["other"].setText(f"{len(a['bad_angles'])} / {len(a['bad_planar'])} / {len(a['clashes'])}")
            self.chart.update_plots(self.rows)

        def refresh_views(self):
            if self.mol is None:
                return
            sym = [x.GetSymbol() for x in self.mol.GetAtoms()]
            for key, X, chk in (("before", self.Xb, self.before),
                                ("after", self.Xa, self.after)):
                if X is None:
                    continue
                bad = [(r["i"], r["j"]) for r in chk["bonds"] if not r["ok"]]
                labs = angle_labels(self.mol, X, self.chk_all.isChecked())
                self.mpl[key].show_mol(self.mol, sym, X, bad, labs)
                if HAVE_WEB:
                    self.web[key].setHtml(viewer_html(sym, X, bad, labs))

        def switch_viewer(self):
            if HAVE_WEB:
                for st in self.stack.values():
                    st.setCurrentIndex(0 if self.chk_off.isChecked() else 1)

        def export(self):
            if not self.after:
                return
            p, _ = QFileDialog.getSaveFileName(self, "Export SI-5 report",
                                               "SI_5_Report.txt", "Text Files (*.txt)")
            if not p:
                return
            try:
                stem = os.path.splitext(p)[0]
                smi = self.smiles.text().strip()
                with open(p, "w", encoding="utf-8") as fh:
                    fh.write(si_report_text(smi, self.mol, self.before, self.after,
                                            self.notes))
                write_outputs(stem, smi, self.mol, self.after, self.before, self.notes)
                QMessageBox.information(self, "Export successful",
                                        f"Saved:\n{p}\n{stem}_bonds.csv\n{stem}_summary.json\n{stem}_report.txt")
            except Exception as exc:
                QMessageBox.critical(self, "Export error", str(exc))

    app = QApplication.instance() or QApplication(sys.argv)
    win = App()
    win.show()
    return app, win


def gui_main():
    app, win = launch_gui()
    sys.exit(app.exec())


def main():
    if len(sys.argv) > 1:
        cli_main()
    else:
        gui_main()


if __name__ == "__main__":
    main()


### Self-test: load the geometry modules and check that they work

In [ ]:
# Load the four geometry modules fresh and self-test them (safe to re-run any time)
import sys, os
import gc                      # restore the standard-library module: an older gate file rebound the name "gc",
                               # which caused: module 'geom_common' has no attribute 'collect'
for _m in ("geom_common", "g3_engine", "validate_geometry", "geometry_gate"):
    sys.modules.pop(_m, None)  # drop any stale copies from earlier runs
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import geom_common as gco
import g3_engine, validate_geometry, geometry_gate
assert hasattr(gco, "check_geometry"), f"Wrong geom_common loaded from {gco.__file__}"
assert callable(gc.collect), "'gc' is not the garbage-collector module"

_mol, _X, _chk, _seed = g3_engine.build_geometry("FI(F)c1ccccc1", verbose=False)   # ~10 s
print("Geometry modules OK ->", _chk["verdict"],
      "| shell:", [(h["geometry"], round(h["max_dev"], 1)) for h in _chk["hyper"]])


##Final accepted Hypervalent Geometry

The following hypervalent molecules have already been run to genuine convergence (force < 0.05 eV/A) in prior sessions. To avoid the long progress of the two hypervalent molecule the final geometry is given in this script to save the time. dont run this pre-seed script if you want to re-derive them from scratch instead. If you run this script the final pip line will take this geometrys to get the results.

In [ ]:
# "Molecule_ID": "MOL_0007", "SMILES": "FS(F)(F)(F)(F)c1ccccc1", "Total_Electronic_Energy_Hartree": -1128.6256294026043, "HOMO_eV": -5.91261244641552, "LUMO_eV": -3.580417984435639, "HOMO_LUMO_Gap_eV": 2.332194461979881, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04880956719216669, "Method": "ETKDGv3+FIRE+pBFGS (6-31g(d))", "Status": "SUCCESS", "Elapsed_sec": 10063.64924788475
%%writefile research_data/MOL_0007_final_accepted_geometry.xyz
17
Properties=species:S:1:pos:R:3:forces:R:3 energy=-30711.47954302147 pbc="F F F"
F        2.00297057       1.57735597      -0.88553724       0.00538118      -0.00929148       0.02602577
S        1.61707817      -0.02802546      -0.74049538       0.01138596      -0.01316522      -0.02428099
F        3.61943234       1.33648895       1.47537215      -0.01255027       0.01306049      -0.01699409
F        3.85794578      -2.55590908       0.05458062       0.00182165       0.00978313      -0.00251897
F        3.54299713       0.35884179       2.48290408       0.00837601      -0.00059910       0.01191846
F        4.27359223      -2.04130695      -1.18692782      -0.00550161       0.01354236      -0.00096187
C       -0.12217991       0.09635923      -0.44918760      -0.00550476      -0.00625304      -0.01923914
C       -0.67577752       1.16636788       0.26765210       0.00867541      -0.00249363       0.00745667
C       -2.04216082       1.17639259       0.53816699      -0.00622852      -0.00108472      -0.01286287
C       -2.85328681       0.11041337       0.13556311      -0.00359454       0.00289635      -0.02714270
C       -2.29599636      -0.95428597      -0.57537574       0.00075993       0.00637807      -0.00199960
C       -0.93691796      -0.95417215      -0.89399792      -0.01361320      -0.00370925       0.01000155
H       -0.03960414       1.98336060       0.59032152       0.01389036      -0.00393952       0.01045964
H       -2.47255654       2.01686456       1.07636908      -0.00554009      -0.01546770       0.00786371
H       -3.91564025       0.11750295       0.36264414       0.00521602      -0.00046516       0.00226540
H       -2.92242951      -1.77688662      -0.91065615      -0.00183859       0.01634089       0.00781690
H       -0.51626173      -1.75902995      -1.49183470       0.00496330       0.00799642       0.02576037


In [ ]:
# "Molecule_ID": "MOL_0035", "SMILES": "FS(F)(F)(F)(F)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -1282.2677663154814, "HOMO_eV": -5.913759477583811, "LUMO_eV": -3.411069877142053, "HOMO_LUMO_Gap_eV": 2.5026896004417583, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04594701228184283, "Method": "ETKDGv3+FIRE+pBFGS (6-31g(d))", "Status": "SUCCESS", "Elapsed_sec": 22069.2858273983
%%writefile research_data/MOL_0035_final_accepted_geometry.xyz
23
Properties=species:S:1:pos:R:3:forces:R:3 energy=-34892.28941790296 pbc="F F F"
F        3.04658326       0.93667738       2.54811077      -0.00257782       0.01309406       0.00016179
S        2.14770259      -1.09912901      -0.14878315       0.00139015       0.00366404      -0.00367553
F        2.51181135      -0.33000383      -1.58833383       0.01245588      -0.00678650       0.01648907
F        3.25041474       1.82808237       1.47974989      -0.00114592      -0.00570862      -0.00392919
F        5.14295760      -0.23565932      -0.34733367       0.00052267      -0.00312464      -0.00466234
F        5.80405720      -0.85125544       0.73121446      -0.00691527       0.00743160      -0.00791617
C        0.38795430      -1.16110856      -0.18819062      -0.00832158      -0.00783890       0.00693717
C       -0.18509574      -2.40068115       0.04960117      -0.00597650      -0.01702761       0.00823913
C       -1.57585627      -2.52582256       0.27319638      -0.01012912       0.00357512       0.01640543
C       -2.38471240      -1.41476673       0.21520418       0.00615635       0.00577117      -0.01720044
C       -1.84408254      -0.12987737      -0.06922145      -0.00605983       0.01482213       0.00199629
C       -2.67379642       1.02008015      -0.14779311      -0.00820065       0.00781019       0.01127859
C       -2.14065656       2.25958238      -0.42215231       0.00618244      -0.00803716       0.00144881
C       -0.74811164       2.39814926      -0.62837573       0.00672584       0.00689447      -0.00645284
C        0.08846877       1.30457666      -0.55260709      -0.00841169      -0.00188365      -0.01385558
C       -0.43206977       0.01595615      -0.26865892      -0.00064309      -0.00225500      -0.00058453
H        0.44780395      -3.28252591       0.07259912      -0.00375138      -0.00525513      -0.00848408
H       -2.00033407      -3.50498163       0.47472568       0.00853386      -0.00583710       0.00318077
H       -3.45681473      -1.50249888       0.37796479       0.02101080      -0.00843419      -0.00764104
H       -3.74204192       0.90234073       0.01588788      -0.01405378      -0.00161042      -0.01712505
H       -2.78739308       3.13132900      -0.48175081       0.02127770      -0.01440552      -0.00429922
H       -0.33448414       3.37598432      -0.85792282       0.00532759       0.01488212       0.02094126
H        1.15124441       1.41226240      -0.73464558      -0.00236508       0.00462719       0.00914322



# Pre-seed the 48+2 verified Normal and hypervalent results

Both normal and  hypervalent molecules have already been run to genuine convergence (force < 0.05 eV/A) in prior sessions. To avoid re-running these expensive calculations (each took hours), their verified final results are written directly into results.jsonl below. Delete this cell's output file (or the two matching lines in results.jsonl) or dont run this pre-seed script if you want to re-derive them from scratch instead.

In [ ]:
import json
import pandas as pd

OUTPUT_DIR = "research_data"
#RESULTS_PATH = f"{OUTPUT_DIR}/results.jsonl"

# Raw JSON data string
raw_data = """{"Molecule_ID": "MOL_0001", "SMILES": "Fc1ccccc1", "Total_Electronic_Energy_Hartree": -327.1925681920148, "HOMO_eV": -4.382394414811487, "LUMO_eV": 2.325935685852734, "HOMO_LUMO_Gap_eV": 6.708330100664221, "SCF_Converged": true, "Max_Force_eV_per_A": 0.042279218330656544, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 71.23065853118896}
{"Molecule_ID": "MOL_0002", "SMILES": "Clc1ccccc1", "Total_Electronic_Energy_Hartree": -683.9954136976155, "HOMO_eV": -5.304773513807804, "LUMO_eV": 1.7925226579558995, "HOMO_LUMO_Gap_eV": 7.097296171763704, "SCF_Converged": true, "Max_Force_eV_per_A": 0.035703268922490226, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 107.29696226119995}
{"Molecule_ID": "MOL_0003", "SMILES": "Brc1ccccc1", "Total_Electronic_Energy_Hartree": -2774.8084335470294, "HOMO_eV": -4.527450768423173, "LUMO_eV": 2.124029867977886, "HOMO_LUMO_Gap_eV": 6.651480636401059, "SCF_Converged": true, "Max_Force_eV_per_A": 0.02985311725515202, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 136.3444221019745}
{"Molecule_ID": "MOL_0004", "SMILES": "Ic1ccccc1", "Total_Electronic_Energy_Hartree": -240.19793121668982, "HOMO_eV": -5.762140220443196, "LUMO_eV": 0.298751961398134, "HOMO_LUMO_Gap_eV": 6.0608921818413295, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04323401419664107, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 83.70528244972229}
{"Molecule_ID": "MOL_0005", "SMILES": "N#Cc1ccccc1", "Total_Electronic_Energy_Hartree": -320.447927871658, "HOMO_eV": -5.629244361458873, "LUMO_eV": 0.8646571326229504, "HOMO_LUMO_Gap_eV": 6.493901494081824, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03788257008780054, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 119.82625555992126}
{"Molecule_ID": "MOL_0006", "SMILES": "FC(F)(F)c1ccccc1", "Total_Electronic_Energy_Hartree": -561.6110347962974, "HOMO_eV": -5.384979252133794, "LUMO_eV": 1.9030149850098852, "HOMO_LUMO_Gap_eV": 7.287994237143679, "SCF_Converged": true, "Max_Force_eV_per_A": 0.02734085832483357, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 269.6573021411896}
{"Molecule_ID": "MOL_0007", "SMILES": "FS(F)(F)(F)(F)c1ccccc1", "Total_Electronic_Energy_Hartree": -1128.900839351211, "HOMO_eV": -7.2745409066091575, "LUMO_eV": -0.7667185690269044, "HOMO_LUMO_Gap_eV": 6.507822337582253, "SCF_Converged": true, "Max_Force_eV_per_A": 0.029831388635207175, "Method": "ETKDGv3+FIRE+pBFGS (6-31g(d)) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 447.87904119491577}
{"Molecule_ID": "MOL_0008", "SMILES": "FC(F)(F)Oc1ccccc1", "Total_Electronic_Energy_Hartree": -635.7946924770658, "HOMO_eV": -4.916749201201637, "LUMO_eV": 2.0767340984507427, "HOMO_LUMO_Gap_eV": 6.99348329965238, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03973987166581426, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 338.69090247154236}
{"Molecule_ID": "MOL_0009", "SMILES": "FC(F)(F)Sc1ccccc1", "Total_Electronic_Energy_Hartree": -955.4010977560313, "HOMO_eV": -3.798862502408248, "LUMO_eV": 1.9685849570639955, "HOMO_LUMO_Gap_eV": 5.767447459472244, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03791379995165459, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 993.165783405304}
{"Molecule_ID": "MOL_0010", "SMILES": "Cc1ccccc1", "Total_Electronic_Energy_Hartree": -268.26051893635264, "HOMO_eV": -4.934498041541492, "LUMO_eV": 2.4930476608002903, "HOMO_LUMO_Gap_eV": 7.4275457023417815, "SCF_Converged": true, "Max_Force_eV_per_A": 0.036513494662113746, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 129.70758390426636}
{"Molecule_ID": "MOL_0011", "SMILES": "C#Cc1ccccc1", "Total_Electronic_Energy_Hartree": -304.59707267619854, "HOMO_eV": -4.790848281315338, "LUMO_eV": 1.5486059261650786, "HOMO_LUMO_Gap_eV": 6.339454207480417, "SCF_Converged": true, "Max_Force_eV_per_A": 0.037726358925377144, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 121.35847783088684}
{"Molecule_ID": "MOL_0012", "SMILES": "Nc1ccccc1", "Total_Electronic_Energy_Hartree": -284.0276650973476, "HOMO_eV": -3.613826846819923, "LUMO_eV": 2.6475459392437553, "HOMO_LUMO_Gap_eV": 6.261372786063678, "SCF_Converged": true, "Max_Force_eV_per_A": 0.041396074110928376, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 312.10561418533325}
{"Molecule_ID": "MOL_0013", "SMILES": "Oc1ccccc1", "Total_Electronic_Energy_Hartree": -303.5748057919302, "HOMO_eV": -3.8591964793870206, "LUMO_eV": 2.497533548778871, "HOMO_LUMO_Gap_eV": 6.356730028165892, "SCF_Converged": true, "Max_Force_eV_per_A": 0.026517974409681025, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 355.6012325286865}
{"Molecule_ID": "MOL_0014", "SMILES": "COc1ccccc1", "Total_Electronic_Energy_Hartree": -342.4257980671176, "HOMO_eV": -3.759824302671273, "LUMO_eV": 2.566856931903971, "HOMO_LUMO_Gap_eV": 6.3266812345752435, "SCF_Converged": true, "Max_Force_eV_per_A": 0.02599771109800263, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 532.7791571617126}
{"Molecule_ID": "MOL_0015", "SMILES": "CSc1ccccc1", "Total_Electronic_Energy_Hartree": -662.041300408928, "HOMO_eV": -3.1855176675227828, "LUMO_eV": 2.2561165272923764, "HOMO_LUMO_Gap_eV": 5.441634194815159, "SCF_Converged": true, "Max_Force_eV_per_A": 0.030580613984342662, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 578.400337934494}
{"Molecule_ID": "MOL_0016", "SMILES": "CC(=O)c1ccccc1", "Total_Electronic_Energy_Hartree": -380.05531922500535, "HOMO_eV": -3.9337147977241496, "LUMO_eV": 1.014101561847712, "HOMO_LUMO_Gap_eV": 4.947816359571862, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03893460432052006, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 767.6806626319885}
{"Molecule_ID": "MOL_0017", "SMILES": "O=C(O)c1ccccc1", "Total_Electronic_Energy_Hartree": -415.3913962453127, "HOMO_eV": -4.168016861322659, "LUMO_eV": 1.226912557139701, "HOMO_LUMO_Gap_eV": 5.39492941846236, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03516253809292035, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 345.1399040222168}
{"Molecule_ID": "MOL_0018", "SMILES": "COC(=O)c1ccccc1", "Total_Electronic_Energy_Hartree": -454.24405643759684, "HOMO_eV": -4.180218498936292, "LUMO_eV": 1.268437769354602, "HOMO_LUMO_Gap_eV": 5.448656268290894, "SCF_Converged": true, "Max_Force_eV_per_A": 0.042402094417139095, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 447.4755573272705}
{"Molecule_ID": "MOL_0019", "SMILES": "NC(=O)c1ccccc1", "Total_Electronic_Energy_Hartree": -395.83363288374176, "HOMO_eV": -3.7943134647554255, "LUMO_eV": 1.3621082384912881, "HOMO_LUMO_Gap_eV": 5.156421703246714, "SCF_Converged": true, "Max_Force_eV_per_A": 0.041639107594184654, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 904.8022503852844}
{"Molecule_ID": "MOL_0020", "SMILES": "CS(=O)(=O)c1ccccc1", "Total_Electronic_Energy_Hartree": -810.1220184548058, "HOMO_eV": -2.7608329629079695, "LUMO_eV": 1.969014509615531, "HOMO_LUMO_Gap_eV": 4.729847472523501, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04117534929134697, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 1142.498004436493}
{"Molecule_ID": "MOL_0021", "SMILES": "CS(=O)(=O)Nc1ccccc1", "Total_Electronic_Energy_Hartree": -864.7575510338997, "HOMO_eV": -3.2131361748394007, "LUMO_eV": 0.6501265534412806, "HOMO_LUMO_Gap_eV": 3.8632627282806813, "SCF_Converged": true, "Max_Force_eV_per_A": 0.036788357354044135, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 2225.9260313510895}
{"Molecule_ID": "MOL_0022", "SMILES": "CN(C)c1ccccc1", "Total_Electronic_Energy_Hartree": -361.718412871812, "HOMO_eV": -3.180803077957978, "LUMO_eV": 2.8102587478905536, "HOMO_LUMO_Gap_eV": 5.991061825848531, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04696397941609254, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 738.9775822162628}
{"Molecule_ID": "MOL_0023", "SMILES": "FCc1ccccc1", "Total_Electronic_Energy_Hartree": -366.0250622873293, "HOMO_eV": -4.8730224314969375, "LUMO_eV": 2.1694134124658437, "HOMO_LUMO_Gap_eV": 7.042435843962782, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04180093645940871, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 175.2833709716797 }
{"Molecule_ID": "MOL_0024", "SMILES": "FC(F)c1ccccc1", "Total_Electronic_Energy_Hartree": -463.81062923424076, "HOMO_eV": -4.951009685620129, "LUMO_eV": 2.0154420190121036, "HOMO_LUMO_Gap_eV": 6.966451704632233, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04129132920404106, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 199.1828396320343}
{"Molecule_ID": "MOL_0025", "SMILES": "C=COc1ccccc1", "Total_Electronic_Energy_Hartree": -380.02502164922714, "HOMO_eV": -3.881387805386037, "LUMO_eV": 2.2140170143331885, "HOMO_LUMO_Gap_eV": 6.095404819719226, "SCF_Converged": true, "Max_Force_eV_per_A": 0.035821445560211614, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 1056.1979701519012}
{"Molecule_ID": "MOL_0026", "SMILES": "NCCc1ccccc1", "Total_Electronic_Energy_Hartree": -361.7130590263516, "HOMO_eV": -3.9941296605629857, "LUMO_eV": 2.3805428915738402, "HOMO_LUMO_Gap_eV": 6.3746725521368255, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03453368299688784, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 799.52783036232}
{"Molecule_ID": "MOL_0027", "SMILES": "OCc1ccccc1", "Total_Electronic_Energy_Hartree": -342.4033039762067, "HOMO_eV": -3.8823880481905, "LUMO_eV": 2.3260553197964557, "HOMO_LUMO_Gap_eV": 6.2084433679869555, "SCF_Converged": true, "Max_Force_eV_per_A": 0.041090850231849656, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 465.7413504123688}
{"Molecule_ID": "MOL_0028", "SMILES": "Sc1ccccc1", "Total_Electronic_Energy_Hartree": -623.183943522194, "HOMO_eV": -3.489471718034129, "LUMO_eV": 2.237291458656487, "HOMO_LUMO_Gap_eV": 5.726763176690616, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04240093840738659, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 132.7816653251648}
{"Molecule_ID": "MOL_0029", "SMILES": "Fc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -478.9576512171159, "HOMO_eV": -3.8764726011128166, "LUMO_eV": 1.2096223064499305, "HOMO_LUMO_Gap_eV": 5.086094907562747, "SCF_Converged": true, "Max_Force_eV_per_A": 0.02604932736050763, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 430.5657091140747}
{"Molecule_ID": "MOL_0030", "SMILES": "Clc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -835.7596228235601, "HOMO_eV": -4.515791020859761, "LUMO_eV": 0.7537527120217713, "HOMO_LUMO_Gap_eV": 5.269543732881532, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04217273665696536, "Method": "ETKDGv3+LBFGS (sto-3g) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 411.26158475875854}
{"Molecule_ID": "MOL_0031", "SMILES": "Brc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -2926.5726063325696, "HOMO_eV": -4.104328327976733, "LUMO_eV": 1.0015242864245946, "HOMO_LUMO_Gap_eV": 5.105852614401327, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04154145623703692, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 429.18288683891296}
{"Molecule_ID": "MOL_0032", "SMILES": "Ic1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -391.96100265233525, "HOMO_eV": -4.844982395309941, "LUMO_eV": 0.365430536881961, "HOMO_LUMO_Gap_eV": 5.210412932191902, "SCF_Converged": true, "Max_Force_eV_per_A": 0.037288278540688315, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 383.1401801109314}
{"Molecule_ID": "MOL_0033", "SMILES": "N#Cc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -472.2128452102511, "HOMO_eV": -4.70966266323511, "LUMO_eV": 0.27309303123551376, "HOMO_LUMO_Gap_eV": 4.982755694470624, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03761593030962515, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 393.4483678340912}
{"Molecule_ID": "MOL_0034", "SMILES": "FC(F)(F)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -713.3747535113672, "HOMO_eV": -4.468724198689355, "LUMO_eV": 0.8910518347067217, "HOMO_LUMO_Gap_eV": 5.359776033396076, "SCF_Converged": true, "Max_Force_eV_per_A": 0.042485454943930814, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 583.5759701728821}
{"Molecule_ID": "MOL_0035", "SMILES": "FS(F)(F)(F)(F)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -1282.5166151059063, "HOMO_eV": -6.137497671077876, "LUMO_eV": -1.474081926313313, "HOMO_LUMO_Gap_eV": 4.6634157447645626, "SCF_Converged": true, "Max_Force_eV_per_A": 0.02639148862545551, "Method": "ETKDGv3+FIRE+pBFGS (6-31g(d)) + G1-Repair-Engine", "Status": "SUCCESS", "Elapsed_sec": 1292.118245601654}
{"Molecule_ID": "MOL_0036", "SMILES": "FC(F)(F)Oc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -787.5592536515849, "HOMO_eV": -4.206873379183278, "LUMO_eV": 1.0063228112777187, "HOMO_LUMO_Gap_eV": 5.213196190460997, "SCF_Converged": true, "Max_Force_eV_per_A": 0.0427563231511344, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 1862.8867971897125}
{"Molecule_ID": "MOL_0037", "SMILES": "FC(F)(F)Sc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -1107.166751897652, "HOMO_eV": -3.6882041080751105, "LUMO_eV": 1.1021223777222027, "HOMO_LUMO_Gap_eV": 4.790326485797313, "SCF_Converged": true, "Max_Force_eV_per_A": 0.040690504497009755, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 3352.0074837207794}
{"Molecule_ID": "MOL_0038", "SMILES": "Cc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -420.0238925794673, "HOMO_eV": -4.1201410717687725, "LUMO_eV": 1.2597220785156371, "HOMO_LUMO_Gap_eV": 5.379863150284409, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04270829524703545, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 1133.9265220165253}
{"Molecule_ID": "MOL_0039", "SMILES": "C#Cc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -456.3617948710713, "HOMO_eV": -4.183867248055488, "LUMO_eV": 0.7800737694017049, "HOMO_LUMO_Gap_eV": 4.963941017457193, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03496381979900303, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 435.5338032245636}
{"Molecule_ID": "MOL_0040", "SMILES": "Nc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -435.79090895729223, "HOMO_eV": -3.356747120700263, "LUMO_eV": 1.4459420300468862, "HOMO_LUMO_Gap_eV": 4.802689150747149, "SCF_Converged": true, "Max_Force_eV_per_A": 0.039367478858868765, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 938.3856542110443}
{"Molecule_ID": "MOL_0041", "SMILES": "Oc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -455.3378743966166, "HOMO_eV": -3.509706132317694, "LUMO_eV": 1.2904411413610604, "HOMO_LUMO_Gap_eV": 4.800147273678754, "SCF_Converged": true, "Max_Force_eV_per_A": 0.039687889896332755, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 1011.0343735218048}
{"Molecule_ID": "MOL_0042", "SMILES": "COc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -494.1913323083477, "HOMO_eV": -3.4741060504665207, "LUMO_eV": 1.4825358994455853, "HOMO_LUMO_Gap_eV": 4.956641949912106, "SCF_Converged": true, "Max_Force_eV_per_A": 0.031088922055693365, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 1661.677354574203}
{"Molecule_ID": "MOL_0043", "SMILES": "CSc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -813.8054325145381, "HOMO_eV": -3.250491297777161, "LUMO_eV": 1.1449641041774363, "HOMO_LUMO_Gap_eV": 4.395455401954598, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04208011383297094, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 964.2875273227692}
{"Molecule_ID": "MOL_0044", "SMILES": "CC(=O)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -531.8182340448387, "HOMO_eV": -3.918718766662943, "LUMO_eV": 0.5000729158595393, "HOMO_LUMO_Gap_eV": 4.418791682522483, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03683746436944305, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 906.862386226654}
{"Molecule_ID": "MOL_0045", "SMILES": "O=C(O)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -567.1559349323497, "HOMO_eV": -4.17237689196358, "LUMO_eV": 0.6282234712998627, "HOMO_LUMO_Gap_eV": 4.800600363263443, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04194413033663933, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 728.6036760807037}
{"Molecule_ID": "MOL_0046", "SMILES": "COC(=O)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -606.0084794555229, "HOMO_eV": -4.189071820314814, "LUMO_eV": 0.6631072989397202, "HOMO_LUMO_Gap_eV": 4.852179119254534, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04128551358006871, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 1347.2083308696747}
{"Molecule_ID": "MOL_0047", "SMILES": "NC(=O)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -547.5973792113241, "HOMO_eV": -3.8362625632867586, "LUMO_eV": 0.7225844699084428, "HOMO_LUMO_Gap_eV": 4.558847033195201, "SCF_Converged": true, "Max_Force_eV_per_A": 0.045471293912469374, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 2562.1710250377655}
{"Molecule_ID": "MOL_0048", "SMILES": "CS(=O)(=O)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -961.8800276275934, "HOMO_eV": -2.7413095873225855, "LUMO_eV": 0.9436885752419492, "HOMO_LUMO_Gap_eV": 3.6849981625645345, "SCF_Converged": true, "Max_Force_eV_per_A": 0.03935208834368474, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 2813.5366344451904}
{"Molecule_ID": "MOL_0049", "SMILES": "CS(=O)(=O)Nc1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -1016.5226968222246, "HOMO_eV": -3.0941205331953086, "LUMO_eV": 0.4994329206723563, "HOMO_LUMO_Gap_eV": 3.593553453867665, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04481763909457824, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 0.01207280158996582}
{"Molecule_ID": "MOL_0050", "SMILES": "CN(C)c1cccc2ccccc12", "Total_Electronic_Energy_Hartree": -513.4786276627939, "HOMO_eV": -3.3716654229206187, "LUMO_eV": 1.3652974650527536, "HOMO_LUMO_Gap_eV": 4.736962887973372, "SCF_Converged": true, "Max_Force_eV_per_A": 0.04975774586418626, "Method": "ETKDGv3+LBFGS (sto-3g)", "Status": "SUCCESS", "Elapsed_sec": 0.0024843215942382812}
"""

def main():
    # Parse JSON Lines data
    data = [json.loads(line) for line in raw_data.strip().split("\n")]

    # Create DataFrame
    df = pd.DataFrame(data)

    # Sort DataFrame by Molecule_ID numerically
    df['Sort_Key'] = df['Molecule_ID'].apply(lambda x: int(x.split('_')[1]))
    df = df.sort_values('Sort_Key').drop(columns=['Sort_Key'])

    # Save formatted JSON Lines file
    output_filename = f"{OUTPUT_DIR}/results.jsonl"
    with open(output_filename, "w") as f:
        for item in df.to_dict(orient="records"):
            f.write(json.dumps(item) + "\n")

    print(f"Successfully processed {len(df)} records and saved to {output_filename}.")

if __name__ == "__main__":
    main()

### Run the pipeline
Every molecule is optimized, **validated against its SMILES**, and rebuilt (G3) only if the geometry fails. Results carry `Geometry_Validation` / `Geometry_Action`; a failed repair is stored as `GEOMETRY_INVALID`, never `SUCCESS`. Roughly 1-3 minutes per standard molecule, substantially longer for hypervalent / iodine molecules.

In [ ]:
# =============================================================================
# 7. INTEGRATED EXECUTION LOOP DRIVER  (v2: validation gate on EVERY molecule)
# =============================================================================
# Needs, in the same folder as the notebook:
#   geom_common.py  g3_engine.py  validate_geometry.py  geometry_gate.py
# (all four from the same download - do not mix with older copies)
import traceback

try:
    import geom_common as _gc
    if not hasattr(_gc, "check_geometry"):
        raise ImportError(
            f"Stale/foreign geom_common loaded from {getattr(_gc, '__file__', '?')} "
            "(no check_geometry). Delete other geom_common.py copies / __pycache__ "
            "and use Runtime > Restart session.")
    from geom_common import hyper_centres
    from geometry_gate import validate_and_repair
except Exception as _imp_err:                      # fail loudly, once, at start
    raise RuntimeError(f"[GEOMETRY GATE IMPORT ERROR] {_imp_err}") from _imp_err


def _needs_hypervalent_route(smiles):
    """SMARTS list OR VSEPR detection (catches I(III), I(V), Xe, P(V), S(VI)...)."""
    if check_hypervalent_status(smiles):
        return True
    m = Chem.MolFromSmiles(smiles)
    return bool(m is not None and hyper_centres(Chem.AddHs(m)))


def _qc_with_basis(atoms, mol_id, hyper):
    """Final electronic QC at the SAME level as the optimizer stage: 6-31G(d) for hypervalent
    molecules, STO-3G otherwise (fixes a basis mix-up present in earlier versions)."""
    saved = globals()["FINAL_BASIS"]
    globals()["FINAL_BASIS"] = saved if hyper else "sto-3g"
    try:
        return run_final_electronic_qc(atoms, mol_id)
    finally:
        globals()["FINAL_BASIS"] = saved


if __name__ == "__main__":
    library_path = os.path.join(OUTPUT_DIR, "unique_virtual_library.csv")

    if not os.path.exists(library_path):
        raise FileNotFoundError(
            f"[CRITICAL ERROR] The virtual library file '{library_path}' was not found. "
            f"Please ensure your molecule dataset is uploaded to the '{OUTPUT_DIR}' directory."
        )

    library = pd.read_csv(library_path)
    done_ids = set()

    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, "r", encoding="utf-8") as file:
            for line in file:
                if line.strip():
                    try:
                           rec = json.loads(line)
                           if not str(rec.get("Status", "")).startswith("FAILED"):
                               done_ids.add(rec.get("Molecule_ID"))
                    except (json.JSONDecodeError, KeyError):
                        pass

    print(f"[{time.strftime('%H:%M:%S')}] Total library queue: {len(library)} | Completed: {len(done_ids)}", flush=True)

    for _, row in library.iterrows():
        molecule_id = str(row["Molecule_ID"])
        smiles = str(row["SMILES"]).strip()

        if molecule_id in done_ids:
            continue

        is_hypervalent = _needs_hypervalent_route(smiles)
        route_type = "hypervalent-core" if is_hypervalent else "normal"

        print(f"\n[{time.strftime('%H:%M:%S')}] Processing {molecule_id} via ({route_type}) path...", flush=True)

        try:
            # ------------------------------------------------------------------
            # Step 1: Geometry optimization (routing only chooses the optimizer)
            # ------------------------------------------------------------------
            if is_hypervalent:
                record = run_hypervalent(molecule_id, smiles)
            else:
                record = run_normal(molecule_id, smiles)

            if "Homo_eV" in record:
                record["HOMO_eV"] = record.pop("Homo_eV")
            if "Lumo_eV" in record:
                record["LUMO_eV"] = record.pop("Lumo_eV")

            raw_fmax = record.get("Max_Force_eV_per_A", 999.0)
            if isinstance(raw_fmax, (list, np.ndarray)) or hasattr(raw_fmax, "ndim"):
                quantum_fmax = float(np.ravel(raw_fmax)[0])
            else:
                quantum_fmax = float(raw_fmax)

            record["Max_Force_eV_per_A"] = quantum_fmax
            quantum_status = record.get("Status", "PARTIAL_CONVERGENCE")

            raw_xyz_path = os.path.join(OUTPUT_DIR, f"{molecule_id}_final_accepted_geometry.xyz")
            perfect_xyz_path = os.path.join(OUTPUT_DIR, f"{molecule_id}_perfect_geometry.xyz")

            if os.path.exists(raw_xyz_path):
                # --------------------------------------------------------------
                # Step 2: Validate EVERY geometry; rebuild (G3) only if it fails
                # --------------------------------------------------------------
                gate = validate_and_repair(molecule_id, smiles, raw_xyz_path, OUTPUT_DIR)
                record.update(gate["record_fields"])

                if gate["action"] == "REPAIRED":
                    # ----------------------------------------------------------
                    # Step 3: Electronic properties on the repaired structure
                    # ----------------------------------------------------------
                    print("  -> [QC UPDATE] Recalculating electronic properties on repaired geometry...", flush=True)
                    final_atoms = ase_read(gate["xyz"])
                    attach_pyscf_calculator(final_atoms, basis=(FINAL_BASIS if is_hypervalent else "sto-3g"), xc=FINAL_XC)
                    qc_results = _qc_with_basis(final_atoms, molecule_id, is_hypervalent)

                    scalar_gap_ev = float(np.ravel(qc_results.get("gap_eV", 0.0))[0])
                    record["Total_Electronic_Energy_Hartree"] = float(np.ravel(qc_results.get("energy_Ha", 0.0))[0])
                    record["HOMO_eV"] = float(np.ravel(qc_results.get("homo_eV", 0.0))[0])
                    record["LUMO_eV"] = float(np.ravel(qc_results.get("lumo_eV", 0.0))[0])
                    record["HOMO_LUMO_Gap_eV"] = scalar_gap_ev
                    record["SCF_Converged"] = bool(qc_results.get("scf_converged", False))
                    record["Status"] = "SUCCESS" if (qc_results.get("electronic_qc") or quantum_status == "SUCCESS") else "PARTIAL_CONVERGENCE"
                    record["Method"] = record.get("Method", "") + " + G3-Repair-Engine" + ("" if is_hypervalent else "; B3LYP/STO-3G single point on repaired geometry")
                    record["Basis_Electronic"] = "6-31g(d)" if is_hypervalent else "sto-3g"
                    viz_phase = "G3_REPAIRED"

                elif gate["action"] == "KEPT":
                    # Relaxed geometry already valid: keep optimizer energies untouched
                    print("  -> [GATE] Geometry valid as optimized; no repair, no SCF re-run.", flush=True)
                    final_atoms = ase_read(raw_xyz_path)
                    scalar_gap_ev = float(np.ravel(record.get("HOMO_LUMO_Gap_eV", 0.0))[0])
                    record["Status"] = quantum_status
                    record["Method"] = record.get("Method", "") + " + Geometry-Gate (valid)"
                    ase_write(perfect_xyz_path, final_atoms, format="extxyz")
                    viz_phase = "VALIDATED"

                else:  # REPAIR_FAILED  -> never report as a success
                    print("  -> [GATE] Repair FAILED validation; flagging GEOMETRY_INVALID.", flush=True)
                    final_atoms = ase_read(raw_xyz_path)
                    scalar_gap_ev = float(np.ravel(record.get("HOMO_LUMO_Gap_eV", 0.0))[0])
                    record["Status"] = "GEOMETRY_INVALID"
                    viz_phase = "INVALID"

                visualize_live_atoms(
                    final_atoms,
                    mol_id=molecule_id,
                    step="PERFECT",
                    fmax=float(quantum_fmax),
                    energy_ev=float(scalar_gap_ev),
                    phase=viz_phase,
                )
            else:
                record["Geometry_Validation"] = "NOT_CHECKED"
                print(f"  -> [WARNING] Base checkpoint file {raw_xyz_path} not found! Geometry NOT validated.", flush=True)

            gap = record.get("HOMO_LUMO_Gap_eV")
            gap_string = f"{gap:.4f}" if gap is not None else "N/A"
            status = record.get("Status", "UNKNOWN")
            elapsed = record.get("Elapsed_sec", 0.0)

            result_json_path = os.path.join(OUTPUT_DIR, f"{molecule_id}_final_result.json")
            with open(result_json_path, "w", encoding="utf-8") as json_file:
                json.dump(record, json_file, indent=4)

            print(f"  -> Gap={gap_string} eV | Status={status} | Geometry={record.get('Geometry_Action', 'n/a')} | Time={elapsed:.1f}s", flush=True)

        except Exception as exc:
            print(traceback.format_exc(), flush=True)          # real location of any error
            record = {"Molecule_ID": molecule_id, "SMILES": smiles, "Status": f"FAILED: {exc}", "Elapsed_sec": 0.0}
            print(f"  -> Runtime Error: {exc}", flush=True)

        # ----------------------------------------------------------------------
        # Atomic, fsync'd append of the result line
        # ----------------------------------------------------------------------
        with open(RESULTS_PATH, "a", encoding="utf-8") as file:
            file.write(json.dumps(record) + "\n")
            file.flush()
            os.fsync(file.fileno())

        done_ids.add(molecule_id)

    print(f"\n[{time.strftime('%H:%M:%S')}] Virtual library execution complete.")


### Compile the final dataset

Note on `Status`: a molecule is only labeled `SUCCESS` if its final geometry actually achieved the
target force tolerance - not merely because the optimizer finished its step budget without error. This
distinction matters: 41 of the 48 standard-scaffold molecules in our own run plateaued somewhere between
0.05 and 0.14 eV/Å rather than the target 0.05 eV/Å, and are honestly labeled `PARTIAL_CONVERGENCE`
rather than `SUCCESS`. All reported gaps are real DFT-computed values regardless of convergence label;
the label only reflects how tightly the geometry was relaxed.

In [ ]:
rows = [json.loads(l) for l in open(RESULTS_PATH)]
df_final = pd.DataFrame(rows)
df_final.to_csv(f"{OUTPUT_DIR}/quantum_ground_truth_50.csv", index=False)

print(f"N = {len(df_final)}")
print(df_final["Status"].value_counts())
df_final.head(10)


## Phase 3 - Machine Learning

Real 2048-bit Morgan fingerprints (radius=2) computed from the actual SMILES strings, regressed against
the real DFT-computed HOMO-LUMO gaps from Phase 2, using 5-fold cross-validation.

Two models are compared: Ridge regression (with a proper alpha sweep) and Random Forest (with a
max_depth / min_samples_leaf grid search). **Ridge outperforms Random Forest on this dataset** - with
only N=50 samples and 2048-dimensional features, a regularized linear model generalizes substantially
better than a flexible tree ensemble, which is prone to overfitting in this small-sample,
high-dimensional regime. Both results are reported in full below; Ridge is the model used for the
manuscript's headline statistics and Figure 6.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error

df_ml = pd.read_csv(f"{OUTPUT_DIR}/quantum_ground_truth_50.csv")
print(f"Dataset for ML: N={len(df_ml)}")

fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X, y, mol_ids = [], [], []
for _, row in df_ml.iterrows():
    mol = Chem.MolFromSmiles(row["SMILES"])
    if mol is None:
        continue
    X.append(fp_gen.GetCountFingerprintAsNumPy(mol))
    y.append(row["HOMO_LUMO_Gap_eV"])
    mol_ids.append(row["Molecule_ID"])

X, y = np.array(X), np.array(y)
print(f"Feature matrix: {X.shape}")

kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# --- Ridge regression: alpha sweep ---
ridge_results = []
for alpha in [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    cv_pred = cross_val_predict(ridge, X, y, cv=kf)
    cv_r2, cv_mae = r2_score(y, cv_pred), mean_absolute_error(y, cv_pred)
    ridge.fit(X, y)
    train_r2 = r2_score(y, ridge.predict(X))
    train_mae = mean_absolute_error(y, ridge.predict(X))
    ridge_results.append({"alpha": alpha, "Train_R2": train_r2, "Train_MAE": train_mae,
                           "CV_R2": cv_r2, "CV_MAE": cv_mae, "cv_pred": cv_pred.copy()})

best_ridge = max(ridge_results, key=lambda r: r["CV_R2"])
print(f"Best Ridge: alpha={best_ridge['alpha']}, CV R²={best_ridge['CV_R2']:.4f}, CV MAE={best_ridge['CV_MAE']:.4f} eV")

pd.DataFrame([{k: v for k, v in r.items() if k != "cv_pred"} for r in ridge_results])


In [ ]:
# --- Random Forest: grid search ---
rf_results = []
for max_depth in [2, 3, 4, 5, 6, None]:
    for min_samples_leaf in [1, 2, 3, 4, 5]:
        rf = RandomForestRegressor(n_estimators=100, max_depth=max_depth,
                                    min_samples_leaf=min_samples_leaf, random_state=42)
        cv_pred = cross_val_predict(rf, X, y, cv=kf)
        cv_r2, cv_mae = r2_score(y, cv_pred), mean_absolute_error(y, cv_pred)
        rf.fit(X, y)
        train_r2 = r2_score(y, rf.predict(X))
        train_mae = mean_absolute_error(y, rf.predict(X))
        rf_results.append({"max_depth": max_depth, "min_samples_leaf": min_samples_leaf,
                            "Train_R2": train_r2, "Train_MAE": train_mae,
                            "CV_R2": cv_r2, "CV_MAE": cv_mae})

best_rf = max(rf_results, key=lambda r: r["CV_R2"])
print(f"Best RF: max_depth={best_rf['max_depth']}, min_samples_leaf={best_rf['min_samples_leaf']}, "
      f"CV R²={best_rf['CV_R2']:.4f}, CV MAE={best_rf['CV_MAE']:.4f} eV")
print(f"(Ridge CV R²={best_ridge['CV_R2']:.4f} outperforms Random Forest here - "
      f"expected in this small-N, high-dimensional regime)")

df_table3 = pd.DataFrame(rf_results).sort_values("CV_R2", ascending=False)
df_table3.to_csv(f"{OUTPUT_DIR}/table3_grid_search.csv", index=False)
df_table3.head(10)


## Figures

### Figure 6 - Cross-validated parity plot (Ridge, the best-performing model)

In [ ]:
import matplotlib.pyplot as plt

best_pred = best_ridge["cv_pred"]
fig, ax = plt.subplots(figsize=(8, 7))
lims = [min(y.min(), best_pred.min()) - 0.5, max(y.max(), best_pred.max()) + 0.5]
ax.plot(lims, lims, "r--", label="Ideal Accuracy (y = x)")
ax.scatter(y, best_pred, s=70, edgecolor="black", facecolor="#4C8FD6",
           label=f"Out-of-Fold CV Predictions (N={len(y)})", zorder=3)

textstr = (f"Ridge Regression (L2-regularized linear model)\n"
           f"Dataset Scale: N = {len(y)} (real DFT data)\n"
           f"Hyperparameter: alpha = {best_ridge['alpha']}\n"
           f"Cross-Validation Metrics:\n"
           f"  • CV R² = {best_ridge['CV_R2']:.4f}\n"
           f"  • CV MAE = {best_ridge['CV_MAE']:.4f} eV")
ax.text(0.03, 0.97, textstr, transform=ax.transAxes, fontsize=9.5, va="top",
        bbox=dict(boxstyle="round", facecolor="#F0F0F0", edgecolor="gray"))
ax.set_xlabel("Actual Quantum HOMO-LUMO Gap via PySCF (eV)", fontsize=11, fontweight="bold")
ax.set_ylabel("Predicted HOMO-LUMO Gap via Ridge Regression (eV)", fontsize=11, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, linestyle=":")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure6_parity_plot.png", dpi=150)
plt.show()


### Figure 5 - Gap distribution: standard scaffolds vs. hypervalent -SF5 derivatives

In [ ]:
standard_gaps = df_ml[~df_ml["Molecule_ID"].isin(["MOL_0007", "MOL_0035"])]["HOMO_LUMO_Gap_eV"].tolist()
hyper_gaps = df_ml[df_ml["Molecule_ID"].isin(["MOL_0007", "MOL_0035"])]["HOMO_LUMO_Gap_eV"].tolist()

fig, ax = plt.subplots(figsize=(10, 7))
bins = np.arange(1, 8.5, 0.33)
ax.hist(standard_gaps, bins=bins, color="#4C8FD6", edgecolor="black",
        label=f"Standard Scaffolds (N={len(standard_gaps)})")
ax.hist(hyper_gaps, bins=bins, color="#D9695F", edgecolor="black",
        label=f"Hypervalent -SF5 Derivatives (N={len(hyper_gaps)})")
ax.set_xlabel("Calculated Molecular HOMO-LUMO Gap via PySCF (eV)", fontsize=12, fontweight="bold")
ax.set_ylabel("Molecular Candidate Entry Count", fontsize=12, fontweight="bold")
ax.legend(loc="upper left", fontsize=11)
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figure5_distribution.png", dpi=150)
plt.show()


##Figure 4- Multi-panel structural conformation and frontier molecular orbital alignment

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# Install rdkit if not already installed and import its modules
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
except ImportError:
    %pip install rdkit
    from rdkit import Chem
    from rdkit.Chem import Draw

# Set clean, professional academic publication layout styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=300, gridspec_kw={'width_ratios': [1, 1.2]})

# PHASE 1: GENERATE CHEMICAL GRAPH ASSETS VIA RDKIT
mol1 = Chem.MolFromSmiles("Fc1ccccc1")
mol7 = Chem.MolFromSmiles("FS(F)(F)(F)(F)c1ccccc1")
img1 = Draw.MolToImage(mol1, size=(400, 400), fitImage=True)
img7 = Draw.MolToImage(mol7, size=(400, 400), fitImage=True)

# PANEL A: STEREOCHEMICAL SCHEMATICS
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.text(0, 9.8, "Panel A: 3D Conformational Steric Profiles & Molecular Structures", fontsize=12, fontweight='bold', ha='left')

ax1.text(3.8, 8.2, "MOL_0001 (Fluorobenzene)", color='#0275d8', fontsize=11, fontweight='bold')
ax1.text(3.8, 6.8, "• Geometry: Completely Planar Framework\n• Steric Strain: Minimum Molecular Torsion\n• Interactions: Unhindered Intermolecular π-π Stacking", fontsize=10, fontweight='bold', color='#555555', style='italic', va='top')
ax1.imshow(img1, extent=[0.2, 3.2, 5.8, 8.8], zorder=2)

ax1.hlines(5.0, 0.2, 9.8, colors='#e0e0e0', linewidths=1.5)

ax1.text(3.8, 4.2, r"MOL_0007 (Pentafluorosulfanyl-Benzene, $-SF_5$)", color='#d9534f', fontsize=11, fontweight='bold')
ax1.text(3.8, 2.8, "• Geometry: Hypervalent Octahedral Core Matrix\n• Steric Strain: Highly Crowded Localized Cloud\n• Orbital Effect: Out-of-Plane Conjugation Disruption", fontsize=10, fontweight='bold', color='#555555', style='italic', va='top')
ax1.imshow(img7, extent=[0.2, 3.2, 1.8, 4.8], zorder=2)

# ==========================================
# PANEL B: Frontier Molecular Orbital Shift
# ==========================================
ax2.set_xlim(0.3, 2.7)
ax2.set_ylim(-9, 4)

# Y-Axis & Spine Configuration
ax2.set_ylabel("Orbital Energy Levels vs Vacuum Baseline (eV)", fontsize=11, fontweight='bold')
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["MOL_0001\n(Standard Reference)", "MOL_0007\n(Exotic $-SF_5$ Tag)"], fontsize=10, fontweight='bold')

# Remove Top and Right spines
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_linewidth(1.2)
ax2.spines['bottom'].set_linewidth(1.2)

# Gridlines
ax2.yaxis.grid(True, linestyle=':', alpha=0.4, color='#cccccc')
ax2.set_axisbelow(True)

# Title
ax2.set_title("Panel B: Frontier Molecular Orbital Shift", fontsize=12, fontweight='bold', loc='left', pad=12)

# Vacuum Baseline (0 eV)
ax2.axhline(0, color='#888888', linestyle='--', linewidth=1.2)
ax2.text(0.35, 0.15, "Vacuum Baseline (0 eV)", color='#888888', fontsize=9, style='italic')

# --- Data Values ---
m1_homo, m1_lumo = -4.38, +2.32
m2_homo, m2_lumo = -7.27, -0.77

bar_width = 0.22

# Plot Levels (MOL_0001)
ax2.hlines(y=m1_lumo, xmin=1 - bar_width, xmax=1 + bar_width, colors='red', lw=4, label='LUMO (Virtual State)')
ax2.hlines(y=m1_homo, xmin=1 - bar_width, xmax=1 + bar_width, colors='blue', lw=4, label='HOMO (Occupied State)')

# Plot Levels (MOL_0007)
ax2.hlines(y=m2_lumo, xmin=2 - bar_width, xmax=2 + bar_width, colors='red', lw=4)
ax2.hlines(y=m2_homo, xmin=2 - bar_width, xmax=2 + bar_width, colors='blue', lw=4)

# Text Annotations for Levels
ax2.text(1, m1_lumo + 0.35, "LUMO: +2.32 eV", color='red', fontweight='bold', fontsize=10, ha='center')
ax2.text(1, m1_homo - 0.55, "HOMO: -4.38 eV", color='blue', fontweight='bold', fontsize=10, ha='center')

ax2.text(2, m2_lumo + 0.35, "LUMO: -0.77 eV", color='red', fontweight='bold', fontsize=10, ha='center')
ax2.text(2, m2_homo - 0.55, "HOMO: -7.27 eV", color='blue', fontweight='bold', fontsize=10, ha='center')

# Gap Arrows and Text Annotations
# MOL_0001 Gap Arrow
ax2.annotate('', xy=(1, m1_lumo - 0.1), xytext=(1, m1_homo + 0.1),
             arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))
ax2.text(1.22, (m1_lumo + m1_homo)/2, "Gap: 6.71 eV\n(Wide-Gap)",
         color='black', fontsize=9.5, fontweight='bold', va='center', ha='left')

# MOL_0007 Gap Arrow
ax2.annotate('', xy=(2, m2_lumo - 0.1), xytext=(2, m2_homo + 0.1),
             arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))
ax2.text(2.22, (m2_lumo + m2_homo)/2, "Gap: 6.51 eV\n(Stable Window)",
         color='black', fontsize=9.5, fontweight='bold', va='center', ha='left')

# Legend
ax2.legend(loc='upper right', frameon=True, facecolor='#fbfbfb', edgecolor='#e0e0e0', fontsize=8.5)

# Adjust layout and show
plt.tight_layout()
plt.show()

##Figure 3: Computational execution latency performance benchmark

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ============================================================
# RDKit -> ASE 50-Molecule Batch Performance Benchmark
# Publication-Quality Plot (Canonical Standard Baseline)
# ============================================================

# ------------------------------------------------------------
# 1. Publication-quality plotting style
# ------------------------------------------------------------

sns.set_theme(style="whitegrid", font="sans-serif")

plt.rcParams.update({
    "font.sans-serif": ["DejaVu Sans"],  # Use a commonly available font
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 300,
})

# ------------------------------------------------------------
# 2. Benchmark data (Canonical Aggregated Baseline)
# ------------------------------------------------------------

methods = [
    "File-Based Transfer\n(.xyz Serialization + I/O)",
    "In-Memory Transfer\n(Direct Coordinate Extraction)",
]

means = np.array([18.96, 4.95])

stds = np.array([0.22, 0.10])

# Benchmark comparison metrics
mean_speedup = 3.84
time_reduction = 73.93

molecules_per_batch = 50
timed_batches = 100

# ------------------------------------------------------------
# 3. Create figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.0, 5.5))
x = np.arange(len(methods))
width = 0.55

# Colors
colors = ["#d95f02", "#2ca02c"]  # Colors for bars

# ------------------------------------------------------------
# 4. Draw bars with error bars
# ------------------------------------------------------------

bars = ax.bar(
    x,
    means,
    width=width,
    yerr=stds,
    capsize=7,
    color=colors,
    edgecolor="black",
    linewidth=1.2,
    alpha=0.88,
    error_kw={"ecolor": "#333333", "elinewidth": 1.6, "capthick": 1.6},
)

# ------------------------------------------------------------
# 5. Add mean ± SD labels inside bars
# ------------------------------------------------------------

for bar, mean, std in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() / 2,
        f"{mean:.2f} ms\n± {std:.2f}",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="white",
    )

# ------------------------------------------------------------
# 6. Add upper error-bar values
# ------------------------------------------------------------

for bar, mean, std in zip(bars, means, stds):
    upper_value = mean + std
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        upper_value + 0.4,
        f"{upper_value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color="#333333",
    )

# ------------------------------------------------------------
# 7. Performance summary annotation box
# ------------------------------------------------------------

summary_text = (
    f"Mean speedup: {mean_speedup:.2f}×\n"
    f"Time reduction: {time_reduction:.2f}%\n"
    f"{molecules_per_batch} molecules/batch; N = {timed_batches}"
)

ax.text(
    0.97,
    0.96,
    summary_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9.5,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.55",
        facecolor="#eef2f5",
        edgecolor="#9eabb3",
        linewidth=1.1,
    ),
)

# ------------------------------------------------------------
# 8. Axis labels
# ------------------------------------------------------------

ax.set_ylabel("Batch Transfer Latency (ms)", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(methods)

# ------------------------------------------------------------
# 9. Y-axis limit (Adjusted for canonical range)
# ------------------------------------------------------------

ax.set_ylim(0, 25)

# ------------------------------------------------------------
# 10. Grid and spine formatting
# ------------------------------------------------------------

ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.5)
ax.grid(axis="x", visible=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ------------------------------------------------------------
# 11. Layout
# ------------------------------------------------------------

plt.tight_layout()

# ------------------------------------------------------------
# 12. Save figure in multiple formats
# ------------------------------------------------------------

#plt.savefig(
#    "Figure_3_RDKit_ASE_50Molecule_Benchmark.png", dpi=300, bbox_inches="tight"
#)
#plt.savefig("Figure_3_RDKit_ASE_50Molecule_Benchmark.pdf", bbox_inches="tight")
#plt.savefig(
#    "Figure_3_RDKit_ASE_50Molecule_Benchmark.tiff", dpi=600, bbox_inches="tight"
#)

# ------------------------------------------------------------
# 13. Show plot
# ------------------------------------------------------------

plt.show()

### Table 2 - Representative subset (first 10 molecules, matching the manuscript)

In [ ]:
table2_ids = [f"MOL_{i:04d}" for i in range(1, 11)]
df_table2 = df_ml[df_ml["Molecule_ID"].isin(table2_ids)][
    ["Molecule_ID", "SMILES", "Total_Electronic_Energy_Hartree", "HOMO_eV", "LUMO_eV",
     "HOMO_LUMO_Gap_eV", "Status", "Max_Force_eV_per_A"]
]
df_table2.to_csv(f"{OUTPUT_DIR}/table2_representative_subset.csv", index=False)
df_table2
